# Gold Causal-Analysis EDA — Updated Three-Sheet System

This notebook uses three distinct analytical datasets:

1. **Monthly Causal Data** — one global observation per month.
2. **Quarterly Demand Data** — quarterly WGC demand, jewellery and aggregated controls.
3. **Regional ETF Panel** — a balanced `Month × Region` panel for North America, Europe, Asia and Others.

Central-bank data are deliberately **global only**. The regional panel contains genuine cross-sectional variation in ETF flows; repeated global variables do not become independent observations merely because they appear in four regional rows.

In [1]:
# CELL 1: Libraries, workbook upload and three-sheet loading

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from scipy.stats import combine_pvalues
from statsmodels.tsa.stattools import adfuller, kpss

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

DEFAULT_FILE = "Gold_1(2).xlsx"

if Path(DEFAULT_FILE).exists():
    file_name = DEFAULT_FILE
else:
    try:
        from google.colab import files
        uploaded = files.upload()
        file_name = next(iter(uploaded))
    except ImportError as exc:
        raise FileNotFoundError(
            f"Place {DEFAULT_FILE} in the working directory or run this notebook in Colab and upload it."
        ) from exc

monthly = pd.read_excel(file_name, sheet_name="Monthly Causal Data")
quarterly = pd.read_excel(file_name, sheet_name="Quarterly Demand Data")
panel_df = pd.read_excel(file_name, sheet_name="Regional ETF Panel")

# Remove the source/methodology footer rows included below each data table.
monthly["Month"] = pd.to_datetime(monthly["Month"], errors="coerce")
monthly = monthly.loc[monthly["Month"].notna()].copy()
quarterly["Quarter_End"] = pd.to_datetime(quarterly["Quarter_End"], errors="coerce")
quarterly = quarterly.loc[quarterly["Quarter_End"].notna()].copy()
panel_df["Month"] = pd.to_datetime(panel_df["Month"], errors="coerce")
panel_df = panel_df.loc[panel_df["Month"].notna()].copy()

monthly = monthly.sort_values("Month").reset_index(drop=True)
quarterly = quarterly.sort_values("Quarter_End").reset_index(drop=True)
panel_df["Region"] = pd.Categorical(
    panel_df["Region"],
    categories=["North America", "Europe", "Asia", "Others"],
    ordered=True,
)
panel_df = panel_df.sort_values(["Month", "Region_ID"]).reset_index(drop=True)

print("Workbook:", file_name)
print("Monthly data:", monthly.shape, monthly["Month"].min(), "to", monthly["Month"].max())
print("Quarterly data:", quarterly.shape, quarterly["Quarter"].min(), "to", quarterly["Quarter"].max())
print("Regional ETF panel:", panel_df.shape)
display(monthly.head(3))
display(panel_df.head(8))

Saving Gold_1.xlsx to Gold_1.xlsx
Workbook: Gold_1.xlsx
Monthly data: (91, 47) 2019-01-01 00:00:00 to 2026-07-01 00:00:00
Quarterly data: (30, 40) 2019-Q1 to 2026-Q2
Regional ETF panel: (364, 49)


,Month,Year,Quarter,Month_ID,Gold_Monthly_Return_pct,ETF_Net_Demand_Global_WGC_tonnes,CB_Reported_Holdings_Change_Global_IMF_tonnes,ETF_plus_CB_IMF_tonnes,CB_Reported_Net_Change_Global_WGC_tonnes,ETF_plus_CB_WGC_tonnes,Jewellery_Quarterly_Control_tonnes,Jewellery_QoQ_Growth_pct,Jewellery_YoY_Growth_pct,US_CPI_Inflation_MoM_pct,SP500_Monthly_Return_pct,Broad_USD_Index_Avg,US_10Y_Yield_MonthEnd_pct,Brent_MonthEnd_USD_per_Barrel,Effective_Fed_Funds_Avg_pct,US_10Y_Real_Yield_Monthly_Avg_pct,US_10Y_Real_Yield_MonthEnd_pct,US_10Y_Real_Yield_Monthly_Change_pp,VIX_Monthly_Avg,VIX_MonthEnd,VIX_Monthly_Change_points,Term_Spread_10Y2Y_Monthly_Avg_pct,Term_Spread_10Y2Y_MonthEnd_pct,USDINR_Monthly_Avg,USDINR_Monthly_Change_pct,USDCNY_Monthly_Avg,USDCNY_Monthly_Change_pct,US_EPU_Monthly_Index,US_EPU_Monthly_Change_pct,US_High_Yield_OAS_Monthly_Avg_pct,US_High_Yield_OAS_MonthEnd_pct,US_High_Yield_OAS_Monthly_Change_pp,CB_Global_Adjacent_Pair_Countries_IMF,CB_Global_Eligible_Countries_IMF,CB_Global_Adjacent_Coverage_pct,CB_IMF_Coverage_50pct_Flag,Core_Causal_Row_Complete_Flag,Post_2022_Dummy,Post_2025_Dummy,ETF_Net_Demand_North_America_WGC_tonnes,ETF_Net_Demand_Europe_WGC_tonnes,ETF_Net_Demand_Asia_WGC_tonnes,ETF_Net_Demand_Others_WGC_tonnes
0,2019-01-01,2019,2019-Q1,1.0000,3.0211,71.8000,68.1649,139.9649,NaN,NaN,540.0040,NaN,NaN,-0.0815,7.7318,104.0386,2.6300,61.8900,2.2909,0.9200,0.7800,-0.2000,19.5724,16.5700,-8.8500,0.1705,0.1800,70.7100,-0.1738,6.7863,-1.4144,201.0320,20.8947,4.5723,4.3700,-0.9600,150.0000,154.0000,0.9740,1.0000,1.0000,0.0000,0.0000,53.0000,20.0000,0.1000,-1.3000
1,2019-02-01,2019,2019-Q1,2.0000,-0.3113,-32.4000,18.2362,-14.1638,NaN,NaN,540.0040,NaN,NaN,0.3001,2.8804,102.9717,2.7300,66.0300,2.2800,0.7995,0.7800,0.0000,15.2347,14.7800,-1.7900,0.1721,0.2100,71.1739,0.6560,6.7367,-0.7318,106.9007,-46.8241,4.1280,3.9200,-0.4500,149.0000,154.0000,0.9675,1.0000,1.0000,0.0000,0.0000,-29.0000,-0.3000,-3.0000,-0.1000
2,2019-03-01,2019,2019-Q1,3.0000,-0.2623,1.7000,46.9922,48.6922,NaN,NaN,540.0040,NaN,NaN,0.3782,1.0953,114.7835,2.4100,68.3900,2.4038,0.6581,0.5300,-0.2500,14.4852,13.7100,-1.0700,0.1614,0.1400,69.4895,-2.3665,6.7119,-0.3673,140.2474,31.1941,4.0105,4.0500,0.1300,148.0000,154.0000,0.9610,1.0000,1.0000,0.0000,0.0000,2.5000,0.2000,-1.2000,0.2000


,Month,Year,Quarter,Month_ID,Region_ID,Region,Panel_Row_ID,ETF_Net_Demand_Regional_WGC_tonnes,ETF_Region_Share_of_Global,ETF_Regional_Deviation_From_Monthly_Mean_tonnes,Gold_Monthly_Return_pct,ETF_Net_Demand_Global_WGC_tonnes,CB_Reported_Holdings_Change_Global_IMF_tonnes,ETF_plus_CB_IMF_tonnes,CB_Reported_Net_Change_Global_WGC_tonnes,ETF_plus_CB_WGC_tonnes,Jewellery_Quarterly_Control_tonnes,Jewellery_QoQ_Growth_pct,Jewellery_YoY_Growth_pct,US_CPI_Inflation_MoM_pct,SP500_Monthly_Return_pct,Broad_USD_Index_Avg,US_10Y_Yield_MonthEnd_pct,Brent_MonthEnd_USD_per_Barrel,Effective_Fed_Funds_Avg_pct,US_10Y_Real_Yield_Monthly_Avg_pct,US_10Y_Real_Yield_MonthEnd_pct,US_10Y_Real_Yield_Monthly_Change_pp,VIX_Monthly_Avg,VIX_MonthEnd,VIX_Monthly_Change_points,Term_Spread_10Y2Y_Monthly_Avg_pct,Term_Spread_10Y2Y_MonthEnd_pct,USDINR_Monthly_Avg,USDINR_Monthly_Change_pct,USDCNY_Monthly_Avg,USDCNY_Monthly_Change_pct,US_EPU_Monthly_Index,US_EPU_Monthly_Change_pct,US_High_Yield_OAS_Monthly_Avg_pct,US_High_Yield_OAS_MonthEnd_pct,US_High_Yield_OAS_Monthly_Change_pp,CB_Global_Adjacent_Pair_Countries_IMF,CB_Global_Eligible_Countries_IMF,CB_Global_Adjacent_Coverage_pct,CB_IMF_Coverage_50pct_Flag,Core_Causal_Row_Complete_Flag,Post_2022_Dummy,Post_2025_Dummy
0,2019-01-01,2019,2019-Q1,1.0000,1.0000,North America,2019-01_1,53.0000,0.7382,35.0500,3.0211,71.8000,68.1649,139.9649,NaN,NaN,540.0040,NaN,NaN,-0.0815,7.7318,104.0386,2.6300,61.8900,2.2909,0.9200,0.7800,-0.2000,19.5724,16.5700,-8.8500,0.1705,0.1800,70.7100,-0.1738,6.7863,-1.4144,201.0320,20.8947,4.5723,4.3700,-0.9600,150.0000,154.0000,0.9740,1.0000,1.0000,NaN,NaN
1,2019-01-01,2019,2019-Q1,1.0000,2.0000,Europe,2019-01_2,20.0000,0.2786,2.0500,3.0211,71.8000,68.1649,139.9649,NaN,NaN,540.0040,NaN,NaN,-0.0815,7.7318,104.0386,2.6300,61.8900,2.2909,0.9200,0.7800,-0.2000,19.5724,16.5700,-8.8500,0.1705,0.1800,70.7100,-0.1738,6.7863,-1.4144,201.0320,20.8947,4.5723,4.3700,-0.9600,150.0000,154.0000,0.9740,1.0000,1.0000,NaN,NaN
2,2019-01-01,2019,2019-Q1,1.0000,3.0000,Asia,2019-01_3,0.1000,0.0014,-17.8500,3.0211,71.8000,68.1649,139.9649,NaN,NaN,540.0040,NaN,NaN,-0.0815,7.7318,104.0386,2.6300,61.8900,2.2909,0.9200,0.7800,-0.2000,19.5724,16.5700,-8.8500,0.1705,0.1800,70.7100,-0.1738,6.7863,-1.4144,201.0320,20.8947,4.5723,4.3700,-0.9600,150.0000,154.0000,0.9740,1.0000,1.0000,NaN,NaN
3,2019-01-01,2019,2019-Q1,1.0000,4.0000,Others,2019-01_4,-1.3000,-0.0181,-19.2500,3.0211,71.8000,68.1649,139.9649,NaN,NaN,540.0040,NaN,NaN,-0.0815,7.7318,104.0386,2.6300,61.8900,2.2909,0.9200,0.7800,-0.2000,19.5724,16.5700,-8.8500,0.1705,0.1800,70.7100,-0.1738,6.7863,-1.4144,201.0320,20.8947,4.5723,4.3700,-0.9600,150.0000,154.0000,0.9740,1.0000,1.0000,NaN,NaN
4,2019-02-01,2019,2019-Q1,2.0000,1.0000,North America,2019-02_1,-29.0000,0.8951,-20.9000,-0.3113,-32.4000,18.2362,-14.1638,NaN,NaN,540.0040,NaN,NaN,0.3001,2.8804,102.9717,2.7300,66.0300,2.2800,0.7995,0.7800,NaN,15.2347,14.7800,-1.7900,0.1721,0.2100,71.1739,0.6560,6.7367,-0.7318,106.9007,-46.8241,4.1280,3.9200,-0.4500,149.0000,154.0000,0.9675,1.0000,1.0000,NaN,NaN
5,2019-02-01,2019,2019-Q1,2.0000,2.0000,Europe,2019-02_2,-0.3000,0.0093,7.8000,-0.3113,-32.4000,18.2362,-14.1638,NaN,NaN,540.0040,NaN,NaN,0.3001,2.8804,102.9717,2.7300,66.0300,2.2800,0.7995,0.7800,NaN,15.2347,14.7800,-1.7900,0.1721,0.2100,71.1739,0.6560,6.7367,-0.7318,106.9007,-46.8241,4.1280,3.9200,-0.4500,149.0000,154.0000,0.9675,1.0000,1.0000,NaN,NaN
6,2019-02-01,2019,2019-Q1,2.0000,3.0000,Asia,2019-02_3,-3.0000,0.0926,5.1000,-0.3113,-32.4000,18.2362,-14.1638,NaN,NaN,540.0040,NaN,NaN,0.3001,2.8804,102.9717,2.7300,66.0300,2.2800,0.7995,0.7800,NaN,15.2347,14.7800,-1.7900,0.1721,0.2100,71.1739,0.6560,6.7367,-0.7318,106.9007,-46.8241,4.1280,3.9200,-0.4500,149.0000,154.0000,0.9675,1.0000,1.0000,NaN,NaN
7,2019-02-01,2019,2019-Q1,2.0000,4.0000,Others,2019-02_4,-0.1000,0.0031,8.0000,-0.3113,-32.4000,18.2362,-14.1638,NaN,NaN,540.0040,NaN,NaN,0.3001,2.8804,102.9717,2.7300,66.0300,2.2800,0.7995,0.7800,NaN,15.2347,14.7800,-1.7900,0.17

## 1. Structural validation

These checks ensure the panel is balanced, identifiers are unique and the four regional ETF observations reconcile to the global monthly ETF total.

In [2]:
# CELL 2: Structural and accounting validation

panel_counts = panel_df.groupby("Month", observed=True)["Region"].nunique()
regional_reconciliation = (
    panel_df.groupby("Month", observed=True)["ETF_Net_Demand_Regional_WGC_tonnes"].sum()
    - monthly.set_index("Month")["ETF_Net_Demand_Global_WGC_tonnes"]
)

validation = pd.DataFrame({
    "Check": [
        "Monthly observations",
        "Quarterly observations",
        "Panel observations",
        "Unique Month–Region keys",
        "Four regions in every month",
        "Maximum ETF regional-total reconciliation error",
    ],
    "Result": [
        len(monthly),
        len(quarterly),
        len(panel_df),
        panel_df["Panel_Row_ID"].nunique(),
        bool(panel_counts.eq(4).all()),
        regional_reconciliation.abs().max(),
    ],
    "Expected": [91, 30, 364, 364, True, "Approximately zero"],
})

display(validation)
assert len(monthly) == 91
assert len(quarterly) == 30
assert len(panel_df) == 364
assert panel_df["Panel_Row_ID"].is_unique
assert panel_counts.eq(4).all()
assert regional_reconciliation.abs().max() < 1e-8

,Check,Result,Expected
0,Monthly observations,91,91
1,Quarterly observations,30,30
2,Panel observations,364,364
3,Unique Month–Region keys,364,364
4,Four regions in every month,True,True
5,Maximum ETF regional-total reconciliation error,0.0000,Approximately zero


In [3]:
# CELL 3: Variable-level coverage audit by analytical dataset

monthly_variables = [
    "Gold_Monthly_Return_pct",
    "ETF_Net_Demand_Global_WGC_tonnes",
    "CB_Reported_Holdings_Change_Global_IMF_tonnes",
    "CB_Reported_Net_Change_Global_WGC_tonnes",
    "Jewellery_Quarterly_Control_tonnes",
    "US_CPI_Inflation_MoM_pct",
    "SP500_Monthly_Return_pct",
    "Broad_USD_Index_Avg",
    "US_10Y_Yield_MonthEnd_pct",
    "US_10Y_Real_Yield_MonthEnd_pct",
    "VIX_MonthEnd",
    "Term_Spread_10Y2Y_MonthEnd_pct",
    "USDINR_Monthly_Change_pct",
    "USDCNY_Monthly_Change_pct",
    "US_EPU_Monthly_Change_pct",
    "US_High_Yield_OAS_MonthEnd_pct",
    "Brent_MonthEnd_USD_per_Barrel",
    "Effective_Fed_Funds_Avg_pct",
]

panel_variables = [
    "ETF_Net_Demand_Regional_WGC_tonnes",
    "ETF_Region_Share_of_Global",
    "ETF_Regional_Deviation_From_Monthly_Mean_tonnes",
]

quarterly_variables = [
    "Gold_Quarterly_Return_pct",
    "ETF_Net_Demand_WGC_Quarterly_tonnes",
    "CB_Reported_Holdings_Change_IMF_Quarterly_tonnes",
    "CB_Reported_Net_Change_WGC_QuarterlySum_tonnes",
    "CB_Total_Demand_WGC_Quarterly_tonnes",
    "Jewellery_Demand_WGC_Quarterly_tonnes",
]

def coverage_table(data, variables, dataset_name):
    out = pd.DataFrame({
        "Dataset": dataset_name,
        "Variable": variables,
        "Available_Observations": [data[v].notna().sum() for v in variables],
        "Missing_Observations": [data[v].isna().sum() for v in variables],
        "Missing_Percent": [100 * data[v].isna().mean() for v in variables],
    })
    return out

missing_audit = pd.concat([
    coverage_table(monthly, monthly_variables, "Monthly global"),
    coverage_table(panel_df, panel_variables, "Regional ETF panel"),
    coverage_table(quarterly, quarterly_variables, "Quarterly"),
], ignore_index=True)

display(missing_audit.sort_values(["Dataset", "Missing_Percent", "Variable"]))

,Dataset,Variable,Available_Observations,Missing_Observations,Missing_Percent
16,Monthly global,Brent_MonthEnd_USD_per_Barrel,91,0,0.0000
7,Monthly global,Broad_USD_Index_Avg,91,0,0.0000
2,Monthly global,CB_Reported_Holdings_Change_Global_IMF_tonnes,91,0,0.0000
1,Monthly global,ETF_Net_Demand_Global_WGC_tonnes,91,0,0.0000
17,Monthly global,Effective_Fed_Funds_Avg_pct,91,0,0.0000
0,Monthly global,Gold_Monthly_Return_pct,91,0,0.0000
6,Monthly global,SP500_Monthly_Return_pct,91,0,0.0000
11,Monthly global,Term_Spread_10Y2Y_MonthEnd_pct,91,0,0.0000
13,Monthly global,USDCNY_Monthly_Change_pct,91,0,0.0000
12,Monthly global,USDINR_Monthly_Change_pct,91,0,0.0000


In [4]:
# CELL 4: Plotly missing-data chart

plot_data = missing_audit.copy()
plot_data["Label"] = plot_data["Dataset"] + " — " + plot_data["Variable"]

fig = px.bar(
    plot_data.sort_values("Missing_Percent"),
    x="Missing_Percent",
    y="Label",
    orientation="h",
    color="Dataset",
    text="Missing_Percent",
    title="Missing-Data Percentage Across the Three Analytical Datasets",
)
fig.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig.update_layout(
    template="plotly_white", height=850,
    xaxis_title="Missing observations (%)", yaxis_title=None,
    legend_title=None,
)
fig.show()

## 2. Central-bank data: global reporting and validation

The IMF series measures adjacent-month changes in reported physical gold holdings. WGC monthly observations are also reported-reserve changes. WGC quarterly total demand is broader and may include estimated unreported activity; it is therefore analysed separately.

In [5]:
# CELL 5: Global IMF reporting coverage

coverage_plot = monthly[[
    "Month", "CB_Global_Adjacent_Coverage_pct",
    "CB_Global_Adjacent_Pair_Countries_IMF",
    "CB_Global_Eligible_Countries_IMF",
]].copy()
coverage_plot["Coverage_Percent"] = 100 * coverage_plot["CB_Global_Adjacent_Coverage_pct"]

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(
    x=coverage_plot["Month"], y=coverage_plot["Coverage_Percent"],
    mode="lines+markers", name="Adjacent-month coverage",
    line=dict(color="#1F77B4", width=2.5), marker=dict(size=5),
    hovertemplate="%{x|%b-%Y}<br>Coverage: %{y:.1f}%<extra></extra>",
), secondary_y=False)
fig.add_trace(go.Bar(
    x=coverage_plot["Month"], y=coverage_plot["CB_Global_Adjacent_Pair_Countries_IMF"],
    name="Country pairs", marker_color="#B0BEC5", opacity=0.45,
    hovertemplate="%{x|%b-%Y}<br>Adjacent pairs: %{y:.0f}<extra></extra>",
), secondary_y=True)
fig.add_hline(y=50, line_dash="dash", line_color="red", annotation_text="50% diagnostic threshold")
fig.update_yaxes(title_text="Coverage (%)", range=[0, 105], ticksuffix="%", secondary_y=False)
fig.update_yaxes(title_text="Adjacent country pairs", secondary_y=True)
fig.update_layout(
    title="IMF Global Central-Bank Gold-Holdings Change Coverage",
    template="plotly_white", height=550, hovermode="x unified",
    legend=dict(orientation="h", y=1.10, x=0.5, xanchor="center"),
)
fig.show()

In [6]:
# CELL 6: IMF versus WGC monthly reported central-bank changes

cb_comparison = monthly[[
    "Month",
    "CB_Reported_Holdings_Change_Global_IMF_tonnes",
    "CB_Reported_Net_Change_Global_WGC_tonnes",
]].copy()
cb_comparison["IMF_minus_WGC_tonnes"] = (
    cb_comparison["CB_Reported_Holdings_Change_Global_IMF_tonnes"]
    - cb_comparison["CB_Reported_Net_Change_Global_WGC_tonnes"]
)
comparison_sample = cb_comparison.dropna(subset=["CB_Reported_Net_Change_Global_WGC_tonnes"])

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.12,
    subplot_titles=["Reported Monthly Changes", "IMF Minus WGC"],
    row_heights=[0.65, 0.35],
)
fig.add_trace(go.Bar(
    x=comparison_sample["Month"],
    y=comparison_sample["CB_Reported_Holdings_Change_Global_IMF_tonnes"],
    name="IMF country aggregation", marker_color="#1F77B4", opacity=0.75,
), row=1, col=1)
fig.add_trace(go.Bar(
    x=comparison_sample["Month"],
    y=comparison_sample["CB_Reported_Net_Change_Global_WGC_tonnes"],
    name="WGC monthly reported", marker_color="#D4A017", opacity=0.75,
), row=1, col=1)
fig.add_trace(go.Bar(
    x=comparison_sample["Month"], y=comparison_sample["IMF_minus_WGC_tonnes"],
    name="IMF − WGC",
    marker_color=np.where(comparison_sample["IMF_minus_WGC_tonnes"] >= 0, "#2E8B57", "#C0392B"),
), row=2, col=1)
for row in (1, 2): fig.add_hline(y=0, line_color="black", line_width=1, row=row, col=1)
fig.update_yaxes(title_text="Tonnes", row=1, col=1)
fig.update_yaxes(title_text="Difference", row=2, col=1)
fig.update_layout(
    title="Validation of IMF Holdings Changes Against WGC Monthly Reported Data",
    template="plotly_white", height=760, barmode="group", hovermode="x unified",
    legend=dict(orientation="h", y=1.08, x=0.5, xanchor="center"),
)
fig.show()

In [7]:
# CELL 7: Quarterly reported changes versus WGC total demand

q_cb = quarterly[[
    "Quarter_End", "Quarter",
    "CB_Reported_Holdings_Change_IMF_Quarterly_tonnes",
    "CB_Reported_Net_Change_WGC_QuarterlySum_tonnes",
    "CB_Total_Demand_WGC_Quarterly_tonnes",
    "WGC_Total_Demand_minus_IMF_Reported_Change_tonnes",
]].copy()

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.12,
    subplot_titles=["Three Central-Bank Measures", "WGC Total Demand Minus IMF Reported Change"],
)
for variable, name, colour in [
    ("CB_Reported_Holdings_Change_IMF_Quarterly_tonnes", "IMF reported holdings change", "#1F77B4"),
    ("CB_Reported_Net_Change_WGC_QuarterlySum_tonnes", "WGC monthly-reported sum", "#7F8C8D"),
    ("CB_Total_Demand_WGC_Quarterly_tonnes", "WGC total quarterly demand", "#D4A017"),
]:
    fig.add_trace(go.Bar(x=q_cb["Quarter"], y=q_cb[variable], name=name, marker_color=colour), row=1, col=1)
fig.add_trace(go.Bar(
    x=q_cb["Quarter"], y=q_cb["WGC_Total_Demand_minus_IMF_Reported_Change_tonnes"],
    name="Total-demand gap", marker_color="#8E44AD",
), row=2, col=1)
for row in (1, 2): fig.add_hline(y=0, line_color="black", line_width=1, row=row, col=1)
fig.update_layout(
    title="Reported Central-Bank Changes Versus Broader WGC Quarterly Demand",
    template="plotly_white", height=780, barmode="group", hovermode="x unified",
    legend=dict(orientation="h", y=1.10, x=0.5, xanchor="center"),
)
fig.update_yaxes(title_text="Tonnes", row=1, col=1)
fig.update_yaxes(title_text="Tonnes", row=2, col=1)
fig.show()

## 3. Regional ETF panel

Only ETF demand varies across regions. North America is retained as the reference region for later fixed-effects or dummy-variable specifications.

In [8]:
# CELL 8: Regional monthly ETF demand — Plotly small multiples

regions = ["North America", "Europe", "Asia", "Others"]
positions = {"North America": (1, 1), "Europe": (1, 2), "Asia": (2, 1), "Others": (2, 2)}
fig = make_subplots(rows=2, cols=2, subplot_titles=regions, shared_xaxes=True,
                    vertical_spacing=0.14, horizontal_spacing=0.08)

for region in regions:
    row, col = positions[region]
    temp = panel_df.loc[panel_df["Region"] == region].sort_values("Month")
    flow = temp["ETF_Net_Demand_Regional_WGC_tonnes"]
    fig.add_trace(go.Bar(
        x=temp["Month"], y=flow,
        marker_color=np.where(flow >= 0, "#1F77B4", "#FF7F0E"),
        hovertemplate="%{x|%b-%Y}<br>ETF net demand: %{y:.2f} t<extra></extra>",
        showlegend=False,
    ), row=row, col=col)
    fig.add_hline(y=0, line_width=1, line_color="black", row=row, col=col)

fig.update_yaxes(title_text="Tonnes", row=1, col=1)
fig.update_yaxes(title_text="Tonnes", row=2, col=1)
fig.update_layout(title="Monthly Gold-ETF Net Demand by WGC Region",
                  template="plotly_white", height=750, bargap=0.15)
fig.show()

In [9]:
# CELL 9: Regional ETF heatmap and within-month deviations

etf_matrix = panel_df.pivot(
    index="Region", columns="Month", values="ETF_Net_Demand_Regional_WGC_tonnes"
).reindex(regions)

fig = go.Figure(go.Heatmap(
    z=etf_matrix.values, x=etf_matrix.columns, y=etf_matrix.index,
    colorscale="RdBu", zmid=0, colorbar=dict(title="Tonnes"),
    hovertemplate="Month: %{x|%b-%Y}<br>Region: %{y}<br>ETF flow: %{z:.2f} t<extra></extra>",
))
fig.update_layout(title="Heatmap of Regional ETF Net Demand", template="plotly_white",
                  height=420, xaxis_title="Month", yaxis_title=None)
fig.show()

## 4. Gold returns and demand components

In [10]:
# CELL 10: Gold monthly returns and annualised rolling volatility

gold_ts = monthly[["Month", "Gold_Monthly_Return_pct"]].copy()
gold_ts["Gold_12M_Rolling_Volatility_pct"] = (
    gold_ts["Gold_Monthly_Return_pct"].rolling(12, min_periods=12).std() * np.sqrt(12)
)
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.10,
                    subplot_titles=["Monthly Gold Returns", "Annualised 12-Month Rolling Volatility"],
                    row_heights=[0.60, 0.40])
fig.add_trace(go.Bar(
    x=gold_ts["Month"], y=gold_ts["Gold_Monthly_Return_pct"],
    marker_color=np.where(gold_ts["Gold_Monthly_Return_pct"] >= 0, "#D4A017", "#8B0000"),
    name="Monthly return",
), row=1, col=1)
fig.add_trace(go.Scatter(
    x=gold_ts["Month"], y=gold_ts["Gold_12M_Rolling_Volatility_pct"],
    mode="lines", line=dict(color="#4B0082", width=2.5), name="12-month volatility",
), row=2, col=1)
fig.add_hline(y=0, line_width=1, line_color="black", row=1, col=1)
for date in ["2022-01-01", "2025-01-01"]:
    fig.add_vline(x=pd.Timestamp(date), line_dash="dash", line_color="grey", line_width=1)
fig.update_yaxes(title_text="Return (%)", row=1, col=1)
fig.update_yaxes(title_text="Volatility (%)", row=2, col=1)
fig.update_layout(title="Gold Returns and Time-Varying Volatility",
                  template="plotly_white", height=700, hovermode="x unified", showlegend=False)
fig.show()

In [11]:
# CELL 11: Global gold return, IMF central-bank change and ETF demand

monthly_core = monthly[[
    "Month", "Gold_Monthly_Return_pct",
    "CB_Reported_Holdings_Change_Global_IMF_tonnes",
    "ETF_Net_Demand_Global_WGC_tonnes",
]].copy()

fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.07,
                    subplot_titles=["Monthly Gold Return", "IMF-Reported Global CB Change", "Global ETF Net Demand"])
series_specs = [
    ("Gold_Monthly_Return_pct", "#D4A017", "#8B0000", 1),
    ("CB_Reported_Holdings_Change_Global_IMF_tonnes", "#2E8B57", "#C0392B", 2),
    ("ETF_Net_Demand_Global_WGC_tonnes", "#1F77B4", "#FF7F0E", 3),
]
for variable, positive, negative, row in series_specs:
    values = monthly_core[variable]
    fig.add_trace(go.Bar(x=monthly_core["Month"], y=values,
                         marker_color=np.where(values >= 0, positive, negative),
                         showlegend=False), row=row, col=1)
    fig.add_hline(y=0, line_color="black", line_width=1, row=row, col=1)
fig.update_yaxes(title_text="Return (%)", row=1, col=1)
fig.update_yaxes(title_text="Tonnes", row=2, col=1)
fig.update_yaxes(title_text="Tonnes", row=3, col=1)
fig.update_layout(title="Gold Return and the Two Principal Monthly Demand Components",
                  template="plotly_white", height=900, hovermode="x unified")
fig.show()

In [12]:
# CELL 12: Jewellery demand alongside gold return and quarterly ETF/CB measures

q_plot = quarterly[[
    "Quarter", "Gold_Quarterly_Return_pct", "Jewellery_Demand_WGC_Quarterly_tonnes",
    "ETF_Net_Demand_WGC_Quarterly_tonnes", "CB_Total_Demand_WGC_Quarterly_tonnes",
]].copy()

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.12,
                    subplot_titles=["Quarterly Gold Return", "WGC Quarterly Demand Components"],
                    specs=[[{}], [{"secondary_y": False}]])
fig.add_trace(go.Bar(x=q_plot["Quarter"], y=q_plot["Gold_Quarterly_Return_pct"],
                     marker_color="#D4A017", name="Gold return"), row=1, col=1)
for variable, name, colour in [
    ("Jewellery_Demand_WGC_Quarterly_tonnes", "Jewellery", "#C65911"),
    ("ETF_Net_Demand_WGC_Quarterly_tonnes", "ETF", "#1F77B4"),
    ("CB_Total_Demand_WGC_Quarterly_tonnes", "CB total demand", "#7F6000"),
]:
    fig.add_trace(go.Scatter(x=q_plot["Quarter"], y=q_plot[variable], mode="lines+markers",
                             name=name, line=dict(color=colour, width=2)), row=2, col=1)
fig.add_hline(y=0, line_color="black", line_width=1, row=1, col=1)
fig.update_yaxes(title_text="Return (%)", row=1, col=1)
fig.update_yaxes(title_text="Tonnes", row=2, col=1)
fig.update_layout(title="Quarterly Gold Return and Physical/Investment Demand",
                  template="plotly_white", height=760, hovermode="x unified",
                  legend=dict(orientation="h", y=1.08, x=0.5, xanchor="center"))
fig.show()

## 5. Distributions and outlier diagnostics

The IQR rule identifies observations for investigation; it does not justify deletion. Event-driven gold flows can be economically genuine.

In [13]:
# CELL 13: Distribution plots for global CB changes and regional ETF flows

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=["Global IMF CB Holdings Changes", "Regional ETF Net Demand"],
                    horizontal_spacing=0.12)
fig.add_trace(go.Box(
    y=monthly["CB_Reported_Holdings_Change_Global_IMF_tonnes"],
    name="Global IMF CB", boxpoints="outliers", marker=dict(size=5),
), row=1, col=1)
for region in regions:
    temp = panel_df.loc[panel_df["Region"] == region]
    fig.add_trace(go.Box(
        x=temp["Region"], y=temp["ETF_Net_Demand_Regional_WGC_tonnes"],
        name=region, boxpoints="outliers", jitter=0.30, marker=dict(size=5),
    ), row=1, col=2)
for col in (1, 2): fig.add_hline(y=0, line_color="black", line_width=1, row=1, col=col)
fig.update_yaxes(title_text="Monthly change (tonnes)", row=1, col=1)
fig.update_yaxes(title_text="Monthly flow (tonnes)", row=1, col=2)
fig.update_layout(title="Distributions and Potential Outliers", template="plotly_white",
                  height=600, boxmode="group", legend_title="Region")
fig.show()

In [14]:
# CELL 14: IQR outlier tables — global CB and regional ETF

def identify_iqr_outliers(data, variable, group=None):
    outputs = []
    grouped = [("Global", data)] if group is None else data.groupby(group, observed=True)
    for name, subset in grouped:
        series = subset[variable].dropna()
        q1, q3 = series.quantile([0.25, 0.75])
        iqr = q3 - q1
        lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        keep = ["Month"] + ([group] if group else []) + [variable]
        flagged = subset.loc[(subset[variable] < lower) | (subset[variable] > upper), keep].copy()
        flagged["Group"] = name
        flagged["Lower_Bound"] = lower
        flagged["Upper_Bound"] = upper
        outputs.append(flagged)
    return pd.concat(outputs, ignore_index=True) if outputs else pd.DataFrame()

cb_outliers = identify_iqr_outliers(
    monthly, "CB_Reported_Holdings_Change_Global_IMF_tonnes"
)
etf_outliers = identify_iqr_outliers(
    panel_df, "ETF_Net_Demand_Regional_WGC_tonnes", group="Region"
)

print("Potential global IMF CB outliers:", len(cb_outliers))
display(cb_outliers.sort_values("CB_Reported_Holdings_Change_Global_IMF_tonnes"))
print("Potential regional ETF outliers:", len(etf_outliers))
display(etf_outliers.sort_values("ETF_Net_Demand_Regional_WGC_tonnes"))

Potential global IMF CB outliers: 5


,Month,CB_Reported_Holdings_Change_Global_IMF_tonnes,Group,Lower_Bound,Upper_Bound
4,2026-03-01,-90.0485,Global,-36.1073,102.7988
2,2023-04-01,-76.2656,Global,-36.1073,102.7988
3,2023-05-01,-44.8705,Global,-36.1073,102.7988
0,2019-06-01,107.6411,Global,-36.1073,102.7988
1,2021-03-01,184.6220,Global,-36.1073,102.7988


Potential regional ETF outliers: 15


,Month,Region,ETF_Net_Demand_Regional_WGC_tonnes,Group,Lower_Bound,Upper_Bound
10,2026-06-01,Asia,-17.5000,Asia,-12.3000,18.1000
14,2023-06-01,Others,-4.0000,Others,-2.6500,3.3500
13,2020-10-01,Others,-2.8000,Others,-2.6500,3.3500
12,2020-07-01,Others,3.4000,Others,-2.6500,3.3500
11,2020-03-01,Others,4.7000,Others,-2.6500,3.3500
3,2024-04-01,Asia,18.9000,Asia,-12.3000,18.1000
4,2024-10-01,Asia,23.4000,Asia,-12.3000,18.1000
8,2025-11-01,Asia,23.6000,Asia,-12.3000,18.1000
5,2025-02-01,Asia,24.4000,Asia,-12.3000,18.1000
7,2025-10-01,Asia,44.8000,Asia,-12.3000,18.1000


## 6. Stationarity before correlation

ADF tests the null of a unit root; KPSS tests the null of stationarity. We treat a variable as clearly stationary when ADF rejects and KPSS does not reject at 5%. Conflicting outcomes are explicitly retained as inconclusive rather than forced into a binary label.

In [15]:
# CELL 15: ADF and KPSS diagnostics for global monthly variables

stationarity_variables = [
    "Gold_Monthly_Return_pct",
    "ETF_Net_Demand_Global_WGC_tonnes",
    "CB_Reported_Holdings_Change_Global_IMF_tonnes",
    "US_CPI_Inflation_MoM_pct",
    "SP500_Monthly_Return_pct",
    "Broad_USD_Index_Avg",
    "US_10Y_Yield_MonthEnd_pct",
    "US_10Y_Real_Yield_MonthEnd_pct",
    "VIX_MonthEnd",
    "Term_Spread_10Y2Y_MonthEnd_pct",
    "USDINR_Monthly_Change_pct",
    "USDCNY_Monthly_Change_pct",
    "US_EPU_Monthly_Change_pct",
    "US_High_Yield_OAS_MonthEnd_pct",
    "Brent_MonthEnd_USD_per_Barrel",
    "Effective_Fed_Funds_Avg_pct",
]

def stationarity_test(series, name):
    x = pd.to_numeric(series, errors="coerce").dropna()
    if x.nunique() < 3 or len(x) < 20:
        return {"Variable": name, "N": len(x), "ADF_p": np.nan, "KPSS_p": np.nan,
                "Assessment": "Insufficient variation/sample"}
    adf_p = adfuller(x, autolag="AIC", regression="c")[1]
    try:
        kpss_p = kpss(x, regression="c", nlags="auto")[1]
    except (ValueError, OverflowError):
        kpss_p = np.nan
    if adf_p < 0.05 and (pd.isna(kpss_p) or kpss_p >= 0.05):
        assessment = "Stationary"
    elif adf_p >= 0.05 and pd.notna(kpss_p) and kpss_p < 0.05:
        assessment = "Non-stationary"
    else:
        assessment = "Inconclusive"
    return {"Variable": name, "N": len(x), "ADF_p": adf_p,
            "KPSS_p": kpss_p, "Assessment": assessment}

stationarity_results = pd.DataFrame([
    stationarity_test(monthly[v], v) for v in stationarity_variables
])
display(stationarity_results.sort_values(["Assessment", "ADF_p"]))

/tmp/ipykernel_4017/441173956.py:29: InterpolationWarning:

The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.


/tmp/ipykernel_4017/441173956.py:29: InterpolationWarning:

The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.


/tmp/ipykernel_4017/441173956.py:29: InterpolationWarning:

The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.


/tmp/ipykernel_4017/441173956.py:29: InterpolationWarning:

The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.


/tmp/ipykernel_4017/441173956.py:29: InterpolationWarning:

The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater

,Variable,N,ADF_p,KPSS_p,Assessment
5,Broad_USD_Index_Avg,91,0.0104,0.0100,Inconclusive
14,Brent_MonthEnd_USD_per_Barrel,91,0.1540,0.0538,Inconclusive
3,US_CPI_Inflation_MoM_pct,91,0.3616,0.1000,Inconclusive
9,Term_Spread_10Y2Y_MonthEnd_pct,91,0.5593,0.1000,Inconclusive
13,US_High_Yield_OAS_MonthEnd_pct,91,0.0822,0.0150,Non-stationary
15,Effective_Fed_Funds_Avg_pct,91,0.5871,0.0100,Non-stationary
6,US_10Y_Yield_MonthEnd_pct,91,0.9034,0.0100,Non-stationary
7,US_10Y_Real_Yield_MonthEnd_pct,91,0.9078,0.0100,Non-stationary
12,US_EPU_Monthly_Change_pct,91,0.0000,0.1000,Stationary
10,USDINR_Monthly_Change_pct,91,0.0000,0.1000,Stationary


In [16]:
from statsmodels.tools.sm_exceptions import InterpolationWarning
import warnings

warnings.filterwarnings("ignore", category=InterpolationWarning)

def stationarity_test(series, name):

    x = pd.to_numeric(series, errors="coerce").dropna()

    if x.nunique() < 3 or len(x) < 20:
        return {
            "Variable": name,
            "N": len(x),
            "ADF_p": np.nan,
            "KPSS_p": np.nan,
            "KPSS_p_Display": "–",
            "Assessment": "Insufficient sample/variation"
        }

    adf_result = adfuller(x, autolag="AIC", regression="c")
    adf_p = adf_result[1]

    kpss_result = kpss(x, regression="c", nlags="auto")
    kpss_p = kpss_result[1]

    if np.isclose(kpss_p, 0.10):
        kpss_display = "> 0.10"
    elif np.isclose(kpss_p, 0.01):
        kpss_display = "< 0.01"
    else:
        kpss_display = f"{kpss_p:.4f}"

    adf_rejects = adf_p < 0.05
    kpss_rejects = kpss_p < 0.05

    if adf_rejects and not kpss_rejects:
        assessment = "Stationary"
    elif not adf_rejects and kpss_rejects:
        assessment = "Non-stationary"
    elif adf_rejects and kpss_rejects:
        assessment = "Conflicting evidence"
    else:
        assessment = "Inconclusive / low power"

    return {
        "Variable": name,
        "N": len(x),
        "ADF_p": adf_p,
        "KPSS_p": kpss_p,
        "KPSS_p_Display": kpss_display,
        "Assessment": assessment
    }


stationarity_results = pd.DataFrame([
    stationarity_test(monthly[v], v)
    for v in stationarity_variables
])

display(
    stationarity_results[
        [
            "Variable",
            "N",
            "ADF_p",
            "KPSS_p_Display",
            "Assessment"
        ]
    ].sort_values(["Assessment", "ADF_p"])
)

,Variable,N,ADF_p,KPSS_p_Display,Assessment
5,Broad_USD_Index_Avg,91,0.0104,< 0.01,Conflicting evidence
14,Brent_MonthEnd_USD_per_Barrel,91,0.1540,0.0538,Inconclusive / low power
3,US_CPI_Inflation_MoM_pct,91,0.3616,> 0.10,Inconclusive / low power
9,Term_Spread_10Y2Y_MonthEnd_pct,91,0.5593,> 0.10,Inconclusive / low power
13,US_High_Yield_OAS_MonthEnd_pct,91,0.0822,0.0150,Non-stationary
15,Effective_Fed_Funds_Avg_pct,91,0.5871,< 0.01,Non-stationary
6,US_10Y_Yield_MonthEnd_pct,91,0.9034,< 0.01,Non-stationary
7,US_10Y_Real_Yield_MonthEnd_pct,91,0.9078,< 0.01,Non-stationary
12,US_EPU_Monthly_Change_pct,91,0.0000,> 0.10,Stationary
10,USDINR_Monthly_Change_pct,91,0.0000,> 0.10,Stationary


In [17]:
# CELL 19: Verify stationarity after transformations

transformed_variables = [
    "Gold_Return",
    "ETF_Flow",
    "CB_IMF_Change",
    "CPI_MoM",
    "SP500_Return",
    "USDINR_Change",
    "USDCNY_Change",
    "EPU_Change",
    "Broad_USD_Change",
    "US10Y_Change_pp",
    "Real10Y_Change_pp",
    "VIX_Change",
    "Term_Spread_Change_pp",
    "HY_OAS_Change_pp",
    "Brent_Return_pct",
    "Fed_Funds_Change_pp",
]

transformed_stationarity_results = pd.DataFrame([
    stationarity_test(
        stationary_df[variable],
        variable
    )
    for variable in transformed_variables
])

transformed_stationarity_results["Model_Ready"] = np.where(
    transformed_stationarity_results["Assessment"] == "Stationary",
    "Yes",
    "Review"
)

display(
    transformed_stationarity_results[
        [
            "Variable",
            "N",
            "ADF_p",
            "KPSS_p_Display",
            "Assessment",
            "Model_Ready"
        ]
    ].sort_values(
        ["Model_Ready", "Assessment", "ADF_p"],
        ascending=[False, True, True]
    )
)

NameError: name 'stationary_df' is not defined

In [18]:
# CPI: retain the original monthly inflation rate
stationary_df["CPI_MoM"] = monthly[
    "US_CPI_Inflation_MoM_pct"
]

# Fed funds: continuous monthly change in percentage points
stationary_df["Fed_Funds_Change_pp"] = monthly[
    "Effective_Fed_Funds_Avg_pct"
].diff()

# Categorical direction of Federal Reserve policy action
stationary_df["Fed_Policy_Action"] = np.select(
    [
        stationary_df["Fed_Funds_Change_pp"] > 0,
        stationary_df["Fed_Funds_Change_pp"] < 0
    ],
    [
         1,   # Rate hike
        -1    # Rate cut
    ],
    default=0  # No change
).astype(int)

NameError: name 'stationary_df' is not defined

In [ ]:
stationary_df["Fed_Hike_Dummy"] = (
    stationary_df["Fed_Funds_Change_pp"] > 0
).astype(int)

stationary_df["Fed_Cut_Dummy"] = (
    stationary_df["Fed_Funds_Change_pp"] < 0
).astype(int)

In [19]:
# CELL 21: Compare original and transformed stationarity results

transformation_map = pd.DataFrame({
    "Original_Variable": [
        "Gold_Monthly_Return_pct",
        "ETF_Net_Demand_Global_WGC_tonnes",
        "CB_Reported_Holdings_Change_Global_IMF_tonnes",
        "US_CPI_Inflation_MoM_pct",
        "SP500_Monthly_Return_pct",
        "USDINR_Monthly_Change_pct",
        "USDCNY_Monthly_Change_pct",
        "US_EPU_Monthly_Change_pct",
        "Broad_USD_Index_Avg",
        "US_10Y_Yield_MonthEnd_pct",
        "US_10Y_Real_Yield_MonthEnd_pct",
        "VIX_MonthEnd",
        "Term_Spread_10Y2Y_MonthEnd_pct",
        "US_High_Yield_OAS_MonthEnd_pct",
        "Brent_MonthEnd_USD_per_Barrel",
        "Effective_Fed_Funds_Avg_pct",
    ],

    "Transformed_Variable": [
        "Gold_Return",
        "ETF_Flow",
        "CB_IMF_Change",
        "CPI_MoM",
        "SP500_Return",
        "USDINR_Change",
        "USDCNY_Change",
        "EPU_Change",
        "Broad_USD_Change",
        "US10Y_Change_pp",
        "Real10Y_Change_pp",
        "VIX_Change",
        "Term_Spread_Change_pp",
        "HY_OAS_Change_pp",
        "Brent_Return_pct",
        "Fed_Funds_Change_pp",
    ],

    "Transformation": [
        "Already a monthly return",
        "Already a monthly flow",
        "Already an adjacent-month holdings change",
        "Retained as monthly inflation rate",
        "Already a monthly return",
        "Already a monthly percentage change",
        "Already a monthly percentage change",
        "Already a monthly percentage change",
        "First difference",
        "First difference in percentage points",
        "First difference in percentage points",
        "First difference in index points",
        "First difference in percentage points",
        "First difference in percentage points",
        "Monthly percentage return",
        "First difference in percentage points",
    ],

    "Primary_Model_Variable": [
        "Gold_Return",
        "ETF_Flow",
        "CB_IMF_Change",
        "CPI_MoM",
        "SP500_Return",
        "USDINR_Change",
        "USDCNY_Change",
        "EPU_Change",
        "Broad_USD_Change",
        "US10Y_Change_pp",
        "Real10Y_Change_pp",
        "VIX_Change",
        "Term_Spread_Change_pp",
        "HY_OAS_Change_pp",
        "Brent_Return_pct",
        "Fed_Funds_Change_pp",
    ],

    "Robustness_Variable": [
        "–",
        "Regional ETF flows",
        "WGC monthly reported CB change",
        "Structural-break or inflation-deviation measure",
        "–",
        "–",
        "–",
        "–",
        "Percentage change in broad USD index",
        "–",
        "–",
        "VIX level",
        "Term-spread level",
        "–",
        "–",
        "Fed_Hike_Dummy + Fed_Cut_Dummy",
    ]
})


# Merge original-variable stationarity results
comparison = (
    transformation_map
    .merge(
        stationarity_results[
            [
                "Variable",
                "ADF_p",
                "KPSS_p_Display",
                "Assessment"
            ]
        ].rename(columns={
            "Variable": "Original_Variable",
            "ADF_p": "Original_ADF_p",
            "KPSS_p_Display": "Original_KPSS_p",
            "Assessment": "Original_Assessment"
        }),
        on="Original_Variable",
        how="left"
    )
    .merge(
        transformed_stationarity_results[
            [
                "Variable",
                "ADF_p",
                "KPSS_p_Display",
                "Assessment"
            ]
        ].rename(columns={
            "Variable": "Transformed_Variable",
            "ADF_p": "Transformed_ADF_p",
            "KPSS_p_Display": "Transformed_KPSS_p",
            "Assessment": "Transformed_Assessment"
        }),
        on="Transformed_Variable",
        how="left"
    )
)


# CPI and Fed-funds changes are retained with qualifications
qualified_variables = {
    "CPI_MoM",
    "Fed_Funds_Change_pp"
}

comparison["Final_Decision"] = np.select(
    [
        comparison["Transformed_Assessment"].eq("Stationary"),

        comparison["Transformed_Variable"].isin(
            qualified_variables
        )
    ],
    [
        "Use in primary specification",

        "Use with stationarity qualification"
    ],
    default="Further investigation required"
)


# Explain the two qualified decisions
comparison["Decision_Note"] = np.select(
    [
        comparison["Transformed_Variable"].eq("CPI_MoM"),

        comparison["Transformed_Variable"].eq(
            "Fed_Funds_Change_pp"
        )
    ],
    [
        (
            "Retain because CPI_MoM is already an inflation-rate "
            "measure; avoid over-differencing. Test structural breaks "
            "as robustness."
        ),

        (
            "Retain the continuous policy-rate change. Use separate "
            "Fed_Hike_Dummy and Fed_Cut_Dummy variables in a robustness "
            "specification."
        )
    ],
    default="Stationarity tests support the selected representation."
)


display(
    comparison[
        [
            "Original_Variable",
            "Original_ADF_p",
            "Original_KPSS_p",
            "Original_Assessment",
            "Transformation",
            "Transformed_Variable",
            "Transformed_ADF_p",
            "Transformed_KPSS_p",
            "Transformed_Assessment",
            "Primary_Model_Variable",
            "Robustness_Variable",
            "Final_Decision",
            "Decision_Note"
        ]
    ]
)

NameError: name 'transformed_stationarity_results' is not defined

In [20]:
# CELL 16: Region-specific and Fisher-combined ADF tests for ETF demand

regional_adf = []
for region in regions:
    x = panel_df.loc[panel_df["Region"] == region, "ETF_Net_Demand_Regional_WGC_tonnes"].dropna()
    stat, pvalue, used_lag, nobs, *_ = adfuller(x, autolag="AIC", regression="c")
    regional_adf.append({"Region": region, "N": nobs, "ADF_Statistic": stat,
                         "ADF_p": pvalue, "Used_Lag": used_lag})
regional_adf = pd.DataFrame(regional_adf)
fisher_stat, fisher_p = combine_pvalues(regional_adf["ADF_p"], method="fisher")
display(regional_adf)
print(f"Fisher-combined ADF statistic: {fisher_stat:.4f}")
print(f"Fisher-combined p-value: {fisher_p:.6f}")
print("Note: Fisher combination assumes cross-sectional independence; use CIPS/CADF as the primary panel robustness test later.")

,Region,N,ADF_Statistic,ADF_p,Used_Lag
0,North America,90,-5.0970,0.0000,0
1,Europe,90,-6.5410,0.0000,0
2,Asia,81,-0.6459,0.8603,9
3,Others,90,-6.2317,0.0000,0


Fisher-combined ADF statistic: 93.2478
Fisher-combined p-value: 0.000000
Note: Fisher combination assumes cross-sectional independence; use CIPS/CADF as the primary panel robustness test later.


In [21]:
# Region-specific ADF and KPSS robustness analysis

regional_stationarity = []

for region in regions:

    x = (
        panel_df.loc[
            panel_df["Region"] == region,
            "ETF_Net_Demand_Regional_WGC_tonnes"
        ]
        .dropna()
    )

    # ADF with alternative lag-selection criteria
    adf_aic = adfuller(
        x,
        regression="c",
        autolag="AIC"
    )

    adf_bic = adfuller(
        x,
        regression="c",
        autolag="BIC"
    )

    # KPSS stationarity test
    with warnings.catch_warnings():
        warnings.simplefilter(
            "ignore",
            category=InterpolationWarning
        )

        kpss_result = kpss(
            x,
            regression="c",
            nlags="auto"
        )

    kpss_p = kpss_result[1]

    if np.isclose(kpss_p, 0.10):
        kpss_display = "> 0.10"
    elif np.isclose(kpss_p, 0.01):
        kpss_display = "< 0.01"
    else:
        kpss_display = f"{kpss_p:.4f}"

    regional_stationarity.append({
        "Region": region,
        "ADF_AIC_p": adf_aic[1],
        "ADF_AIC_Lag": adf_aic[2],
        "ADF_BIC_p": adf_bic[1],
        "ADF_BIC_Lag": adf_bic[2],
        "KPSS_p": kpss_p,
        "KPSS_p_Display": kpss_display
    })


regional_stationarity = pd.DataFrame(
    regional_stationarity
)

regional_stationarity["Assessment"] = np.select(
    [
        (
            (regional_stationarity["ADF_AIC_p"] < 0.05)
            & (regional_stationarity["KPSS_p"] >= 0.05)
        ),

        (
            (regional_stationarity["ADF_AIC_p"] >= 0.05)
            & (regional_stationarity["KPSS_p"] < 0.05)
        ),

        (
            (regional_stationarity["ADF_AIC_p"] < 0.05)
            & (regional_stationarity["KPSS_p"] < 0.05)
        )
    ],
    [
        "Stationary",
        "Non-stationary",
        "Conflicting evidence"
    ],
    default="Inconclusive / low power"
)

display(regional_stationarity)

,Region,ADF_AIC_p,ADF_AIC_Lag,ADF_BIC_p,ADF_BIC_Lag,KPSS_p,KPSS_p_Display,Assessment
0,North America,0.0000,0,0.0000,0,0.1000,> 0.10,Stationary
1,Europe,0.0000,0,0.0000,0,0.0493,0.0493,Conflicting evidence
2,Asia,0.8603,9,0.5990,8,0.0172,0.0172,Non-stationary
3,Others,0.0000,0,0.0000,0,0.1000,> 0.10,Stationary


In [22]:
# Structural-break unit-root test for regional ETF demand

from statsmodels.tsa.stattools import zivot_andrews

za_results = []

for region in ["Europe", "Asia"]:

    region_data = (
        panel_df.loc[
            panel_df["Region"] == region,
            ["Month", "ETF_Net_Demand_Regional_WGC_tonnes"]
        ]
        .dropna()
        .sort_values("Month")
        .reset_index(drop=True)
    )

    x = region_data[
        "ETF_Net_Demand_Regional_WGC_tonnes"
    ]

    za_stat, za_p, critical_values, used_lag, break_index = (
        zivot_andrews(
            x,
            trim=0.15,
            maxlag=None,
            regression="ct",
            autolag="AIC"
        )
    )

    break_month = region_data.loc[
        break_index,
        "Month"
    ]

    za_results.append({
        "Region": region,
        "ZA_Statistic": za_stat,
        "ZA_p": za_p,
        "Used_Lag": used_lag,
        "Estimated_Break_Month": break_month,
        "Critical_Value_1pct": critical_values["1%"],
        "Critical_Value_5pct": critical_values["5%"],
        "Critical_Value_10pct": critical_values["10%"],
        "Reject_Unit_Root_5pct": za_stat < critical_values["5%"]
    })


za_results = pd.DataFrame(za_results)

display(za_results)

,Region,ZA_Statistic,ZA_p,Used_Lag,Estimated_Break_Month,Critical_Value_1pct,Critical_Value_5pct,Critical_Value_10pct,Reject_Unit_Root_5pct
0,Europe,-7.9870,0.0010,0,2024-04-01,-5.5756,-5.0733,-4.8267,True
1,Asia,-3.4557,0.8320,9,2022-12-01,-5.5756,-5.0733,-4.8267,False


In [23]:
panel_df["Delta_ETF_Regional_tonnes"] = (
    panel_df
    .sort_values(["Region_ID", "Month"])
    .groupby("Region", observed=True)[
        "ETF_Net_Demand_Regional_WGC_tonnes"
    ]
    .diff()
)

In [24]:
panel_df["Delta_ETF_Regional_tonnes"] = (
    panel_df
    .sort_values(["Region_ID", "Month"])
    .groupby("Region", observed=True)[
        "ETF_Net_Demand_Regional_WGC_tonnes"
    ]
    .diff()
)

In [25]:
delta_etf_stationarity = []

for region in regions:

    x = panel_df.loc[
        panel_df["Region"] == region,
        "Delta_ETF_Regional_tonnes"
    ].dropna()

    adf_result = adfuller(
        x,
        regression="c",
        autolag="AIC"
    )

    with warnings.catch_warnings():
        warnings.simplefilter(
            "ignore",
            category=InterpolationWarning
        )
        kpss_result = kpss(
            x,
            regression="c",
            nlags="auto"
        )

    delta_etf_stationarity.append({
        "Region": region,
        "ADF_p": adf_result[1],
        "KPSS_p": kpss_result[1],
        "Stationary_5pct": (
            adf_result[1] < 0.05
            and kpss_result[1] >= 0.05
        )
    })

display(pd.DataFrame(delta_etf_stationarity))

,Region,ADF_p,KPSS_p,Stationary_5pct
0,North America,0.0000,0.1000,True
1,Europe,0.0000,0.0838,True
2,Asia,0.0000,0.1000,True
3,Others,0.0000,0.1000,True


In [26]:
# Sort panel correctly before constructing time-series transformations

panel_df = (
    panel_df
    .sort_values(["Region_ID", "Month"])
    .reset_index(drop=True)
)


# Level variable: regional monthly ETF net demand

panel_df["ETF_Regional_Level_tonnes"] = (
    panel_df["ETF_Net_Demand_Regional_WGC_tonnes"]
)


# First-difference variable: change from the preceding month

panel_df["ETF_Regional_First_Difference_tonnes"] = (
    panel_df
    .groupby("Region", observed=True)[
        "ETF_Regional_Level_tonnes"
    ]
    .diff()
)

In [27]:
# Model 1: ETF demand in levels
regional_etf_level_variable = "ETF_Regional_Level_tonnes"

# Model 2: First-differenced ETF demand
regional_etf_difference_variable = (
    "ETF_Regional_First_Difference_tonnes"
)

In [28]:
print(
    panel_df[
        "ETF_Regional_First_Difference_tonnes"
    ].isna().sum()
)

4


In [29]:
# CELL 17: Create transformed global and regional analysis variables


# ============================================================
# A. GLOBAL MONTHLY DATASET
# ============================================================

stationary_df = monthly[["Month"]].copy()


# ------------------------------------------------------------
# 1. Dependent variable
# ------------------------------------------------------------

stationary_df["Gold_Return"] = (
    monthly["Gold_Monthly_Return_pct"]
)


# ------------------------------------------------------------
# 2. Gold-demand variables
# ------------------------------------------------------------

# Global ETF demand in levels
stationary_df["ETF_Global_Level_tonnes"] = (
    monthly["ETF_Net_Demand_Global_WGC_tonnes"]
)

# First difference of global ETF demand
stationary_df["ETF_Global_First_Difference_tonnes"] = (
    stationary_df["ETF_Global_Level_tonnes"]
    .diff()
)

# IMF central-bank holdings change is already a flow/change variable
stationary_df["CB_IMF_Change_tonnes"] = (
    monthly[
        "CB_Reported_Holdings_Change_Global_IMF_tonnes"
    ]
)


# ------------------------------------------------------------
# 3. Variables already expressed as returns, inflation or change
# ------------------------------------------------------------

# Retain CPI MoM unchanged to avoid over-differencing inflation
stationary_df["CPI_MoM"] = (
    monthly["US_CPI_Inflation_MoM_pct"]
)

stationary_df["SP500_Return"] = (
    monthly["SP500_Monthly_Return_pct"]
)

stationary_df["USDINR_Change"] = (
    monthly["USDINR_Monthly_Change_pct"]
)

stationary_df["USDCNY_Change"] = (
    monthly["USDCNY_Monthly_Change_pct"]
)

stationary_df["EPU_Change"] = (
    monthly["US_EPU_Monthly_Change_pct"]
)


# ------------------------------------------------------------
# 4. Transform persistent level variables
# ------------------------------------------------------------

stationary_df["Broad_USD_First_Difference"] = (
    monthly["Broad_USD_Index_Avg"]
    .diff()
)

stationary_df["US10Y_First_Difference_pp"] = (
    monthly["US_10Y_Yield_MonthEnd_pct"]
    .diff()
)

stationary_df["Real10Y_First_Difference_pp"] = (
    monthly["US_10Y_Real_Yield_MonthEnd_pct"]
    .diff()
)

stationary_df["Term_Spread_First_Difference_pp"] = (
    monthly["Term_Spread_10Y2Y_MonthEnd_pct"]
    .diff()
)

stationary_df["HY_OAS_First_Difference_pp"] = (
    monthly["US_High_Yield_OAS_MonthEnd_pct"]
    .diff()
)

stationary_df["Brent_Return_pct"] = (
    monthly["Brent_MonthEnd_USD_per_Barrel"]
    .pct_change(fill_method=None)
    * 100
)


# ------------------------------------------------------------
# 5. VIX: retain level and first difference
# ------------------------------------------------------------

# VIX level passed the stationarity tests
stationary_df["VIX_Level"] = (
    monthly["VIX_MonthEnd"]
)

# Change in VIX captures sudden changes in risk sentiment
stationary_df["VIX_First_Difference_points"] = (
    stationary_df["VIX_Level"]
    .diff()
)


# ------------------------------------------------------------
# 6. Federal-funds variables
# ------------------------------------------------------------

stationary_df["Fed_Funds_First_Difference_pp"] = (
    monthly["Effective_Fed_Funds_Avg_pct"]
    .diff()
)

# Directional policy-action indicator:
#  1 = rate hike
#  0 = no change
# -1 = rate cut

stationary_df["Fed_Policy_Action"] = np.select(
    [
        stationary_df[
            "Fed_Funds_First_Difference_pp"
        ] > 0,

        stationary_df[
            "Fed_Funds_First_Difference_pp"
        ] < 0
    ],
    [
         1,
        -1
    ],
    default=0
).astype(int)


# Separate policy-action dummies for robustness specifications

stationary_df["Fed_Hike_Dummy"] = (
    stationary_df[
        "Fed_Funds_First_Difference_pp"
    ] > 0
).astype(int)

stationary_df["Fed_Cut_Dummy"] = (
    stationary_df[
        "Fed_Funds_First_Difference_pp"
    ] < 0
).astype(int)


# ============================================================
# B. REGIONAL ETF PANEL
# ============================================================

panel_df = (
    panel_df
    .sort_values(["Region_ID", "Month"])
    .reset_index(drop=True)
)


# Regional ETF demand in levels

panel_df["ETF_Regional_Level_tonnes"] = (
    panel_df[
        "ETF_Net_Demand_Regional_WGC_tonnes"
    ]
)


# First difference within each region

panel_df["ETF_Regional_First_Difference_tonnes"] = (
    panel_df
    .groupby(
        "Region",
        observed=True
    )["ETF_Regional_Level_tonnes"]
    .diff()
)


# Europe structural-break indicator identified by Zivot–Andrews

panel_df["Europe_Post_Apr2024_Break"] = (
    (
        panel_df["Region"] == "Europe"
    )
    &
    (
        panel_df["Month"]
        >= pd.Timestamp("2024-04-01")
    )
).astype(int)


# ============================================================
# C. VALIDATION
# ============================================================

print(
    "Missing observations in global ETF first difference:",
    stationary_df[
        "ETF_Global_First_Difference_tonnes"
    ].isna().sum()
)

print(
    "Missing observations in regional ETF first difference:",
    panel_df[
        "ETF_Regional_First_Difference_tonnes"
    ].isna().sum()
)

print("\nGlobal transformed dataset:")
display(stationary_df.head())

print("\nRegional ETF panel:")
display(
    panel_df[
        [
            "Month",
            "Region_ID",
            "Region",
            "ETF_Regional_Level_tonnes",
            "ETF_Regional_First_Difference_tonnes",
            "Europe_Post_Apr2024_Break"
        ]
    ].head(12)
)

Missing observations in global ETF first difference: 1
Missing observations in regional ETF first difference: 4

Global transformed dataset:


,Month,Gold_Return,ETF_Global_Level_tonnes,ETF_Global_First_Difference_tonnes,CB_IMF_Change_tonnes,CPI_MoM,SP500_Return,USDINR_Change,USDCNY_Change,EPU_Change,Broad_USD_First_Difference,US10Y_First_Difference_pp,Real10Y_First_Difference_pp,Term_Spread_First_Difference_pp,HY_OAS_First_Difference_pp,Brent_Return_pct,VIX_Level,VIX_First_Difference_points,Fed_Funds_First_Difference_pp,Fed_Policy_Action,Fed_Hike_Dummy,Fed_Cut_Dummy
0,2019-01-01,3.0211,71.8000,NaN,68.1649,-0.0815,7.7318,-0.1738,-1.4144,20.8947,NaN,NaN,NaN,NaN,NaN,NaN,16.5700,NaN,NaN,0,0,0
1,2019-02-01,-0.3113,-32.4000,-104.2000,18.2362,0.3001,2.8804,0.6560,-0.7318,-46.8241,-1.0669,0.1000,0.0000,0.0300,-0.4500,6.6893,14.7800,-1.7900,-0.0109,-1,0,1
2,2019-03-01,-0.2623,1.7000,34.1000,46.9922,0.3782,1.0953,-2.3665,-0.3673,31.1941,11.8118,-0.3200,-0.2500,-0.0700,0.1300,3.5741,13.7100,-1.0700,0.1238,1,1,0
3,2019-04-01,-0.4346,-56.9000,-58.6000,59.1245,0.3760,2.7428,-0.1190,0.0615,-29.5982,0.1202,0.1000,0.0300,0.1000,-0.3200,6.4483,13.1200,-0.5900,0.0194,1,1,0
4,2019-05-01,1.9042,-2.4000,54.5000,43.5958,0.0247,-5.8716,0.5416,2.0219,24.1066,-3.9880,-0.3700,-0.1600,-0.0500,0.8600,-11.4148,18.7100,5.5900,-0.1345,-1,0,1



Regional ETF panel:


,Month,Region_ID,Region,ETF_Regional_Level_tonnes,ETF_Regional_First_Difference_tonnes,Europe_Post_Apr2024_Break
0,2019-01-01,1.0000,North America,53.0000,NaN,0
1,2019-02-01,1.0000,North America,-29.0000,-82.0000,0
2,2019-03-01,1.0000,North America,2.5000,31.5000,0
3,2019-04-01,1.0000,North America,-46.0000,-48.5000,0
4,2019-05-01,1.0000,North America,-13.7000,32.3000,0
5,2019-06-01,1.0000,North America,65.0000,78.7000,0
6,2019-07-01,1.0000,North America,43.0000,-22.0000,0
7,2019-08-01,1.0000,North America,78.0000,35.0000,0
8,2019-09-01,1.0000,North America,64.4000,-13.6000,0
9,2019-10-01,1.0000,North America,13.2000,-51.2000,0


In [30]:
# CELL 18: Pearson and Spearman correlations
# for ETF-level and ETF-first-difference specifications


# ============================================================
# A. VARIABLES COMMON TO BOTH SPECIFICATIONS
# ============================================================

common_correlation_variables = [
    "Gold_Return",
    "CB_IMF_Change_tonnes",
    "CPI_MoM",
    "SP500_Return",
    "USDINR_Change",
    "USDCNY_Change",
    "EPU_Change",
    "Broad_USD_First_Difference",
    "US10Y_First_Difference_pp",
    "Real10Y_First_Difference_pp",
    "VIX_First_Difference_points",
    "Term_Spread_First_Difference_pp",
    "HY_OAS_First_Difference_pp",
    "Brent_Return_pct",
    "Fed_Funds_First_Difference_pp",
]


# ============================================================
# B. TWO SEPARATE CORRELATION SPECIFICATIONS
# ============================================================

correlation_specifications = {

    "ETF Level Specification": [
        "ETF_Global_Level_tonnes",
        *common_correlation_variables
    ],

    "ETF First-Difference Specification": [
        "ETF_Global_First_Difference_tonnes",
        *common_correlation_variables
    ]
}


correlation_results = {}
gold_correlation_tables = {}


# ============================================================
# C. CORRELATION FUNCTION
# ============================================================

def create_correlation_analysis(
    data,
    variables,
    specification_name
):

    corr_data = (
        data[variables]
        .replace([np.inf, -np.inf], np.nan)
        .copy()
    )

    pearson_corr = corr_data.corr(
        method="pearson",
        min_periods=20
    )

    spearman_corr = corr_data.corr(
        method="spearman",
        min_periods=20
    )

    # Pairwise observation-count matrix
    available = corr_data.notna().astype(int)

    pairwise_n = (
        available.T
        @ available
    )

    # --------------------------------------------------------
    # Plotly heatmaps
    # --------------------------------------------------------

    fig = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=[
            "Pearson Correlation",
            "Spearman Correlation"
        ],
        horizontal_spacing=0.12
    )

    for matrix, col in [
        (pearson_corr, 1),
        (spearman_corr, 2)
    ]:

        fig.add_trace(
            go.Heatmap(
                z=matrix.values,
                x=matrix.columns,
                y=matrix.index,
                zmin=-1,
                zmax=1,
                zmid=0,
                colorscale="RdBu_r",

                colorbar=dict(
                    title="Correlation",
                    x=0.45 if col == 1 else 1.02,
                    len=0.85
                ),

                customdata=pairwise_n.values,

                hovertemplate=(
                    "%{y} vs %{x}<br>"
                    "Correlation: %{z:.3f}<br>"
                    "Pairwise N: %{customdata:.0f}"
                    "<extra></extra>"
                )
            ),
            row=1,
            col=col
        )

    fig.update_layout(
        title={
            "text": (
                f"{specification_name}: "
                "Stationarity-Oriented Correlation Matrices"
            ),
            "x": 0.5
        },
        template="plotly_white",
        height=800,
        width=1550
    )

    fig.update_xaxes(
        tickangle=55,
        tickfont=dict(size=9)
    )

    fig.update_yaxes(
        tickfont=dict(size=9)
    )

    fig.show()

    # --------------------------------------------------------
    # Correlations specifically with gold returns
    # --------------------------------------------------------

    gold_correlations = pd.DataFrame({
        "Pearson_with_Gold_Return":
            pearson_corr["Gold_Return"],

        "Spearman_with_Gold_Return":
            spearman_corr["Gold_Return"],

        "Pairwise_N":
            pairwise_n["Gold_Return"]
    })

    gold_correlations = (
        gold_correlations
        .drop(index="Gold_Return")
        .assign(
            Absolute_Spearman=lambda x:
                x["Spearman_with_Gold_Return"].abs()
        )
        .sort_values(
            "Absolute_Spearman",
            ascending=False
        )
        .drop(columns="Absolute_Spearman")
    )

    return {
        "Correlation_Data": corr_data,
        "Pearson": pearson_corr,
        "Spearman": spearman_corr,
        "Pairwise_N": pairwise_n,
        "Gold_Correlations": gold_correlations
    }


# ============================================================
# D. RUN BOTH SPECIFICATIONS
# ============================================================

for specification_name, variables in (
    correlation_specifications.items()
):

    print("=" * 90)
    print(specification_name)
    print("=" * 90)

    results = create_correlation_analysis(
        data=stationary_df,
        variables=variables,
        specification_name=specification_name
    )

    correlation_results[
        specification_name
    ] = results

    gold_correlation_tables[
        specification_name
    ] = results["Gold_Correlations"]

    print(
        "\nCorrelations with monthly gold returns:"
    )

    display(
        results["Gold_Correlations"]
    )

ETF Level Specification



Correlations with monthly gold returns:


,Pearson_with_Gold_Return,Spearman_with_Gold_Return,Pairwise_N
ETF_Global_Level_tonnes,0.5612,0.5715,91
Real10Y_First_Difference_pp,-0.2896,-0.3672,90
US10Y_First_Difference_pp,-0.2557,-0.3330,90
USDCNY_Change,-0.2688,-0.2493,91
EPU_Change,0.1965,0.1984,91
Broad_USD_First_Difference,-0.0656,-0.1909,90
CB_IMF_Change_tonnes,-0.1647,-0.1759,91
USDINR_Change,-0.0866,-0.1657,91
Brent_Return_pct,-0.0554,-0.1633,90
VIX_First_Difference_points,0.0829,0.1382,90


ETF First-Difference Specification



Correlations with monthly gold returns:


,Pearson_with_Gold_Return,Spearman_with_Gold_Return,Pairwise_N
ETF_Global_First_Difference_tonnes,0.3497,0.4049,90
Real10Y_First_Difference_pp,-0.2896,-0.3672,90
US10Y_First_Difference_pp,-0.2557,-0.3330,90
USDCNY_Change,-0.2688,-0.2493,91
EPU_Change,0.1965,0.1984,91
Broad_USD_First_Difference,-0.0656,-0.1909,90
CB_IMF_Change_tonnes,-0.1647,-0.1759,91
USDINR_Change,-0.0866,-0.1657,91
Brent_Return_pct,-0.0554,-0.1633,90
VIX_First_Difference_points,0.0829,0.1382,90


In [31]:
# ============================================================
# E. COMPARE ETF-LEVEL AND ETF-DIFFERENCE CORRELATIONS
# ============================================================

etf_correlation_comparison = pd.DataFrame({
    "Specification": [
        "ETF level",
        "ETF first difference"
    ],

    "ETF_Variable": [
        "ETF_Global_Level_tonnes",
        "ETF_Global_First_Difference_tonnes"
    ],

    "Pearson_with_Gold_Return": [
        correlation_results[
            "ETF Level Specification"
        ]["Pearson"].loc[
            "ETF_Global_Level_tonnes",
            "Gold_Return"
        ],

        correlation_results[
            "ETF First-Difference Specification"
        ]["Pearson"].loc[
            "ETF_Global_First_Difference_tonnes",
            "Gold_Return"
        ]
    ],

    "Spearman_with_Gold_Return": [
        correlation_results[
            "ETF Level Specification"
        ]["Spearman"].loc[
            "ETF_Global_Level_tonnes",
            "Gold_Return"
        ],

        correlation_results[
            "ETF First-Difference Specification"
        ]["Spearman"].loc[
            "ETF_Global_First_Difference_tonnes",
            "Gold_Return"
        ]
    ],

    "Pairwise_N": [
        correlation_results[
            "ETF Level Specification"
        ]["Pairwise_N"].loc[
            "ETF_Global_Level_tonnes",
            "Gold_Return"
        ],

        correlation_results[
            "ETF First-Difference Specification"
        ]["Pairwise_N"].loc[
            "ETF_Global_First_Difference_tonnes",
            "Gold_Return"
        ]
    ]
})

display(etf_correlation_comparison)

,Specification,ETF_Variable,Pearson_with_Gold_Return,Spearman_with_Gold_Return,Pairwise_N
0,ETF level,ETF_Global_Level_tonnes,0.5612,0.5715,91
1,ETF first difference,ETF_Global_First_Difference_tonnes,0.3497,0.4049,90


## Interpretation boundary and next stage

- EDA correlations are descriptive, not causal estimates.
- The regional ETF panel supports region fixed effects with **North America as the base region** and annual/year dummies if required.
- Monthly time fixed effects would absorb gold returns, central-bank variables and all global macro variables because they are identical across regions within a month.
- Before panel estimation, proceed with CIPS/CADF robustness, multicollinearity, FE-versus-RE/Hausman, serial-correlation, heteroskedasticity, cross-sectional-dependence, specification, leverage/outlier and endogeneity diagnostics.
- For global gold-return causality, the one-row-per-month dataset is the primary dataset; the regional ETF panel answers a different cross-sectional question.

# TIME SERIES

In [32]:
# CELL 19: Prepare Model 1
# Combined global ETF + IMF central-bank demand channel


# ============================================================
# A. DEPENDENT VARIABLE
# ============================================================

model_1_df = monthly[["Month"]].copy()

model_1_df["Gold_Return"] = (
    monthly["Gold_Monthly_Return_pct"]
)


# ============================================================
# B. MODEL 1 DEMAND-CHANNEL VARIABLE
# ============================================================

# Combined global ETF demand and IMF-reported CB holdings change

model_1_df["ETF_plus_IMF_CB_tonnes"] = (
    monthly["ETF_plus_CB_IMF_tonnes"]
)


# Reconciliation check

model_1_df["ETF_plus_IMF_CB_Recalculated"] = (
    monthly["ETF_Net_Demand_Global_WGC_tonnes"]
    +
    monthly[
        "CB_Reported_Holdings_Change_Global_IMF_tonnes"
    ]
)

reconciliation_error = (
    model_1_df["ETF_plus_IMF_CB_tonnes"]
    -
    model_1_df["ETF_plus_IMF_CB_Recalculated"]
)

print(
    "Maximum ETF + IMF CB reconciliation error:",
    reconciliation_error.abs().max()
)

assert reconciliation_error.abs().max() < 1e-8

model_1_df = model_1_df.drop(
    columns="ETF_plus_IMF_CB_Recalculated"
)


# ============================================================
# C. MACRO-FINANCIAL CONTROLS
# ============================================================

model_1_df["CPI_MoM"] = (
    monthly["US_CPI_Inflation_MoM_pct"]
)

model_1_df["Broad_USD_Change"] = (
    monthly["Broad_USD_Index_Avg"]
    .diff()
)

model_1_df["US10Y_Change_pp"] = (
    monthly["US_10Y_Yield_MonthEnd_pct"]
    .diff()
)

model_1_df["Real10Y_Change_pp"] = (
    monthly["US_10Y_Real_Yield_MonthEnd_pct"]
    .diff()
)

model_1_df["Term_Spread_Change_pp"] = (
    monthly["Term_Spread_10Y2Y_MonthEnd_pct"]
    .diff()
)

model_1_df["Fed_Funds_Change_pp"] = (
    monthly["Effective_Fed_Funds_Avg_pct"]
    .diff()
)

model_1_df["USDINR_Change"] = (
    monthly["USDINR_Monthly_Change_pct"]
)

model_1_df["USDCNY_Change"] = (
    monthly["USDCNY_Monthly_Change_pct"]
)


# ============================================================
# D. MARKET-RISK CONTROLS
# ============================================================

model_1_df["SP500_Return"] = (
    monthly["SP500_Monthly_Return_pct"]
)

model_1_df["VIX_Change"] = (
    monthly["VIX_MonthEnd"]
    .diff()
)

model_1_df["EPU_Change"] = (
    monthly["US_EPU_Monthly_Change_pct"]
)

model_1_df["HY_OAS_Change_pp"] = (
    monthly["US_High_Yield_OAS_MonthEnd_pct"]
    .diff()
)

model_1_df["Brent_Return_pct"] = (
    monthly["Brent_MonthEnd_USD_per_Barrel"]
    .pct_change(fill_method=None)
    * 100
)


# ============================================================
# E. STRUCTURAL CONTROLS
# ============================================================

model_1_df["Post_2022_Dummy"] = (
    monthly["Post_2022_Dummy"]
)

model_1_df["Post_2025_Dummy"] = (
    monthly["Post_2025_Dummy"]
)


# ============================================================
# F. FINAL MODEL-1 CONTROL LIST
# ============================================================

model_1_controls = [
    "CPI_MoM",
    "Broad_USD_Change",
    "US10Y_Change_pp",
    "Real10Y_Change_pp",
    "Term_Spread_Change_pp",
    "Fed_Funds_Change_pp",
    "USDINR_Change",
    "USDCNY_Change",
    "SP500_Return",
    "VIX_Change",
    "EPU_Change",
    "HY_OAS_Change_pp",
    "Brent_Return_pct",
    "Post_2022_Dummy",
    "Post_2025_Dummy",
]


# ============================================================
# G. DATA VALIDATION
# ============================================================

model_1_variables = [
    "Gold_Return",
    "ETF_plus_IMF_CB_tonnes",
    *model_1_controls
]

model_1_missing = pd.DataFrame({
    "Variable": model_1_variables,
    "Available": [
        model_1_df[v].notna().sum()
        for v in model_1_variables
    ],
    "Missing": [
        model_1_df[v].isna().sum()
        for v in model_1_variables
    ],
    "Missing_Percent": [
        100 * model_1_df[v].isna().mean()
        for v in model_1_variables
    ]
})

display(model_1_missing)

print("Original observations:", len(model_1_df))

model_1_complete = (
    model_1_df[
        [
            "Month",
            *model_1_variables
        ]
    ]
    .dropna()
    .reset_index(drop=True)
)

print(
    "Complete observations available for Model 1:",
    len(model_1_complete)
)

print(
    "Model 1 period:",
    model_1_complete["Month"].min(),
    "to",
    model_1_complete["Month"].max()
)

display(model_1_complete.head())

Maximum ETF + IMF CB reconciliation error: 0.0


,Variable,Available,Missing,Missing_Percent
0,Gold_Return,91,0,0.0000
1,ETF_plus_IMF_CB_tonnes,91,0,0.0000
2,CPI_MoM,91,0,0.0000
3,Broad_USD_Change,90,1,1.0989
4,US10Y_Change_pp,90,1,1.0989
5,Real10Y_Change_pp,90,1,1.0989
6,Term_Spread_Change_pp,90,1,1.0989
7,Fed_Funds_Change_pp,90,1,1.0989
8,USDINR_Change,91,0,0.0000
9,USDCNY_Change,91,0,0.0000


Original observations: 91
Complete observations available for Model 1: 90
Model 1 period: 2019-02-01 00:00:00 to 2026-07-01 00:00:00


,Month,Gold_Return,ETF_plus_IMF_CB_tonnes,CPI_MoM,Broad_USD_Change,US10Y_Change_pp,Real10Y_Change_pp,Term_Spread_Change_pp,Fed_Funds_Change_pp,USDINR_Change,USDCNY_Change,SP500_Return,VIX_Change,EPU_Change,HY_OAS_Change_pp,Brent_Return_pct,Post_2022_Dummy,Post_2025_Dummy
0,2019-02-01,-0.3113,-14.1638,0.3001,-1.0669,0.1000,0.0000,0.0300,-0.0109,0.6560,-0.7318,2.8804,-1.7900,-46.8241,-0.4500,6.6893,0.0000,0.0000
1,2019-03-01,-0.2623,48.6922,0.3782,11.8118,-0.3200,-0.2500,-0.0700,0.1238,-2.3665,-0.3673,1.0953,-1.0700,31.1941,0.1300,3.5741,0.0000,0.0000
2,2019-04-01,-0.4346,2.2245,0.3760,0.1202,0.1000,0.0300,0.1000,0.0194,-0.1190,0.0615,2.7428,-0.5900,-29.5982,-0.3200,6.4483,0.0000,0.0000
3,2019-05-01,1.9042,41.1958,0.0247,-3.9880,-0.3700,-0.1600,-0.0500,-0.1345,0.5416,2.0219,-5.8716,5.5900,24.1066,0.8600,-11.4148,0.0000,0.0000
4,2019-06-01,6.5775,234.4411,-0.0325,4.5113,-0.1400,-0.0900,0.0600,0.0868,-0.5657,0.6687,7.1894,-3.6300,22.5055,-0.5200,3.1943,0.0000,0.0000


In [33]:
# CELL 20: Stationarity test for Model 1 demand channel

model_1_demand_stationarity = pd.DataFrame([
    stationarity_test(
        model_1_df["ETF_plus_IMF_CB_tonnes"],
        "ETF_plus_IMF_CB_tonnes"
    )
])

display(
    model_1_demand_stationarity[
        [
            "Variable",
            "N",
            "ADF_p",
            "KPSS_p_Display",
            "Assessment"
        ]
    ]
)

,Variable,N,ADF_p,KPSS_p_Display,Assessment
0,ETF_plus_IMF_CB_tonnes,91,0.0011,> 0.10,Stationary


In [34]:
# CELL 21: Correct the Broad USD Index using official FRED DTWEXBGS data

fred_usd_url = (
    "https://fred.stlouisfed.org/graph/fredgraph.csv"
    "?id=DTWEXBGS"
    "&cosd=2018-12-01"
    "&coed=2026-07-31"
)

usd_daily = pd.read_csv(fred_usd_url)

usd_daily["observation_date"] = pd.to_datetime(
    usd_daily["observation_date"]
)

usd_daily["DTWEXBGS"] = pd.to_numeric(
    usd_daily["DTWEXBGS"],
    errors="coerce"
)

usd_daily = usd_daily.dropna(
    subset=["DTWEXBGS"]
)


# Convert daily observations into monthly averages

usd_monthly_corrected = (
    usd_daily
    .assign(
        Month=lambda x:
            x["observation_date"]
            .dt.to_period("M")
            .dt.to_timestamp()
    )
    .groupby("Month", as_index=False)
    .agg(
        Broad_USD_Index_Avg_Corrected=(
            "DTWEXBGS",
            "mean"
        )
    )
)


# Compare existing and corrected observations

usd_validation = (
    monthly[
        [
            "Month",
            "Broad_USD_Index_Avg"
        ]
    ]
    .merge(
        usd_monthly_corrected,
        on="Month",
        how="left",
        validate="one_to_one"
    )
)

usd_validation["Difference"] = (
    usd_validation["Broad_USD_Index_Avg"]
    -
    usd_validation[
        "Broad_USD_Index_Avg_Corrected"
    ]
)

display(
    usd_validation.loc[
        usd_validation["Difference"].abs() > 0.01
    ]
)

,Month,Broad_USD_Index_Avg,Broad_USD_Index_Avg_Corrected,Difference
0,2019-01-01,104.0386,114.4425,-10.4039
1,2019-02-01,102.9717,114.4130,-11.4413
4,2019-05-01,110.9158,115.9574,-5.0416
6,2019-07-01,110.0742,115.0776,-5.0034
8,2019-09-01,111.7371,117.3239,-5.5869
...,...,...,...,...
86,2026-03-01,121.0350,119.9199,1.1151
87,2026-04-01,118.6710,119.0363,-0.3653
88,2026-05-01,118.8783,118.7792,0.0991
89,2026-06-01,120.9248,120.0834,0.8414


In [35]:
# Replace the workbook-derived Broad USD observations

monthly = (
    monthly
    .drop(columns="Broad_USD_Index_Avg")
    .merge(
        usd_monthly_corrected.rename(columns={
            "Broad_USD_Index_Avg_Corrected":
                "Broad_USD_Index_Avg"
        }),
        on="Month",
        how="left",
        validate="one_to_one"
    )
    .sort_values("Month")
    .reset_index(drop=True)
)


# Reconstruct the stationary monthly change

monthly["Broad_USD_Change"] = (
    monthly["Broad_USD_Index_Avg"]
    .diff()
)


# Update Model 1

model_1_df["Broad_USD_Change"] = (
    monthly["Broad_USD_Change"]
)

In [36]:
print(
    "Missing corrected USD observations:",
    monthly["Broad_USD_Index_Avg"].isna().sum()
)

print(
    "Largest absolute monthly USD-index change:",
    monthly["Broad_USD_Change"].abs().max()
)

display(
    monthly[
        [
            "Month",
            "Broad_USD_Index_Avg",
            "Broad_USD_Change"
        ]
    ].head(15)
)

Missing corrected USD observations: 0
Largest absolute monthly USD-index change: 4.312198803827741


,Month,Broad_USD_Index_Avg,Broad_USD_Change
0,2019-01-01,114.4425,NaN
1,2019-02-01,114.4130,-0.0295
2,2019-03-01,114.7835,0.3705
3,2019-04-01,114.9038,0.1202
4,2019-05-01,115.9574,1.0536
5,2019-06-01,115.4271,-0.5303
6,2019-07-01,115.0776,-0.3495
7,2019-08-01,117.1004,2.0228
8,2019-09-01,117.3239,0.2235
9,2019-10-01,116.7463,-0.5777


In [37]:
# CELL 22: Replace the inconsistent Broad USD series

monthly = (
    monthly
    .merge(
        usd_monthly_corrected,
        on="Month",
        how="left",
        validate="one_to_one"
    )
    .sort_values("Month")
    .reset_index(drop=True)
)


# Preserve the previous workbook values

monthly["Broad_USD_Index_Avg_Previous"] = (
    monthly["Broad_USD_Index_Avg"]
)


# Replace with the consistently defined official monthly average

monthly["Broad_USD_Index_Avg"] = (
    monthly[
        "Broad_USD_Index_Avg_Corrected"
    ]
)

monthly = monthly.drop(
    columns="Broad_USD_Index_Avg_Corrected"
)


# Verify complete coverage

assert monthly["Broad_USD_Index_Avg"].notna().all()


# Construct the corrected stationary transformation

monthly["Broad_USD_Change"] = (
    monthly["Broad_USD_Index_Avg"]
    .diff()
)

In [38]:
# Update the transformed global dataset

stationary_df["Broad_USD_First_Difference"] = (
    monthly["Broad_USD_Index_Avg"]
    .diff()
)


# Update Model 1

model_1_df["Broad_USD_Change"] = (
    monthly["Broad_USD_Index_Avg"]
    .diff()
)

In [39]:
model_1_variables = [
    "Gold_Return",
    "ETF_plus_IMF_CB_tonnes",
    *model_1_controls
]

model_1_complete = (
    model_1_df[
        [
            "Month",
            *model_1_variables
        ]
    ]
    .dropna()
    .reset_index(drop=True)
)

print(
    "Complete Model 1 observations:",
    len(model_1_complete)
)

print(
    "Period:",
    model_1_complete["Month"].min(),
    "to",
    model_1_complete["Month"].max()
)

Complete Model 1 observations: 90
Period: 2019-02-01 00:00:00 to 2026-07-01 00:00:00


In [40]:
corrected_usd_stationarity = pd.DataFrame([
    stationarity_test(
        monthly["Broad_USD_Index_Avg"],
        "Broad_USD_Index_Avg_Corrected"
    ),

    stationarity_test(
        monthly["Broad_USD_Change"],
        "Broad_USD_Change_Corrected"
    )
])

display(
    corrected_usd_stationarity[
        [
            "Variable",
            "N",
            "ADF_p",
            "KPSS_p_Display",
            "Assessment"
        ]
    ]
)

,Variable,N,ADF_p,KPSS_p_Display,Assessment
0,Broad_USD_Index_Avg_Corrected,91,0.1044,< 0.01,Non-stationary
1,Broad_USD_Change_Corrected,90,0.0000,> 0.10,Stationary


In [41]:
usd_change_summary = (
    monthly["Broad_USD_Change"]
    .describe()
    .to_frame("Broad_USD_Change")
)

display(usd_change_summary)

display(
    monthly.nlargest(
        10,
        "Broad_USD_Change"
    )[
        [
            "Month",
            "Broad_USD_Index_Avg",
            "Broad_USD_Change"
        ]
    ]
)

,Broad_USD_Change
count,90.0000
mean,0.0684
std,1.4375
min,-2.9384
25%,-0.7683
50%,-0.0345
75%,1.0218
max,4.3122


,Month,Broad_USD_Index_Avg,Broad_USD_Change
14,2020-03-01,121.0227,4.3122
44,2022-09-01,125.5946,3.2768
42,2022-07-01,122.7512,2.7255
70,2024-11-01,126.3113,2.7160
40,2022-05-01,119.6962,2.5274
15,2020-04-01,123.2803,2.2576
7,2019-08-01,117.1004,2.0228
86,2026-03-01,119.9199,2.0138
45,2022-10-01,127.4806,1.8860
57,2023-10-01,123.4971,1.8516


In [42]:
# Finalize corrected Broad USD variables safely

if "Broad_USD_Index_Avg_Corrected" in monthly.columns:
    monthly["Broad_USD_Index_Avg"] = monthly[
        "Broad_USD_Index_Avg_Corrected"
    ]

elif "Broad_USD_Index_Avg" not in monthly.columns:
    raise KeyError(
        "Neither Broad_USD_Index_Avg nor "
        "Broad_USD_Index_Avg_Corrected exists."
    )

# Recalculate first difference from the corrected canonical series
monthly = monthly.sort_values("Month").reset_index(drop=True)

monthly["Broad_USD_Change"] = (
    monthly["Broad_USD_Index_Avg"].diff()
)

# Remove temporary columns only when they exist
monthly.drop(
    columns=[
        "Broad_USD_Index_Avg_Corrected",
        "Broad_USD_Change_Corrected",
    ],
    errors="ignore",
    inplace=True
)

print("USD level column:", "Broad_USD_Index_Avg")
print("USD change column:", "Broad_USD_Change")

display(
    monthly[
        ["Month", "Broad_USD_Index_Avg", "Broad_USD_Change"]
    ].head()
)

USD level column: Broad_USD_Index_Avg
USD change column: Broad_USD_Change


,Month,Broad_USD_Index_Avg,Broad_USD_Change
0,2019-01-01,114.4425,NaN
1,2019-02-01,114.4130,-0.0295
2,2019-03-01,114.7835,0.3705
3,2019-04-01,114.9038,0.1202
4,2019-05-01,115.9574,1.0536


In [43]:
# Finalize corrected Broad USD variables safely

if "Broad_USD_Index_Avg_Corrected" in monthly.columns:
    monthly["Broad_USD_Index_Avg"] = monthly[
        "Broad_USD_Index_Avg_Corrected"
    ]

elif "Broad_USD_Index_Avg" not in monthly.columns:
    raise KeyError(
        "Neither Broad_USD_Index_Avg nor "
        "Broad_USD_Index_Avg_Corrected exists."
    )

# Recalculate first difference from the corrected canonical series
monthly = monthly.sort_values("Month").reset_index(drop=True)

monthly["Broad_USD_Change"] = (
    monthly["Broad_USD_Index_Avg"].diff()
)

# Remove temporary columns only when they exist
monthly.drop(
    columns=[
        "Broad_USD_Index_Avg_Corrected",
        "Broad_USD_Change_Corrected",
    ],
    errors="ignore",
    inplace=True
)

print("USD level column:", "Broad_USD_Index_Avg")
print("USD change column:", "Broad_USD_Change")

display(
    monthly[
        ["Month", "Broad_USD_Index_Avg", "Broad_USD_Change"]
    ].head()
)

USD level column: Broad_USD_Index_Avg
USD change column: Broad_USD_Change


,Month,Broad_USD_Index_Avg,Broad_USD_Change
0,2019-01-01,114.4425,NaN
1,2019-02-01,114.4130,-0.0295
2,2019-03-01,114.7835,0.3705
3,2019-04-01,114.9038,0.1202
4,2019-05-01,115.9574,1.0536


In [44]:
stationary_df["Broad_USD_Change"] = monthly[
    "Broad_USD_Change"
].to_numpy()

model_1_df["Broad_USD_Change"] = monthly[
    "Broad_USD_Change"
].to_numpy()

model_1_complete = (
    model_1_df
    .dropna()
    .sort_values("Month")
    .reset_index(drop=True)
)

print("Complete observations:", len(model_1_complete))
print(
    "Analysis period:",
    model_1_complete["Month"].min(),
    "to",
    model_1_complete["Month"].max()
)

Complete observations: 90
Analysis period: 2019-02-01 00:00:00 to 2026-07-01 00:00:00


In [45]:
# CELL 22: Model 1 ADL lag selection using BIC and AIC

import statsmodels.api as sm
import pandas as pd
import numpy as np

dependent = "Gold_Return"
demand_variable = "ETF_plus_IMF_CB_tonnes"

control_variables = [
    "CPI_MoM",
    "Broad_USD_Change",
    "US10Y_Change_pp",
    "Real10Y_Change_pp",
    "Term_Spread_Change_pp",
    "Fed_Funds_Change_pp",
    "USDINR_Change",
    "USDCNY_Change",
    "SP500_Return",
    "VIX_Change",
    "EPU_Change",
    "HY_OAS_Change_pp",
    "Brent_Return_pct",
    "Post_2022_Dummy",
    "Post_2025_Dummy",
]

max_lag = 3
adl_df = model_1_complete.copy()

# Create all lags before selecting the common estimation sample
for lag in range(1, max_lag + 1):
    adl_df[f"{dependent}_L{lag}"] = adl_df[dependent].shift(lag)
    adl_df[f"{demand_variable}_L{lag}"] = (
        adl_df[demand_variable].shift(lag)
    )

# Common sample for every candidate model
required_columns = (
    [dependent, demand_variable]
    + control_variables
    + [f"{dependent}_L{i}" for i in range(1, max_lag + 1)]
    + [f"{demand_variable}_L{i}" for i in range(1, max_lag + 1)]
)

adl_common = (
    adl_df
    .dropna(subset=required_columns)
    .reset_index(drop=True)
)

lag_selection_results = []
candidate_models = {}

# p = number of gold-return lags
# q = number of demand lags
for p in range(0, max_lag + 1):
    for q in range(0, max_lag + 1):

        regressors = [demand_variable]

        regressors += [
            f"{dependent}_L{i}"
            for i in range(1, p + 1)
        ]

        regressors += [
            f"{demand_variable}_L{i}"
            for i in range(1, q + 1)
        ]

        regressors += control_variables

        X = sm.add_constant(
            adl_common[regressors],
            has_constant="add"
        )
        y = adl_common[dependent]

        fitted_model = sm.OLS(y, X).fit()

        candidate_models[(p, q)] = fitted_model

        lag_selection_results.append({
            "Gold_Lags_p": p,
            "Demand_Lags_q": q,
            "N": int(fitted_model.nobs),
            "Parameters": int(fitted_model.df_model + 1),
            "AIC": fitted_model.aic,
            "BIC": fitted_model.bic,
            "Adjusted_R2": fitted_model.rsquared_adj,
        })

lag_selection_table = (
    pd.DataFrame(lag_selection_results)
    .sort_values(
        ["BIC", "AIC", "Parameters"],
        ascending=[True, True, True]
    )
    .reset_index(drop=True)
)

best_p = int(lag_selection_table.loc[0, "Gold_Lags_p"])
best_q = int(lag_selection_table.loc[0, "Demand_Lags_q"])

best_adl_model = candidate_models[(best_p, best_q)]

print("Common-sample observations:", len(adl_common))
print(
    "Common-sample period:",
    adl_common["Month"].min(),
    "to",
    adl_common["Month"].max()
)
print(f"BIC-selected ADL order: p={best_p}, q={best_q}")

display(lag_selection_table.head(10))

Common-sample observations: 87
Common-sample period: 2019-05-01 00:00:00 to 2026-07-01 00:00:00
BIC-selected ADL order: p=0, q=0


,Gold_Lags_p,Demand_Lags_q,N,Parameters,AIC,BIC,Adjusted_R2
0,0,0,87,17,465.1261,507.0465,0.3143
1,1,0,87,18,464.2580,508.6443,0.3269
2,2,0,87,19,463.5579,510.4101,0.3378
3,0,1,87,18,466.8984,511.2848,0.3061
4,1,1,87,19,466.2172,513.0695,0.3173
5,3,0,87,20,464.9997,514.3178,0.3323
6,2,1,87,20,465.5419,514.8600,0.3281
7,0,2,87,19,468.8337,515.6860,0.2965
8,1,2,87,20,468.1938,517.5120,0.3073
9,2,2,87,21,466.7205,518.5045,0.3243


In [46]:
# CELL 23: Conditional Granger causality tests for Model 1

from scipy.stats import chi2
import statsmodels.api as sm
import pandas as pd

granger_results = []
granger_models = {}

for lag_order in range(1, 4):

    test_df = model_1_complete.copy()

    # Include equal lags of gold returns and combined demand
    for lag in range(1, lag_order + 1):
        test_df[f"Gold_Return_L{lag}"] = (
            test_df["Gold_Return"].shift(lag)
        )
        test_df[f"Demand_L{lag}"] = (
            test_df["ETF_plus_IMF_CB_tonnes"].shift(lag)
        )

    regressors = (
        [f"Gold_Return_L{i}" for i in range(1, lag_order + 1)]
        + [f"Demand_L{i}" for i in range(1, lag_order + 1)]
        + control_variables
    )

    estimation_df = (
        test_df[
            ["Month", "Gold_Return"] + regressors
        ]
        .dropna()
        .reset_index(drop=True)
    )

    X = sm.add_constant(
        estimation_df[regressors],
        has_constant="add"
    )
    y = estimation_df["Gold_Return"]

    # HAC covariance for monthly time-series inference
    model = sm.OLS(y, X).fit(
        cov_type="HAC",
        cov_kwds={
            "maxlags": lag_order,
            "use_correction": True
        }
    )

    granger_models[lag_order] = model

    # H0: all coefficients on lagged demand equal zero
    demand_lag_names = [
        f"Demand_L{i}"
        for i in range(1, lag_order + 1)
    ]

    restriction_matrix = pd.DataFrame(
        0.0,
        index=demand_lag_names,
        columns=model.params.index
    )

    for variable in demand_lag_names:
        restriction_matrix.loc[variable, variable] = 1.0

    wald_result = model.wald_test(
        restriction_matrix.to_numpy(),
        scalar=True
    )

    granger_results.append({
        "Lag_Order": lag_order,
        "N": int(model.nobs),
        "Start_Month": estimation_df["Month"].min(),
        "Wald_Statistic": float(wald_result.statistic),
        "Restrictions": lag_order,
        "p_value": float(wald_result.pvalue),
        "Demand_Granger_Causes_Gold_5pct": (
            float(wald_result.pvalue) < 0.05
        ),
        "Demand_Granger_Causes_Gold_10pct": (
            float(wald_result.pvalue) < 0.10
        ),
    })

conditional_granger_results = pd.DataFrame(granger_results)

display(conditional_granger_results)

,Lag_Order,N,Start_Month,Wald_Statistic,Restrictions,p_value,Demand_Granger_Causes_Gold_5pct,Demand_Granger_Causes_Gold_10pct
0,1,89,2019-03-01,0.2049,1,0.6508,False,False
1,2,88,2019-04-01,1.1939,2,0.5505,False,False
2,3,87,2019-05-01,2.4253,3,0.4889,False,False


In [47]:
# CELL 24: Primary Model 1 — contemporaneous ADL(0,0) with HAC inference

primary_regressors = [
    "ETF_plus_IMF_CB_tonnes"
] + control_variables

primary_df = (
    model_1_complete[
        ["Month", "Gold_Return"] + primary_regressors
    ]
    .dropna()
    .reset_index(drop=True)
)

X_primary = sm.add_constant(
    primary_df[primary_regressors],
    has_constant="add"
)

y_primary = primary_df["Gold_Return"]

model_1_primary = sm.OLS(
    y_primary,
    X_primary
).fit(
    cov_type="HAC",
    cov_kwds={
        "maxlags": 3,
        "use_correction": True
    }
)

model_1_results = pd.DataFrame({
    "Coefficient": model_1_primary.params,
    "HAC_SE": model_1_primary.bse,
    "HAC_t": model_1_primary.tvalues,
    "HAC_p": model_1_primary.pvalues,
    "CI_Lower_95pct": model_1_primary.conf_int()[0],
    "CI_Upper_95pct": model_1_primary.conf_int()[1],
})

model_1_results["Significant_5pct"] = (
    model_1_results["HAC_p"] < 0.05
)

print("Model 1: Combined ETF + IMF CB demand")
print("Observations:", int(model_1_primary.nobs))
print("R-squared:", round(model_1_primary.rsquared, 4))
print(
    "Adjusted R-squared:",
    round(model_1_primary.rsquared_adj, 4)
)
print("AIC:", round(model_1_primary.aic, 4))
print("BIC:", round(model_1_primary.bic, 4))

display(
    model_1_results.sort_values("HAC_p")
)

Model 1: Combined ETF + IMF CB demand
Observations: 90
R-squared: 0.4416
Adjusted R-squared: 0.3192
AIC: 477.62
BIC: 520.1167


,Coefficient,HAC_SE,HAC_t,HAC_p,CI_Lower_95pct,CI_Upper_95pct,Significant_5pct
ETF_plus_IMF_CB_tonnes,0.0193,0.0046,4.2313,0.0000,0.0104,0.0282,True
Post_2022_Dummy,1.9564,0.6948,2.8160,0.0049,0.5947,3.3181,True
Broad_USD_Change,-1.2125,0.5012,-2.4194,0.0155,-2.1948,-0.2302,True
EPU_Change,0.0424,0.0198,2.1389,0.0324,0.0035,0.0813,True
VIX_Change,0.1887,0.0976,1.9343,0.0531,-0.0025,0.3799,False
USDINR_Change,0.5628,0.3571,1.5761,0.1150,-0.1371,1.2627,False
const,-0.9559,0.7210,-1.3258,0.1849,-2.3690,0.4572,False
Fed_Funds_Change_pp,-2.1576,2.1158,-1.0198,0.3078,-6.3045,1.9893,False
Post_2025_Dummy,-1.1893,1.5746,-0.7553,0.4501,-4.2755,1.8970,False
HY_OAS_Change_pp,-1.0012,1.4141,-0.7080,0.4789,-3.7729,1.7704,False


In [49]:
# CELL 25: Construct the separate ETF and IMF-CB time-series model

model_2_df = monthly[["Month"]].copy()

# Outcome and demand-channel variables
model_2_df["Gold_Return"] = (
    monthly["Gold_Monthly_Return_pct"]
)

model_2_df["ETF_Flow"] = (
    monthly["ETF_Net_Demand_Global_WGC_tonnes"]
)

model_2_df["CB_IMF_Change"] = (
    monthly[
        "CB_Reported_Holdings_Change_Global_IMF_tonnes"
    ]
)

# Variables already expressed as rates, returns or changes
model_2_df["CPI_MoM"] = (
    monthly["US_CPI_Inflation_MoM_pct"]
)

model_2_df["USDINR_Change"] = (
    monthly["USDINR_Monthly_Change_pct"]
)

model_2_df["USDCNY_Change"] = (
    monthly["USDCNY_Monthly_Change_pct"]
)

model_2_df["SP500_Return"] = (
    monthly["SP500_Monthly_Return_pct"]
)

model_2_df["EPU_Change"] = (
    monthly["US_EPU_Monthly_Change_pct"]
)

# Stationarity-oriented transformations of level variables
model_2_df["Broad_USD_Change"] = (
    monthly["Broad_USD_Index_Avg"].diff()
)

model_2_df["US10Y_Change_pp"] = (
    monthly["US_10Y_Yield_MonthEnd_pct"].diff()
)

model_2_df["Real10Y_Change_pp"] = (
    monthly["US_10Y_Real_Yield_MonthEnd_pct"].diff()
)

model_2_df["Term_Spread_Change_pp"] = (
    monthly["Term_Spread_10Y2Y_MonthEnd_pct"].diff()
)

model_2_df["Fed_Funds_Change_pp"] = (
    monthly["Effective_Fed_Funds_Avg_pct"].diff()
)

model_2_df["VIX_Change"] = (
    monthly["VIX_MonthEnd"].diff()
)

model_2_df["HY_OAS_Change_pp"] = (
    monthly["US_High_Yield_OAS_MonthEnd_pct"].diff()
)

model_2_df["Brent_Return_pct"] = (
    monthly["Brent_MonthEnd_USD_per_Barrel"]
    .pct_change(fill_method=None)
    .mul(100)
)

# Regime controls
model_2_df["Post_2022_Dummy"] = (
    model_2_df["Month"] >= pd.Timestamp("2022-01-01")
).astype(int)

model_2_df["Post_2025_Dummy"] = (
    model_2_df["Month"] >= pd.Timestamp("2025-01-01")
).astype(int)

model_2_controls = [
    "CPI_MoM",
    "Broad_USD_Change",
    "US10Y_Change_pp",
    "Real10Y_Change_pp",
    "Term_Spread_Change_pp",
    "Fed_Funds_Change_pp",
    "USDINR_Change",
    "USDCNY_Change",
    "SP500_Return",
    "VIX_Change",
    "EPU_Change",
    "HY_OAS_Change_pp",
    "Brent_Return_pct",
    "Post_2022_Dummy",
    "Post_2025_Dummy",
]

model_2_variables = [
    "Month",
    "Gold_Return",
    "ETF_Flow",
    "CB_IMF_Change",
] + model_2_controls

model_2_df = (
    model_2_df[model_2_variables]
    .sort_values("Month")
    .reset_index(drop=True)
)

# Verify construction
availability = pd.DataFrame({
    "Variable": model_2_variables[1:],
    "Available": [
        model_2_df[column].notna().sum()
        for column in model_2_variables[1:]
    ],
    "Missing": [
        model_2_df[column].isna().sum()
        for column in model_2_variables[1:]
    ],
})

display(availability)
display(model_2_df.head())

,Variable,Available,Missing
0,Gold_Return,91,0
1,ETF_Flow,91,0
2,CB_IMF_Change,91,0
3,CPI_MoM,91,0
4,Broad_USD_Change,90,1
5,US10Y_Change_pp,90,1
6,Real10Y_Change_pp,90,1
7,Term_Spread_Change_pp,90,1
8,Fed_Funds_Change_pp,90,1
9,USDINR_Change,91,0


,Month,Gold_Return,ETF_Flow,CB_IMF_Change,CPI_MoM,Broad_USD_Change,US10Y_Change_pp,Real10Y_Change_pp,Term_Spread_Change_pp,Fed_Funds_Change_pp,USDINR_Change,USDCNY_Change,SP500_Return,VIX_Change,EPU_Change,HY_OAS_Change_pp,Brent_Return_pct,Post_2022_Dummy,Post_2025_Dummy
0,2019-01-01,3.0211,71.8000,68.1649,-0.0815,NaN,NaN,NaN,NaN,NaN,-0.1738,-1.4144,7.7318,NaN,20.8947,NaN,NaN,0,0
1,2019-02-01,-0.3113,-32.4000,18.2362,0.3001,-0.0295,0.1000,0.0000,0.0300,-0.0109,0.6560,-0.7318,2.8804,-1.7900,-46.8241,-0.4500,6.6893,0,0
2,2019-03-01,-0.2623,1.7000,46.9922,0.3782,0.3705,-0.3200,-0.2500,-0.0700,0.1238,-2.3665,-0.3673,1.0953,-1.0700,31.1941,0.1300,3.5741,0,0
3,2019-04-01,-0.4346,-56.9000,59.1245,0.3760,0.1202,0.1000,0.0300,0.1000,0.0194,-0.1190,0.0615,2.7428,-0.5900,-29.5982,-0.3200,6.4483,0,0
4,2019-05-01,1.9042,-2.4000,43.5958,0.0247,1.0536,-0.3700,-0.1600,-0.0500,-0.1345,0.5416,2.0219,-5.8716,5.5900,24.1066,0.8600,-11.4148,0,0


In [50]:
# CELL 26: ADL lag selection with ETF and IMF CB entered separately

import statsmodels.api as sm
import pandas as pd

max_lag = 3
adl_2_df = model_2_df.copy()

# Create lags for gold returns, ETF flows and IMF CB changes
for lag in range(1, max_lag + 1):

    adl_2_df[f"Gold_Return_L{lag}"] = (
        adl_2_df["Gold_Return"].shift(lag)
    )

    adl_2_df[f"ETF_Flow_L{lag}"] = (
        adl_2_df["ETF_Flow"].shift(lag)
    )

    adl_2_df[f"CB_IMF_Change_L{lag}"] = (
        adl_2_df["CB_IMF_Change"].shift(lag)
    )

# Common sample for all candidate specifications
required_columns = (
    [
        "Gold_Return",
        "ETF_Flow",
        "CB_IMF_Change",
    ]
    + model_2_controls
    + [
        f"Gold_Return_L{i}"
        for i in range(1, max_lag + 1)
    ]
    + [
        f"ETF_Flow_L{i}"
        for i in range(1, max_lag + 1)
    ]
    + [
        f"CB_IMF_Change_L{i}"
        for i in range(1, max_lag + 1)
    ]
)

adl_2_common = (
    adl_2_df
    .dropna(subset=required_columns)
    .reset_index(drop=True)
)

lag_selection_2 = []
candidate_models_2 = {}

# p: gold-return lags
# q: ETF-flow lags
# r: IMF central-bank-change lags
for p in range(0, max_lag + 1):
    for q in range(0, max_lag + 1):
        for r in range(0, max_lag + 1):

            regressors = [
                "ETF_Flow",
                "CB_IMF_Change",
            ]

            regressors += [
                f"Gold_Return_L{i}"
                for i in range(1, p + 1)
            ]

            regressors += [
                f"ETF_Flow_L{i}"
                for i in range(1, q + 1)
            ]

            regressors += [
                f"CB_IMF_Change_L{i}"
                for i in range(1, r + 1)
            ]

            regressors += model_2_controls

            X = sm.add_constant(
                adl_2_common[regressors],
                has_constant="add"
            )
            y = adl_2_common["Gold_Return"]

            fitted_model = sm.OLS(y, X).fit()

            candidate_models_2[(p, q, r)] = fitted_model

            lag_selection_2.append({
                "Gold_Lags_p": p,
                "ETF_Lags_q": q,
                "CB_Lags_r": r,
                "N": int(fitted_model.nobs),
                "Parameters": int(len(fitted_model.params)),
                "AIC": fitted_model.aic,
                "BIC": fitted_model.bic,
                "Adjusted_R2": fitted_model.rsquared_adj,
            })

lag_selection_2_table = (
    pd.DataFrame(lag_selection_2)
    .sort_values(
        ["BIC", "AIC", "Parameters"],
        ascending=[True, True, True]
    )
    .reset_index(drop=True)
)

# Primary BIC specification
best_bic_row = lag_selection_2_table.iloc[0]

best_p_bic = int(best_bic_row["Gold_Lags_p"])
best_q_bic = int(best_bic_row["ETF_Lags_q"])
best_r_bic = int(best_bic_row["CB_Lags_r"])

best_adl_2_bic = candidate_models_2[
    (best_p_bic, best_q_bic, best_r_bic)
]

# AIC specification for robustness
best_aic_row = (
    lag_selection_2_table
    .sort_values(
        ["AIC", "BIC", "Parameters"],
        ascending=[True, True, True]
    )
    .iloc[0]
)

best_p_aic = int(best_aic_row["Gold_Lags_p"])
best_q_aic = int(best_aic_row["ETF_Lags_q"])
best_r_aic = int(best_aic_row["CB_Lags_r"])

print("Common-sample observations:", len(adl_2_common))
print(
    "Common-sample period:",
    adl_2_common["Month"].min(),
    "to",
    adl_2_common["Month"].max()
)

print(
    f"BIC-selected order: "
    f"p={best_p_bic}, "
    f"q={best_q_bic}, "
    f"r={best_r_bic}"
)

print(
    f"AIC-selected order: "
    f"p={best_p_aic}, "
    f"q={best_q_aic}, "
    f"r={best_r_aic}"
)

display(lag_selection_2_table.head(15))

Common-sample observations: 88
Common-sample period: 2019-04-01 00:00:00 to 2026-07-01 00:00:00
BIC-selected order: p=0, q=0, r=0
AIC-selected order: p=2, q=0, r=2


,Gold_Lags_p,ETF_Lags_q,CB_Lags_r,N,Parameters,AIC,BIC,Adjusted_R2
0,0,0,0,88,18,462.3812,506.9732,0.3732
1,2,0,0,88,20,459.0004,508.5471,0.4067
2,0,0,2,88,20,459.8671,509.4138,0.4008
3,1,0,0,88,19,462.5955,509.6649,0.3769
4,2,1,0,88,21,458.1282,510.1523,0.4172
5,0,0,1,88,19,463.3773,510.4467,0.3714
6,0,2,0,88,20,461.1531,510.6999,0.3920
7,0,1,0,88,19,463.7375,510.8069,0.3688
8,1,1,0,88,20,461.5388,511.0855,0.3893
9,2,0,2,88,22,456.6089,511.1103,0.4316


In [51]:
# CELL 27: Multicollinearity diagnostics for the expanded Model 2

from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd

contemporaneous_regressors = [
    "ETF_Flow",
    "CB_IMF_Change",
] + model_2_controls

diagnostic_df = (
    adl_2_common[contemporaneous_regressors]
    .dropna()
    .copy()
)

# 1. Pearson correlation matrix
control_correlation = diagnostic_df.corr()

# Extract highly correlated pairs without duplication
high_correlation_pairs = []

for i, variable_1 in enumerate(control_correlation.columns):
    for j in range(i + 1, len(control_correlation.columns)):

        variable_2 = control_correlation.columns[j]
        correlation = control_correlation.iloc[i, j]

        if abs(correlation) >= 0.70:
            high_correlation_pairs.append({
                "Variable_1": variable_1,
                "Variable_2": variable_2,
                "Correlation": correlation,
                "Absolute_Correlation": abs(correlation),
            })

high_correlation_pairs = (
    pd.DataFrame(high_correlation_pairs)
    .sort_values(
        "Absolute_Correlation",
        ascending=False
    )
    if high_correlation_pairs
    else pd.DataFrame(
        columns=[
            "Variable_1",
            "Variable_2",
            "Correlation",
            "Absolute_Correlation",
        ]
    )
)

# 2. Variance inflation factors
X_vif = diagnostic_df.astype(float)

vif_results = pd.DataFrame({
    "Variable": X_vif.columns,
    "VIF": [
        variance_inflation_factor(
            X_vif.values,
            index
        )
        for index in range(X_vif.shape[1])
    ],
})

vif_results["Assessment"] = pd.cut(
    vif_results["VIF"],
    bins=[-np.inf, 5, 10, np.inf],
    labels=[
        "Acceptable",
        "Moderate",
        "High",
    ],
    right=False
)

vif_results = (
    vif_results
    .sort_values("VIF", ascending=False)
    .reset_index(drop=True)
)

# 3. Condition number after standardisation
X_standardised = StandardScaler().fit_transform(X_vif)

singular_values = np.linalg.svd(
    X_standardised,
    compute_uv=False
)

standardised_condition_number = (
    singular_values.max() / singular_values.min()
)

print("Observations used:", len(diagnostic_df))
print(
    "Standardised condition number:",
    round(standardised_condition_number, 4)
)

if standardised_condition_number < 10:
    condition_assessment = "Low multicollinearity"
elif standardised_condition_number < 30:
    condition_assessment = "Moderate multicollinearity"
else:
    condition_assessment = "Potentially severe multicollinearity"

print("Condition-number assessment:", condition_assessment)

print("\nHighly correlated pairs: |correlation| >= 0.70")
display(high_correlation_pairs)

print("\nVariance inflation factors")
display(vif_results)

Observations used: 88
Standardised condition number: 8.0687
Condition-number assessment: Low multicollinearity

Highly correlated pairs: |correlation| >= 0.70


,Variable_1,Variable_2,Correlation,Absolute_Correlation
0,US10Y_Change_pp,Real10Y_Change_pp,0.8362,0.8362
1,SP500_Return,VIX_Change,-0.7846,0.7846
2,SP500_Return,HY_OAS_Change_pp,-0.7657,0.7657



Variance inflation factors


,Variable,VIF,Assessment
0,Real10Y_Change_pp,9.2052,Moderate
1,US10Y_Change_pp,7.3146,Moderate
2,SP500_Return,7.0303,Moderate
3,HY_OAS_Change_pp,5.6583,Moderate
4,Broad_USD_Change,3.3728,Acceptable
5,VIX_Change,3.3351,Acceptable
6,Post_2022_Dummy,2.9516,Acceptable
7,CPI_MoM,2.2393,Acceptable
8,Fed_Funds_Change_pp,2.2345,Acceptable
9,USDCNY_Change,2.1783,Acceptable


In [52]:
# CELL 28: Define primary, expanded and alternative control sets

# Primary economically parsimonious controls
primary_controls = [
    "CPI_MoM",
    "Broad_USD_Change",
    "Real10Y_Change_pp",
    "Term_Spread_Change_pp",
    "Fed_Funds_Change_pp",
    "USDINR_Change",
    "USDCNY_Change",
    "VIX_Change",
    "EPU_Change",
    "HY_OAS_Change_pp",
    "Brent_Return_pct",
]

# Expanded robustness model
expanded_controls = [
    "CPI_MoM",
    "Broad_USD_Change",
    "US10Y_Change_pp",
    "Real10Y_Change_pp",
    "Term_Spread_Change_pp",
    "Fed_Funds_Change_pp",
    "USDINR_Change",
    "USDCNY_Change",
    "SP500_Return",
    "VIX_Change",
    "EPU_Change",
    "HY_OAS_Change_pp",
    "Brent_Return_pct",
    "Post_2022_Dummy",
    "Post_2025_Dummy",
]

# Alternative equity-risk representation
alternative_risk_controls = [
    "CPI_MoM",
    "Broad_USD_Change",
    "Real10Y_Change_pp",
    "Term_Spread_Change_pp",
    "Fed_Funds_Change_pp",
    "USDINR_Change",
    "USDCNY_Change",
    "SP500_Return",
    "EPU_Change",
    "Brent_Return_pct",
]

control_specifications = {
    "Primary_Core": primary_controls,
    "Expanded_All": expanded_controls,
    "Alternative_SP500": alternative_risk_controls,
}

control_summary = pd.DataFrame([
    {
        "Specification": name,
        "Number_of_Controls": len(controls),
        "Controls": ", ".join(controls),
    }
    for name, controls in control_specifications.items()
])

display(control_summary)

,Specification,Number_of_Controls,Controls
0,Primary_Core,11,"CPI_MoM, Broad_USD_Change, Real10Y_Change_pp, ..."
1,Expanded_All,15,"CPI_MoM, Broad_USD_Change, US10Y_Change_pp, Re..."
2,Alternative_SP500,10,"CPI_MoM, Broad_USD_Change, Real10Y_Change_pp, ..."


In [53]:
# Corrected VIF diagnostic for primary controls

from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm

primary_vif_df = (
    model_2_df[
        ["ETF_Flow", "CB_IMF_Change"] + primary_controls
    ]
    .dropna()
)

X_primary_vif = sm.add_constant(
    primary_vif_df,
    has_constant="add"
)

primary_vif_results = pd.DataFrame({
    "Variable": X_primary_vif.columns,
    "VIF": [
        variance_inflation_factor(
            X_primary_vif.values,
            i
        )
        for i in range(X_primary_vif.shape[1])
    ]
})

primary_vif_results = (
    primary_vif_results[
        primary_vif_results["Variable"] != "const"
    ]
    .sort_values("VIF", ascending=False)
    .reset_index(drop=True)
)

primary_vif_results["Assessment"] = np.select(
    [
        primary_vif_results["VIF"] >= 10,
        primary_vif_results["VIF"] >= 5,
    ],
    [
        "High",
        "Moderate",
    ],
    default="Acceptable"
)

display(primary_vif_results)

,Variable,VIF,Assessment
0,Broad_USD_Change,2.9883,Acceptable
1,HY_OAS_Change_pp,2.9081,Acceptable
2,USDCNY_Change,2.1201,Acceptable
3,VIX_Change,2.0122,Acceptable
4,Fed_Funds_Change_pp,1.8665,Acceptable
5,Brent_Return_pct,1.6904,Acceptable
6,ETF_Flow,1.6563,Acceptable
7,EPU_Change,1.5950,Acceptable
8,USDINR_Change,1.5920,Acceptable
9,Real10Y_Change_pp,1.5755,Acceptable


In [54]:
# CELL 29: Primary-model ADL lag selection

max_lag = 3
primary_adl_df = model_2_df.copy()

# Create all required lags
for lag in range(1, max_lag + 1):

    primary_adl_df[f"Gold_Return_L{lag}"] = (
        primary_adl_df["Gold_Return"].shift(lag)
    )

    primary_adl_df[f"ETF_Flow_L{lag}"] = (
        primary_adl_df["ETF_Flow"].shift(lag)
    )

    primary_adl_df[f"CB_IMF_Change_L{lag}"] = (
        primary_adl_df["CB_IMF_Change"].shift(lag)
    )

# Common sample across all candidate lag orders
primary_required_columns = (
    [
        "Gold_Return",
        "ETF_Flow",
        "CB_IMF_Change",
    ]
    + primary_controls
    + [
        f"Gold_Return_L{i}"
        for i in range(1, max_lag + 1)
    ]
    + [
        f"ETF_Flow_L{i}"
        for i in range(1, max_lag + 1)
    ]
    + [
        f"CB_IMF_Change_L{i}"
        for i in range(1, max_lag + 1)
    ]
)

primary_adl_common = (
    primary_adl_df
    .dropna(subset=primary_required_columns)
    .reset_index(drop=True)
)

primary_lag_results = []
primary_candidate_models = {}

for p in range(0, max_lag + 1):
    for q in range(0, max_lag + 1):
        for r in range(0, max_lag + 1):

            regressors = [
                "ETF_Flow",
                "CB_IMF_Change",
            ]

            regressors += [
                f"Gold_Return_L{i}"
                for i in range(1, p + 1)
            ]

            regressors += [
                f"ETF_Flow_L{i}"
                for i in range(1, q + 1)
            ]

            regressors += [
                f"CB_IMF_Change_L{i}"
                for i in range(1, r + 1)
            ]

            regressors += primary_controls

            X = sm.add_constant(
                primary_adl_common[regressors],
                has_constant="add"
            )
            y = primary_adl_common["Gold_Return"]

            fitted_model = sm.OLS(y, X).fit()

            primary_candidate_models[(p, q, r)] = fitted_model

            primary_lag_results.append({
                "Gold_Lags_p": p,
                "ETF_Lags_q": q,
                "CB_Lags_r": r,
                "N": int(fitted_model.nobs),
                "Parameters": int(len(fitted_model.params)),
                "AIC": fitted_model.aic,
                "BIC": fitted_model.bic,
                "Adjusted_R2": fitted_model.rsquared_adj,
            })

primary_lag_table = (
    pd.DataFrame(primary_lag_results)
    .sort_values(
        ["BIC", "AIC", "Parameters"],
        ascending=[True, True, True]
    )
    .reset_index(drop=True)
)

# BIC-selected model
bic_row = primary_lag_table.iloc[0]

primary_bic_p = int(bic_row["Gold_Lags_p"])
primary_bic_q = int(bic_row["ETF_Lags_q"])
primary_bic_r = int(bic_row["CB_Lags_r"])

primary_bic_model = primary_candidate_models[
    (primary_bic_p, primary_bic_q, primary_bic_r)
]

# AIC-selected model
aic_row = (
    primary_lag_table
    .sort_values(
        ["AIC", "BIC", "Parameters"],
        ascending=[True, True, True]
    )
    .iloc[0]
)

primary_aic_p = int(aic_row["Gold_Lags_p"])
primary_aic_q = int(aic_row["ETF_Lags_q"])
primary_aic_r = int(aic_row["CB_Lags_r"])

primary_aic_model = primary_candidate_models[
    (primary_aic_p, primary_aic_q, primary_aic_r)
]

print("Primary-model observations:", len(primary_adl_common))
print(
    "Period:",
    primary_adl_common["Month"].min(),
    "to",
    primary_adl_common["Month"].max()
)

print(
    f"BIC-selected order: "
    f"p={primary_bic_p}, "
    f"q={primary_bic_q}, "
    f"r={primary_bic_r}"
)

print(
    f"AIC-selected order: "
    f"p={primary_aic_p}, "
    f"q={primary_aic_q}, "
    f"r={primary_aic_r}"
)

display(primary_lag_table.head(15))

Primary-model observations: 88
Period: 2019-04-01 00:00:00 to 2026-07-01 00:00:00
BIC-selected order: p=0, q=0, r=0
AIC-selected order: p=1, q=2, r=0


,Gold_Lags_p,ETF_Lags_q,CB_Lags_r,N,Parameters,AIC,BIC,Adjusted_R2
0,0,0,0,88,14,461.2534,495.9362,0.3590
1,1,2,0,88,17,454.8163,496.9310,0.4199
2,0,2,0,88,16,458.0717,497.7091,0.3928
3,1,1,0,88,16,458.4509,498.0883,0.3902
4,1,0,0,88,15,461.2295,498.3896,0.3650
5,2,1,0,88,17,456.9748,499.0895,0.4055
6,0,1,0,88,15,461.9488,499.1089,0.3597
7,2,0,0,88,16,459.8259,499.4633,0.3806
8,0,0,1,88,15,462.6240,499.7840,0.3548
9,2,2,0,88,18,456.0270,500.6190,0.4169


In [55]:
# CELL 30: Estimate primary ADL(1,0,1) with HAC inference

adl_101_regressors = [
    "Gold_Return_L1",
    "ETF_Flow",
    "CB_IMF_Change",
    "CB_IMF_Change_L1",
] + primary_controls

# Use maximum available sample for this specific model
adl_101_df = (
    primary_adl_df[
        ["Month", "Gold_Return"] + adl_101_regressors
    ]
    .dropna()
    .reset_index(drop=True)
)

X_101 = sm.add_constant(
    adl_101_df[adl_101_regressors],
    has_constant="add"
)

y_101 = adl_101_df["Gold_Return"]

# Ordinary estimates with HAC/Newey-West inference
adl_101_model = sm.OLS(
    y_101,
    X_101
).fit(
    cov_type="HAC",
    cov_kwds={
        "maxlags": 3,
        "use_correction": True,
    }
)

adl_101_results = pd.DataFrame({
    "Coefficient": adl_101_model.params,
    "HAC_SE": adl_101_model.bse,
    "HAC_t": adl_101_model.tvalues,
    "HAC_p": adl_101_model.pvalues,
    "CI_Lower_95pct": adl_101_model.conf_int()[0],
    "CI_Upper_95pct": adl_101_model.conf_int()[1],
})

adl_101_results["Significant_5pct"] = (
    adl_101_results["HAC_p"] < 0.05
)

adl_101_results["Significant_10pct"] = (
    adl_101_results["HAC_p"] < 0.10
)

print("Primary ADL(1,0,1)")
print("Observations:", int(adl_101_model.nobs))
print(
    "Period:",
    adl_101_df["Month"].min(),
    "to",
    adl_101_df["Month"].max()
)
print("R-squared:", round(adl_101_model.rsquared, 4))
print(
    "Adjusted R-squared:",
    round(adl_101_model.rsquared_adj, 4)
)
print("AIC:", round(adl_101_model.aic, 4))
print("BIC:", round(adl_101_model.bic, 4))

display(
    adl_101_results.loc[
        [
            "ETF_Flow",
            "CB_IMF_Change",
            "CB_IMF_Change_L1",
            "Gold_Return_L1",
        ]
    ]
)

print("\nComplete coefficient table")
display(
    adl_101_results.sort_values("HAC_p")
)

Primary ADL(1,0,1)
Observations: 90
Period: 2019-02-01 00:00:00 to 2026-07-01 00:00:00
R-squared: 0.4752
Adjusted R-squared: 0.3689
AIC: 470.0283
BIC: 510.0253


,Coefficient,HAC_SE,HAC_t,HAC_p,CI_Lower_95pct,CI_Upper_95pct,Significant_5pct,Significant_10pct
ETF_Flow,0.0244,0.0057,4.2869,0.0000,0.0133,0.0356,True,True
CB_IMF_Change,-0.0060,0.0063,-0.9535,0.3404,-0.0182,0.0063,False,False
CB_IMF_Change_L1,0.0086,0.0058,1.4923,0.1356,-0.0027,0.0200,False,False
Gold_Return_L1,0.1346,0.0897,1.5009,0.1334,-0.0412,0.3103,False,False



Complete coefficient table


,Coefficient,HAC_SE,HAC_t,HAC_p,CI_Lower_95pct,CI_Upper_95pct,Significant_5pct,Significant_10pct
ETF_Flow,0.0244,0.0057,4.2869,0.0000,0.0133,0.0356,True,True
Broad_USD_Change,-1.1374,0.3360,-3.3852,0.0007,-1.7959,-0.4789,True,True
EPU_Change,0.0309,0.0194,1.5938,0.1110,-0.0071,0.0688,False,False
Gold_Return_L1,0.1346,0.0897,1.5009,0.1334,-0.0412,0.3103,False,False
CB_IMF_Change_L1,0.0086,0.0058,1.4923,0.1356,-0.0027,0.0200,False,False
USDINR_Change,0.4224,0.2915,1.4489,0.1474,-0.1490,0.9937,False,False
Brent_Return_pct,-0.0349,0.0252,-1.3875,0.1653,-0.0842,0.0144,False,False
HY_OAS_Change_pp,-1.1091,0.9180,-1.2081,0.2270,-2.9085,0.6902,False,False
VIX_Change,0.0701,0.0670,1.0452,0.2959,-0.0613,0.2014,False,False
CB_IMF_Change,-0.0060,0.0063,-0.9535,0.3404,-0.0182,0.0063,False,False


In [56]:
# CELL 31: Wald test of lagged IMF central-bank activity

cb_lag_test_101 = adl_101_model.wald_test(
    "CB_IMF_Change_L1 = 0",
    scalar=True
)

cb_lag_test_table = pd.DataFrame({
    "Hypothesis": [
        "Lagged IMF CB change does not predict gold return"
    ],
    "Wald_Statistic": [
        float(cb_lag_test_101.statistic)
    ],
    "p_value": [
        float(cb_lag_test_101.pvalue)
    ],
})

cb_lag_test_table["Reject_5pct"] = (
    cb_lag_test_table["p_value"] < 0.05
)

cb_lag_test_table["Reject_10pct"] = (
    cb_lag_test_table["p_value"] < 0.10
)

display(cb_lag_test_table)

,Hypothesis,Wald_Statistic,p_value,Reject_5pct,Reject_10pct
0,Lagged IMF CB change does not predict gold return,2.2269,0.1356,False,False


In [57]:
# CELL 32: Estimate AIC-selected ADL(1,2,0)

adl_120_regressors = [
    "Gold_Return_L1",
    "ETF_Flow",
    "ETF_Flow_L1",
    "ETF_Flow_L2",
    "CB_IMF_Change",
] + primary_controls

# Maximum available sample for ADL(1,2,0)
adl_120_df = (
    primary_adl_df[
        ["Month", "Gold_Return"] + adl_120_regressors
    ]
    .dropna()
    .reset_index(drop=True)
)

X_120 = sm.add_constant(
    adl_120_df[adl_120_regressors],
    has_constant="add"
)

y_120 = adl_120_df["Gold_Return"]

adl_120_model = sm.OLS(
    y_120,
    X_120
).fit(
    cov_type="HAC",
    cov_kwds={
        "maxlags": 3,
        "use_correction": True,
    }
)

adl_120_results = pd.DataFrame({
    "Coefficient": adl_120_model.params,
    "HAC_SE": adl_120_model.bse,
    "HAC_t": adl_120_model.tvalues,
    "HAC_p": adl_120_model.pvalues,
    "CI_Lower_95pct": adl_120_model.conf_int()[0],
    "CI_Upper_95pct": adl_120_model.conf_int()[1],
})

adl_120_results["Significant_5pct"] = (
    adl_120_results["HAC_p"] < 0.05
)

adl_120_results["Significant_10pct"] = (
    adl_120_results["HAC_p"] < 0.10
)

print("AIC-selected ADL(1,2,0)")
print("Observations:", int(adl_120_model.nobs))
print(
    "Period:",
    adl_120_df["Month"].min(),
    "to",
    adl_120_df["Month"].max()
)
print("R-squared:", round(adl_120_model.rsquared, 4))
print(
    "Adjusted R-squared:",
    round(adl_120_model.rsquared_adj, 4)
)
print("AIC:", round(adl_120_model.aic, 4))
print("BIC:", round(adl_120_model.bic, 4))

print("\nDemand-channel and autoregressive coefficients")

display(
    adl_120_results.loc[
        [
            "ETF_Flow",
            "ETF_Flow_L1",
            "ETF_Flow_L2",
            "CB_IMF_Change",
            "Gold_Return_L1",
        ]
    ]
)

print("\nComplete coefficient table")

display(
    adl_120_results.sort_values("HAC_p")
)

AIC-selected ADL(1,2,0)
Observations: 89
Period: 2019-03-01 00:00:00 to 2026-07-01 00:00:00
R-squared: 0.5267
Adjusted R-squared: 0.4215
AIC: 458.7665
BIC: 501.0733

Demand-channel and autoregressive coefficients


,Coefficient,HAC_SE,HAC_t,HAC_p,CI_Lower_95pct,CI_Upper_95pct,Significant_5pct,Significant_10pct
ETF_Flow,0.0313,0.0055,5.6745,0.0000,0.0205,0.0421,True,True
ETF_Flow_L1,-0.0083,0.0069,-1.1992,0.2304,-0.0218,0.0053,False,False
ETF_Flow_L2,-0.0134,0.0061,-2.2028,0.0276,-0.0254,-0.0015,True,True
CB_IMF_Change,-0.0074,0.0064,-1.1505,0.2499,-0.0200,0.0052,False,False
Gold_Return_L1,0.2181,0.1049,2.0796,0.0376,0.0125,0.4236,True,True



Complete coefficient table


,Coefficient,HAC_SE,HAC_t,HAC_p,CI_Lower_95pct,CI_Upper_95pct,Significant_5pct,Significant_10pct
ETF_Flow,0.0313,0.0055,5.6745,0.0000,0.0205,0.0421,True,True
Broad_USD_Change,-1.2479,0.2866,-4.3548,0.0000,-1.8096,-0.6863,True,True
ETF_Flow_L2,-0.0134,0.0061,-2.2028,0.0276,-0.0254,-0.0015,True,True
Gold_Return_L1,0.2181,0.1049,2.0796,0.0376,0.0125,0.4236,True,True
USDINR_Change,0.4661,0.2417,1.9283,0.0538,-0.0076,0.9398,False,True
const,0.8270,0.4764,1.7361,0.0826,-0.1067,1.7608,False,True
EPU_Change,0.0264,0.0190,1.3903,0.1644,-0.0108,0.0636,False,False
ETF_Flow_L1,-0.0083,0.0069,-1.1992,0.2304,-0.0218,0.0053,False,False
CB_IMF_Change,-0.0074,0.0064,-1.1505,0.2499,-0.0200,0.0052,False,False
Term_Spread_Change_pp,2.1684,2.1765,0.9963,0.3191,-2.0975,6.4343,False,False


In [58]:
# CELL 33: ETF Granger and cumulative-effect tests for ADL(1,2,0)

# Conditional ETF Granger-causality test
# H0: both lagged ETF coefficients equal zero
etf_granger_test = adl_120_model.wald_test(
    [
        "ETF_Flow_L1 = 0",
        "ETF_Flow_L2 = 0",
    ],
    scalar=True
)

# Joint test of current and lagged ETF coefficients
etf_total_joint_test = adl_120_model.wald_test(
    [
        "ETF_Flow = 0",
        "ETF_Flow_L1 = 0",
        "ETF_Flow_L2 = 0",
    ],
    scalar=True
)

# Test whether the cumulative ETF coefficient equals zero
etf_cumulative_test = adl_120_model.t_test(
    "ETF_Flow + ETF_Flow_L1 + ETF_Flow_L2 = 0"
)

# Contemporaneous IMF CB coefficient test
cb_current_test = adl_120_model.t_test(
    "CB_IMF_Change = 0"
)

hypothesis_tests_120 = pd.DataFrame({
    "Test": [
        "ETF Granger causality: L1 = L2 = 0",
        "ETF current and lags jointly equal zero",
        "Cumulative ETF coefficient equals zero",
        "Contemporaneous IMF CB coefficient equals zero",
    ],
    "Statistic": [
        float(etf_granger_test.statistic),
        float(etf_total_joint_test.statistic),
        float(etf_cumulative_test.statistic),
        float(cb_current_test.statistic),
    ],
    "p_value": [
        float(etf_granger_test.pvalue),
        float(etf_total_joint_test.pvalue),
        float(etf_cumulative_test.pvalue),
        float(cb_current_test.pvalue),
    ],
})

hypothesis_tests_120["Reject_5pct"] = (
    hypothesis_tests_120["p_value"] < 0.05
)

hypothesis_tests_120["Reject_10pct"] = (
    hypothesis_tests_120["p_value"] < 0.10
)

display(hypothesis_tests_120)

/tmp/ipykernel_4017/2436901497.py:43: DeprecationWarning:

Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)

/tmp/ipykernel_4017/2436901497.py:44: DeprecationWarning:

Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)



,Test,Statistic,p_value,Reject_5pct,Reject_10pct
0,ETF Granger causality: L1 = L2 = 0,12.3980,0.0020,True,True
1,ETF current and lags jointly equal zero,39.4488,0.0000,True,True
2,Cumulative ETF coefficient equals zero,1.5673,0.1170,False,False
3,Contemporaneous IMF CB coefficient equals zero,-1.1505,0.2499,False,False


In [59]:
# CELL 34: Estimate ADL(1,2,1) with HAC inference

adl_121_regressors = [
    "Gold_Return_L1",
    "ETF_Flow",
    "ETF_Flow_L1",
    "ETF_Flow_L2",
    "CB_IMF_Change",
    "CB_IMF_Change_L1",
] + primary_controls

adl_121_df = (
    primary_adl_df[
        ["Month", "Gold_Return"] + adl_121_regressors
    ]
    .dropna()
    .reset_index(drop=True)
)

X_121 = sm.add_constant(
    adl_121_df[adl_121_regressors],
    has_constant="add"
)

y_121 = adl_121_df["Gold_Return"]

adl_121_model = sm.OLS(
    y_121,
    X_121
).fit(
    cov_type="HAC",
    cov_kwds={
        "maxlags": 3,
        "use_correction": True,
    }
)

adl_121_results = pd.DataFrame({
    "Coefficient": adl_121_model.params,
    "HAC_SE": adl_121_model.bse,
    "HAC_t": adl_121_model.tvalues,
    "HAC_p": adl_121_model.pvalues,
    "CI_Lower_95pct": adl_121_model.conf_int()[0],
    "CI_Upper_95pct": adl_121_model.conf_int()[1],
})

adl_121_results["Significant_5pct"] = (
    adl_121_results["HAC_p"] < 0.05
)

adl_121_results["Significant_10pct"] = (
    adl_121_results["HAC_p"] < 0.10
)

print("ADL(1,2,1)")
print("Observations:", int(adl_121_model.nobs))
print(
    "Period:",
    adl_121_df["Month"].min(),
    "to",
    adl_121_df["Month"].max()
)
print("R-squared:", round(adl_121_model.rsquared, 4))
print(
    "Adjusted R-squared:",
    round(adl_121_model.rsquared_adj, 4)
)
print("AIC:", round(adl_121_model.aic, 4))
print("BIC:", round(adl_121_model.bic, 4))

demand_terms_121 = [
    "ETF_Flow",
    "ETF_Flow_L1",
    "ETF_Flow_L2",
    "CB_IMF_Change",
    "CB_IMF_Change_L1",
    "Gold_Return_L1",
]

print("\nDemand-channel and autoregressive coefficients")
display(adl_121_results.loc[demand_terms_121])

print("\nComplete coefficient table")
display(adl_121_results.sort_values("HAC_p"))

ADL(1,2,1)
Observations: 89
Period: 2019-03-01 00:00:00 to 2026-07-01 00:00:00
R-squared: 0.5285
Adjusted R-squared: 0.4157
AIC: 460.4216
BIC: 505.2171

Demand-channel and autoregressive coefficients


,Coefficient,HAC_SE,HAC_t,HAC_p,CI_Lower_95pct,CI_Upper_95pct,Significant_5pct,Significant_10pct
ETF_Flow,0.0316,0.0056,5.6451,0.0000,0.0206,0.0425,True,True
ETF_Flow_L1,-0.0083,0.0069,-1.2121,0.2255,-0.0218,0.0051,False,False
ETF_Flow_L2,-0.0129,0.0062,-2.1027,0.0355,-0.0250,-0.0009,True,True
CB_IMF_Change,-0.0084,0.0064,-1.3185,0.1873,-0.0208,0.0041,False,False
CB_IMF_Change_L1,0.0049,0.0074,0.6655,0.5058,-0.0096,0.0195,False,False
Gold_Return_L1,0.2229,0.1055,2.1133,0.0346,0.0162,0.4296,True,True



Complete coefficient table


,Coefficient,HAC_SE,HAC_t,HAC_p,CI_Lower_95pct,CI_Upper_95pct,Significant_5pct,Significant_10pct
ETF_Flow,0.0316,0.0056,5.6451,0.0000,0.0206,0.0425,True,True
Broad_USD_Change,-1.2494,0.2887,-4.3273,0.0000,-1.8153,-0.6835,True,True
Gold_Return_L1,0.2229,0.1055,2.1133,0.0346,0.0162,0.4296,True,True
ETF_Flow_L2,-0.0129,0.0062,-2.1027,0.0355,-0.0250,-0.0009,True,True
USDINR_Change,0.4221,0.2763,1.5275,0.1266,-0.1195,0.9637,False,False
CB_IMF_Change,-0.0084,0.0064,-1.3185,0.1873,-0.0208,0.0041,False,False
EPU_Change,0.0256,0.0195,1.3113,0.1897,-0.0127,0.0639,False,False
const,0.6889,0.5303,1.2990,0.1940,-0.3505,1.7283,False,False
ETF_Flow_L1,-0.0083,0.0069,-1.2121,0.2255,-0.0218,0.0051,False,False
USDCNY_Change,0.3196,0.3245,0.9849,0.3247,-0.3164,0.9556,False,False


In [60]:
# CELL 35: Hypothesis tests for ADL(1,2,1)

def scalar_value(result):
    value = np.asarray(result)
    return value.item() if value.size == 1 else np.nan

# ETF Granger test: lagged ETF terms only
etf_granger_121 = adl_121_model.wald_test(
    [
        "ETF_Flow_L1 = 0",
        "ETF_Flow_L2 = 0",
    ],
    scalar=True
)

# CB Granger test: lagged CB term
cb_granger_121 = adl_121_model.wald_test(
    "CB_IMF_Change_L1 = 0",
    scalar=True
)

# Joint predictive-demand test
joint_granger_121 = adl_121_model.wald_test(
    [
        "ETF_Flow_L1 = 0",
        "ETF_Flow_L2 = 0",
        "CB_IMF_Change_L1 = 0",
    ],
    scalar=True
)

# Complete ETF channel: current plus two lags
etf_complete_121 = adl_121_model.wald_test(
    [
        "ETF_Flow = 0",
        "ETF_Flow_L1 = 0",
        "ETF_Flow_L2 = 0",
    ],
    scalar=True
)

# Complete CB channel: current and one lag
cb_complete_121 = adl_121_model.wald_test(
    [
        "CB_IMF_Change = 0",
        "CB_IMF_Change_L1 = 0",
    ],
    scalar=True
)

# Cumulative coefficient tests
etf_cumulative_121 = adl_121_model.t_test(
    "ETF_Flow + ETF_Flow_L1 + ETF_Flow_L2 = 0"
)

cb_cumulative_121 = adl_121_model.t_test(
    "CB_IMF_Change + CB_IMF_Change_L1 = 0"
)

test_objects = [
    (
        "ETF Granger causality: L1 = L2 = 0",
        etf_granger_121
    ),
    (
        "CB Granger causality: L1 = 0",
        cb_granger_121
    ),
    (
        "ETF and CB lags jointly equal zero",
        joint_granger_121
    ),
    (
        "Complete ETF channel equals zero",
        etf_complete_121
    ),
    (
        "Complete CB channel equals zero",
        cb_complete_121
    ),
    (
        "Cumulative ETF coefficient equals zero",
        etf_cumulative_121
    ),
    (
        "Cumulative CB coefficient equals zero",
        cb_cumulative_121
    ),
]

adl_121_tests = pd.DataFrame([
    {
        "Test": name,
        "Statistic": scalar_value(test.statistic),
        "p_value": scalar_value(test.pvalue),
    }
    for name, test in test_objects
])

adl_121_tests["Reject_5pct"] = (
    adl_121_tests["p_value"] < 0.05
)

adl_121_tests["Reject_10pct"] = (
    adl_121_tests["p_value"] < 0.10
)

display(adl_121_tests)

,Test,Statistic,p_value,Reject_5pct,Reject_10pct
0,ETF Granger causality: L1 = L2 = 0,10.8443,0.0044,True,True
1,CB Granger causality: L1 = 0,0.4428,0.5058,False,False
2,ETF and CB lags jointly equal zero,13.6320,0.0035,True,True
3,Complete ETF channel equals zero,38.1244,0.0000,True,True
4,Complete CB channel equals zero,1.9935,0.3691,False,False
5,Cumulative ETF coefficient equals zero,1.5755,0.1151,False,False
6,Cumulative CB coefficient equals zero,-0.3760,0.7069,False,False


In [61]:
# CELL 36: Direct comparison of ADL(1,2,0) and ADL(1,2,1)

adl_comparison = pd.DataFrame({
    "Model": [
        "ADL(1,2,0)",
        "ADL(1,2,1)",
    ],
    "N": [
        int(adl_120_model.nobs),
        int(adl_121_model.nobs),
    ],
    "Parameters": [
        len(adl_120_model.params),
        len(adl_121_model.params),
    ],
    "Adjusted_R2": [
        adl_120_model.rsquared_adj,
        adl_121_model.rsquared_adj,
    ],
    "AIC": [
        adl_120_model.aic,
        adl_121_model.aic,
    ],
    "BIC": [
        adl_120_model.bic,
        adl_121_model.bic,
    ],
})

display(adl_comparison)

,Model,N,Parameters,Adjusted_R2,AIC,BIC
0,"ADL(1,2,0)",89,17,0.4215,458.7665,501.0733
1,"ADL(1,2,1)",89,18,0.4157,460.4216,505.2171


In [62]:
# CELL 37: Diagnostic tests for primary ADL(1,2,0)

from statsmodels.stats.diagnostic import (
    acorr_breusch_godfrey,
    acorr_ljungbox,
    het_breuschpagan,
    het_arch,
    linear_reset,
    breaks_cusumolsresid,
)
from statsmodels.stats.stattools import jarque_bera
import pandas as pd
import numpy as np
import statsmodels.api as sm

# Re-estimate without covariance correction for diagnostic functions
adl_120_ols = sm.OLS(
    y_120,
    X_120
).fit()

diagnostic_results = []

# 1. Breusch-Godfrey serial-correlation tests
for lag in [1, 2, 3, 6]:

    bg_lm, bg_lm_p, bg_f, bg_f_p = (
        acorr_breusch_godfrey(
            adl_120_ols,
            nlags=lag
        )
    )

    diagnostic_results.append({
        "Category": "Serial correlation",
        "Test": f"Breusch-Godfrey LM({lag})",
        "Statistic": bg_lm,
        "p_value": bg_lm_p,
        "Null_Hypothesis": (
            f"No residual serial correlation through lag {lag}"
        ),
    })

# 2. Ljung-Box residual autocorrelation tests
ljung_box = acorr_ljungbox(
    adl_120_ols.resid,
    lags=[6, 12],
    return_df=True
)

for lag in [6, 12]:

    diagnostic_results.append({
        "Category": "Serial correlation",
        "Test": f"Ljung-Box Q({lag})",
        "Statistic": ljung_box.loc[lag, "lb_stat"],
        "p_value": ljung_box.loc[lag, "lb_pvalue"],
        "Null_Hypothesis": (
            f"No residual autocorrelation through lag {lag}"
        ),
    })

# 3. Breusch-Pagan/Koenker heteroskedasticity test
bp_lm, bp_lm_p, bp_f, bp_f_p = het_breuschpagan(
    adl_120_ols.resid,
    adl_120_ols.model.exog,
    robust=True
)

diagnostic_results.append({
    "Category": "Heteroskedasticity",
    "Test": "Koenker-Breusch-Pagan",
    "Statistic": bp_lm,
    "p_value": bp_lm_p,
    "Null_Hypothesis": "Homoskedastic residuals",
})

# 4. ARCH effects
for lag in [1, 3, 6]:

    arch_lm, arch_lm_p, arch_f, arch_f_p = het_arch(
        adl_120_ols.resid,
        nlags=lag,
        ddof=int(adl_120_ols.df_model) + 1
    )

    diagnostic_results.append({
        "Category": "Conditional variance",
        "Test": f"ARCH LM({lag})",
        "Statistic": arch_lm,
        "p_value": arch_lm_p,
        "Null_Hypothesis": (
            f"No ARCH effects through lag {lag}"
        ),
    })

# 5. Ramsey RESET functional-form test
for power in [2, 3]:

    reset_result = linear_reset(
        adl_120_ols,
        power=power,
        test_type="fitted",
        use_f=True
    )

    diagnostic_results.append({
        "Category": "Functional form",
        "Test": f"Ramsey RESET power {power}",
        "Statistic": float(np.asarray(
            reset_result.fvalue
        ).item()),
        "p_value": float(np.asarray(
            reset_result.pvalue
        ).item()),
        "Null_Hypothesis": (
            "Linear functional form is correctly specified"
        ),
    })

# 6. CUSUM parameter-stability test
cusum_stat, cusum_p, cusum_critical_values = (
    breaks_cusumolsresid(
        adl_120_ols.resid,
        ddof=int(adl_120_ols.df_model) + 1
    )
)

diagnostic_results.append({
    "Category": "Parameter stability",
    "Test": "CUSUM",
    "Statistic": cusum_stat,
    "p_value": cusum_p,
    "Null_Hypothesis": "Regression parameters are stable",
})

# 7. Jarque-Bera residual-normality test
jb_stat, jb_p, residual_skewness, residual_kurtosis = (
    jarque_bera(adl_120_ols.resid)
)

diagnostic_results.append({
    "Category": "Residual distribution",
    "Test": "Jarque-Bera",
    "Statistic": jb_stat,
    "p_value": jb_p,
    "Null_Hypothesis": "Residuals are normally distributed",
})

diagnostic_table = pd.DataFrame(diagnostic_results)

diagnostic_table["Reject_Null_5pct"] = (
    diagnostic_table["p_value"] < 0.05
)

diagnostic_table["Conclusion"] = np.where(
    diagnostic_table["Reject_Null_5pct"],
    "Potential problem detected",
    "No evidence of problem"
)

display(diagnostic_table)

,Category,Test,Statistic,p_value,Null_Hypothesis,Reject_Null_5pct,Conclusion
0,Serial correlation,Breusch-Godfrey LM(1),0.2738,0.6008,No residual serial correlation through lag 1,False,No evidence of problem
1,Serial correlation,Breusch-Godfrey LM(2),0.7110,0.7008,No residual serial correlation through lag 2,False,No evidence of problem
2,Serial correlation,Breusch-Godfrey LM(3),0.7114,0.8705,No residual serial correlation through lag 3,False,No evidence of problem
3,Serial correlation,Breusch-Godfrey LM(6),0.8293,0.9913,No residual serial correlation through lag 6,False,No evidence of problem
4,Serial correlation,Ljung-Box Q(6),0.6710,0.9951,No residual autocorrelation through lag 6,False,No evidence of problem
5,Serial correlation,Ljung-Box Q(12),5.5967,0.9350,No residual autocorrelation through lag 12,False,No evidence of problem
6,Heteroskedasticity,Koenker-Breusch-Pagan,22.3567,0.1320,Homoskedastic residuals,False,No evidence of problem
7,Conditional variance,ARCH LM(1),4.8653,0.0274,No ARCH effects through lag 1,True,Potential problem detected
8,Conditional variance,ARCH LM(3),5.2574,0.1539,No ARCH effects through lag 3,False,No evidence of problem
9,Conditional variance,ARCH LM(6),6.2805,0.3925,No ARCH effects through lag 6,False,No evidence of problem


In [63]:
# CELL 38: Leverage, studentised residuals and Cook's distance

influence = adl_120_ols.get_influence()

influence_df = pd.DataFrame({
    "Month": adl_120_df["Month"],
    "Gold_Return": adl_120_df["Gold_Return"],
    "Fitted_Return": adl_120_ols.fittedvalues,
    "Residual": adl_120_ols.resid,
    "Studentised_Residual": (
        influence.resid_studentized_external
    ),
    "Leverage": influence.hat_matrix_diag,
    "Cooks_Distance": influence.cooks_distance[0],
})

n_obs = len(influence_df)
n_parameters = len(adl_120_ols.params)

cook_threshold = 4 / n_obs
leverage_threshold = 2 * n_parameters / n_obs

influence_df["Large_Studentised_Residual"] = (
    influence_df["Studentised_Residual"].abs() > 2
)

influence_df["High_Leverage"] = (
    influence_df["Leverage"] > leverage_threshold
)

influence_df["High_Cooks_Distance"] = (
    influence_df["Cooks_Distance"] > cook_threshold
)

influence_df["Potentially_Influential"] = (
    influence_df[
        [
            "Large_Studentised_Residual",
            "High_Leverage",
            "High_Cooks_Distance",
        ]
    ].any(axis=1)
)

print("Cook's-distance threshold:", round(cook_threshold, 4))
print("Leverage threshold:", round(leverage_threshold, 4))
print(
    "Potentially influential observations:",
    int(influence_df["Potentially_Influential"].sum())
)

display(
    influence_df[
        influence_df["Potentially_Influential"]
    ].sort_values(
        "Cooks_Distance",
        ascending=False
    )
)

Cook's-distance threshold: 0.0449
Leverage threshold: 0.382
Potentially influential observations: 11


,Month,Gold_Return,Fitted_Return,Residual,Studentised_Residual,Leverage,Cooks_Distance,Large_Studentised_Residual,High_Leverage,High_Cooks_Distance,Potentially_Influential
12,2020-03-01,-0.3300,2.4092,-2.7392,-1.6177,0.6574,0.2889,False,True,True,True
13,2020-04-01,5.7400,1.9122,3.8278,1.7779,0.4421,0.1430,False,True,True,True
87,2026-06-01,-12.2333,-5.1773,-7.0561,-2.8898,0.2321,0.1348,True,False,True,True
14,2020-05-01,1.9500,4.2080,-2.2580,-1.1880,0.5754,0.1119,False,True,True,True
79,2025-10-01,10.6600,2.0762,8.5838,3.3920,0.1424,0.0981,True,False,True,True
6,2019-09-01,-5.1879,2.2490,-7.4369,-2.8705,0.1367,0.0698,True,False,True,True
61,2024-04-01,8.0400,1.0140,7.0260,2.6602,0.1160,0.0504,True,False,True,True
78,2025-09-01,8.9000,1.4096,7.4904,2.7429,0.0496,0.0212,True,False,False,True
36,2022-03-01,4.9400,6.2426,-1.3026,-0.5868,0.4293,0.0154,False,True,False,True
38,2022-05-01,-4.5700,-5.1801,0.6101,0.2698,0.4099,0.0030,False,True,False,True


In [64]:
# CELL 39: Plotly residual and influence diagnostics

from plotly.subplots import make_subplots
import plotly.graph_objects as go

fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=[
        "Residuals Over Time",
        "Residuals versus Fitted Returns",
        "Studentised Residuals",
        "Cook's Distance",
    ],
    vertical_spacing=0.14,
    horizontal_spacing=0.10,
)

# Residuals through time
fig.add_trace(
    go.Scatter(
        x=influence_df["Month"],
        y=influence_df["Residual"],
        mode="lines+markers",
        name="Residual",
        hovertemplate=(
            "%{x|%b %Y}<br>"
            "Residual: %{y:.3f}<extra></extra>"
        ),
    ),
    row=1,
    col=1,
)

fig.add_hline(
    y=0,
    line_dash="dash",
    line_color="black",
    row=1,
    col=1,
)

# Residuals versus fitted values
fig.add_trace(
    go.Scatter(
        x=influence_df["Fitted_Return"],
        y=influence_df["Residual"],
        mode="markers",
        marker=dict(
            color=influence_df["Cooks_Distance"],
            colorscale="Reds",
            showscale=True,
            colorbar=dict(title="Cook's D"),
        ),
        text=influence_df["Month"].dt.strftime("%b %Y"),
        name="Residual vs fitted",
        hovertemplate=(
            "%{text}<br>"
            "Fitted: %{x:.3f}<br>"
            "Residual: %{y:.3f}<extra></extra>"
        ),
    ),
    row=1,
    col=2,
)

fig.add_hline(
    y=0,
    line_dash="dash",
    line_color="black",
    row=1,
    col=2,
)

# Studentised residuals
fig.add_trace(
    go.Scatter(
        x=influence_df["Month"],
        y=influence_df["Studentised_Residual"],
        mode="markers",
        name="Studentised residual",
        hovertemplate=(
            "%{x|%b %Y}<br>"
            "Studentised residual: %{y:.3f}<extra></extra>"
        ),
    ),
    row=2,
    col=1,
)

for threshold in [-2, 2]:
    fig.add_hline(
        y=threshold,
        line_dash="dash",
        line_color="red",
        row=2,
        col=1,
    )

# Cook's distance
fig.add_trace(
    go.Bar(
        x=influence_df["Month"],
        y=influence_df["Cooks_Distance"],
        name="Cook's distance",
        hovertemplate=(
            "%{x|%b %Y}<br>"
            "Cook's D: %{y:.4f}<extra></extra>"
        ),
    ),
    row=2,
    col=2,
)

fig.add_hline(
    y=cook_threshold,
    line_dash="dash",
    line_color="red",
    annotation_text="4/N threshold",
    row=2,
    col=2,
)

fig.update_layout(
    title="ADL(1,2,0): Residual and Influence Diagnostics",
    template="plotly_white",
    height=850,
    width=1400,
    showlegend=False,
)

fig.update_xaxes(title_text="Month", row=1, col=1)
fig.update_xaxes(title_text="Fitted gold return", row=1, col=2)
fig.update_xaxes(title_text="Month", row=2, col=1)
fig.update_xaxes(title_text="Month", row=2, col=2)

fig.update_yaxes(title_text="Residual", row=1, col=1)
fig.update_yaxes(title_text="Residual", row=1, col=2)
fig.update_yaxes(title_text="Studentised residual", row=2, col=1)
fig.update_yaxes(title_text="Cook's distance", row=2, col=2)

fig.show()

In [65]:
# CELL 40: Audit all variables for influential months

influential_months = (
    influence_df.loc[
        influence_df["Potentially_Influential"],
        "Month"
    ]
    .sort_values()
    .tolist()
)

audit_variables = [
    "Month",
    "Gold_Return",
    "ETF_Flow",
    "ETF_Flow_L1",
    "ETF_Flow_L2",
    "CB_IMF_Change",
    "Gold_Return_L1",
] + primary_controls

influential_data_audit = (
    primary_adl_df[
        primary_adl_df["Month"].isin(influential_months)
    ][audit_variables]
    .sort_values("Month")
    .reset_index(drop=True)
)

display(influential_data_audit)

,Month,Gold_Return,ETF_Flow,ETF_Flow_L1,ETF_Flow_L2,CB_IMF_Change,Gold_Return_L1,CPI_MoM,Broad_USD_Change,Real10Y_Change_pp,Term_Spread_Change_pp,Fed_Funds_Change_pp,USDINR_Change,USDCNY_Change,VIX_Change,EPU_Change,HY_OAS_Change_pp,Brent_Return_pct
0,2019-09-01,-5.1879,78.6000,122.0000,52.2000,49.2058,6.9111,0.1539,0.2235,0.2000,0.0500,-0.1702,0.1712,0.7202,-2.7400,-29.4553,-0.1100,0.5792
1,2020-03-01,-0.3300,150.9000,84.5000,61.7000,37.0671,2.3400,-0.2200,4.3122,0.1100,0.2000,-0.9300,4.2202,0.3394,13.4300,76.3985,3.7100,-54.2494
2,2020-04-01,5.7400,170.1000,150.9000,84.5000,41.1127,-0.3300,-0.6800,2.2576,-0.2600,-0.0300,-0.6000,2.1731,0.7176,-19.3900,-5.1319,-1.1400,-6.2409
3,2020-05-01,1.9500,154.3000,170.1000,150.9000,27.0649,5.7400,0.0100,-0.6653,-0.0700,0.0500,0.0000,-0.6705,0.4351,-6.6400,30.4685,-1.0900,84.9714
4,2022-03-01,4.9400,187.3000,35.3000,46.3000,-5.0673,2.2200,1.3400,1.2931,0.2700,-0.3500,0.1200,1.6275,0.0158,-9.5900,37.6368,-0.3400,4.5771
5,2022-05-01,-4.5700,-53.3000,42.9000,187.3000,43.1320,-0.5700,1.1100,2.5274,0.2000,0.1300,0.4400,1.4722,4.1666,-7.2100,8.6245,0.2500,9.3604
6,2024-04-01,8.0400,-33.1000,-13.6000,-49.2000,34.6700,6.6700,0.3900,1.4840,0.4000,0.0400,0.0000,0.4522,0.4991,2.6400,14.2507,0.0300,-0.5598
7,2025-09-01,8.9000,20.1000,53.0000,22.7000,36.9766,0.8400,0.2600,-0.5468,-0.0200,-0.0800,-0.1100,0.8586,-0.6865,0.9200,-11.7618,-0.0400,-1.8490
8,2025-10-01,10.6600,54.9000,20.1000,53.0000,47.2922,8.9000,-0.2500,0.7327,0.0100,-0.0500,-0.1300,0.0540,-0.0490,1.1600,-3.0009,0.1400,-2.2479
9,2026-03-01,-3.2800,-84.8000,26.2000,120.1000,-90.0485,5.6300,1.0500,2.0138,0.2800,-0.0800,0.0000,2.2837,-0.2067,5.3900,0.1033,0.1600,53.6141


In [66]:
# CELL 41: Leave-one-out robustness for key ADL coefficients

key_coefficients = [
    "ETF_Flow",
    "ETF_Flow_L1",
    "ETF_Flow_L2",
    "CB_IMF_Change",
    "Gold_Return_L1",
    "Broad_USD_Change",
]

leave_one_out_results = []

for row_to_remove in range(len(adl_120_df)):

    reduced_df = (
        adl_120_df
        .drop(index=row_to_remove)
        .reset_index(drop=True)
    )

    X_reduced = sm.add_constant(
        reduced_df[adl_120_regressors],
        has_constant="add"
    )

    y_reduced = reduced_df["Gold_Return"]

    reduced_model = sm.OLS(
        y_reduced,
        X_reduced
    ).fit(
        cov_type="HAC",
        cov_kwds={
            "maxlags": 3,
            "use_correction": True,
        }
    )

    removed_month = adl_120_df.loc[
        row_to_remove,
        "Month"
    ]

    for variable in key_coefficients:

        leave_one_out_results.append({
            "Removed_Month": removed_month,
            "Variable": variable,
            "Coefficient": reduced_model.params[variable],
            "HAC_p": reduced_model.pvalues[variable],
        })

leave_one_out_df = pd.DataFrame(
    leave_one_out_results
)

full_model_coefficients = (
    adl_120_results
    .loc[key_coefficients, "Coefficient"]
)

loo_summary = (
    leave_one_out_df
    .groupby("Variable")
    .agg(
        Minimum_Coefficient=("Coefficient", "min"),
        Maximum_Coefficient=("Coefficient", "max"),
        Median_Coefficient=("Coefficient", "median"),
        Minimum_p=("HAC_p", "min"),
        Maximum_p=("HAC_p", "max"),
        Significant_5pct_Share=(
            "HAC_p",
            lambda x: (x < 0.05).mean()
        ),
    )
    .reset_index()
)

loo_summary["Full_Model_Coefficient"] = (
    loo_summary["Variable"]
    .map(full_model_coefficients)
)

loo_summary["Sign_Stable"] = (
    np.sign(loo_summary["Minimum_Coefficient"])
    == np.sign(loo_summary["Maximum_Coefficient"])
)

display(loo_summary)

,Variable,Minimum_Coefficient,Maximum_Coefficient,Median_Coefficient,Minimum_p,Maximum_p,Significant_5pct_Share,Full_Model_Coefficient,Sign_Stable
0,Broad_USD_Change,-1.4014,-1.0315,-1.2478,0.0000,0.0005,1.0000,-1.2479,True
1,CB_IMF_Change,-0.0104,-0.0047,-0.0074,0.1062,0.4594,0.0000,-0.0074,True
2,ETF_Flow,0.0279,0.0337,0.0313,0.0000,0.0000,1.0000,0.0313,True
3,ETF_Flow_L1,-0.0117,-0.0032,-0.0083,0.1059,0.5864,0.0000,-0.0083,True
4,ETF_Flow_L2,-0.0153,-0.0098,-0.0134,0.0035,0.1178,0.9551,-0.0134,True
5,Gold_Return_L1,0.1343,0.2577,0.2181,0.0058,0.1980,0.8427,0.2181,True


In [67]:
# CELL 42: Plotly leave-one-out coefficient stability

from plotly.subplots import make_subplots
import plotly.graph_objects as go

fig = make_subplots(
    rows=3,
    cols=2,
    subplot_titles=key_coefficients,
    vertical_spacing=0.12,
    horizontal_spacing=0.10,
)

for position, variable in enumerate(key_coefficients):

    row = position // 2 + 1
    col = position % 2 + 1

    plot_df = leave_one_out_df[
        leave_one_out_df["Variable"] == variable
    ]

    fig.add_trace(
        go.Scatter(
            x=plot_df["Removed_Month"],
            y=plot_df["Coefficient"],
            mode="markers",
            name=variable,
            text=plot_df["HAC_p"].round(4),
            hovertemplate=(
                "Removed: %{x|%b %Y}<br>"
                "Coefficient: %{y:.4f}<br>"
                "HAC p-value: %{text}<extra></extra>"
            ),
        ),
        row=row,
        col=col,
    )

    fig.add_hline(
        y=full_model_coefficients[variable],
        line_dash="dash",
        line_color="red",
        row=row,
        col=col,
    )

    fig.add_hline(
        y=0,
        line_dash="dot",
        line_color="black",
        row=row,
        col=col,
    )

fig.update_layout(
    title="ADL(1,2,0): Leave-One-Out Coefficient Stability",
    template="plotly_white",
    height=1000,
    width=1400,
    showlegend=False,
)

fig.show()

In [68]:
# CELL 43: Identify fragile months and test leave-one-out ETF Granger causality

fragile_individual_results = (
    leave_one_out_df[
        (
            leave_one_out_df["Variable"].isin(
                ["ETF_Flow_L2", "Gold_Return_L1"]
            )
        )
        & (leave_one_out_df["HAC_p"] >= 0.05)
    ]
    .sort_values(
        ["Variable", "HAC_p"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

print("Deletions causing individual p-values to exceed 5%:")
display(fragile_individual_results)

loo_granger_results = []

for row_to_remove in range(len(adl_120_df)):

    reduced_df = (
        adl_120_df
        .drop(index=row_to_remove)
        .reset_index(drop=True)
    )

    X_reduced = sm.add_constant(
        reduced_df[adl_120_regressors],
        has_constant="add"
    )
    y_reduced = reduced_df["Gold_Return"]

    reduced_model = sm.OLS(
        y_reduced,
        X_reduced
    ).fit(
        cov_type="HAC",
        cov_kwds={
            "maxlags": 3,
            "use_correction": True,
        }
    )

    etf_joint_test = reduced_model.wald_test(
        [
            "ETF_Flow_L1 = 0",
            "ETF_Flow_L2 = 0",
        ],
        scalar=True
    )

    loo_granger_results.append({
        "Removed_Month": adl_120_df.loc[
            row_to_remove,
            "Month"
        ],
        "Wald_Statistic": scalar_value(
            etf_joint_test.statistic
        ),
        "p_value": scalar_value(
            etf_joint_test.pvalue
        ),
    })

loo_granger_df = pd.DataFrame(
    loo_granger_results
)

loo_granger_summary = pd.DataFrame({
    "Minimum_p_value": [
        loo_granger_df["p_value"].min()
    ],
    "Maximum_p_value": [
        loo_granger_df["p_value"].max()
    ],
    "Median_p_value": [
        loo_granger_df["p_value"].median()
    ],
    "Reject_5pct_Share": [
        (loo_granger_df["p_value"] < 0.05).mean()
    ],
    "Reject_10pct_Share": [
        (loo_granger_df["p_value"] < 0.10).mean()
    ],
})

display(loo_granger_summary)

print("Least favourable leave-one-out Granger results:")

display(
    loo_granger_df
    .sort_values("p_value", ascending=False)
    .head(10)
)

Deletions causing individual p-values to exceed 5%:


,Removed_Month,Variable,Coefficient,HAC_p
0,2026-06-01,ETF_Flow_L2,-0.0098,0.1178
1,2020-09-01,ETF_Flow_L2,-0.0106,0.1096
2,2025-06-01,ETF_Flow_L2,-0.0118,0.0740
3,2019-11-01,ETF_Flow_L2,-0.0127,0.0507
4,2025-10-01,Gold_Return_L1,0.1343,0.1980
5,2024-04-01,Gold_Return_L1,0.1606,0.1222
6,2020-10-01,Gold_Return_L1,0.1891,0.0927
7,2025-11-01,Gold_Return_L1,0.2439,0.0666
8,2019-08-01,Gold_Return_L1,0.2064,0.0644
9,2026-02-01,Gold_Return_L1,0.2000,0.0638


,Minimum_p_value,Maximum_p_value,Median_p_value,Reject_5pct_Share,Reject_10pct_Share
0,0.0006,0.0106,0.0021,1.0000,1.0000


Least favourable leave-one-out Granger results:


,Removed_Month,Wald_Statistic,p_value
19,2020-10-01,9.0864,0.0106
40,2022-07-01,9.1698,0.0102
15,2020-06-01,9.2401,0.0099
87,2026-06-01,9.8670,0.0072
18,2020-09-01,10.2982,0.0058
8,2019-11-01,10.4326,0.0054
49,2023-04-01,11.1181,0.0039
26,2021-05-01,11.1383,0.0038
79,2025-10-01,11.1620,0.0038
3,2019-06-01,11.2082,0.0037


In [69]:
# CELL 44: Strict forward Granger causality — ETF flows to gold returns

strict_granger_df = model_2_df.copy()

strict_controls = primary_controls

# Create lags of outcome, demand variables and controls
for lag in [1, 2]:

    strict_granger_df[f"Gold_Return_L{lag}"] = (
        strict_granger_df["Gold_Return"].shift(lag)
    )

    strict_granger_df[f"ETF_Flow_L{lag}"] = (
        strict_granger_df["ETF_Flow"].shift(lag)
    )

strict_granger_df["CB_IMF_Change_L1"] = (
    strict_granger_df["CB_IMF_Change"].shift(1)
)

for variable in strict_controls:
    strict_granger_df[f"{variable}_L1"] = (
        strict_granger_df[variable].shift(1)
    )

lagged_control_variables = [
    f"{variable}_L1"
    for variable in strict_controls
]

forward_regressors = [
    "Gold_Return_L1",
    "ETF_Flow_L1",
    "ETF_Flow_L2",
    "CB_IMF_Change_L1",
] + lagged_control_variables

forward_df = (
    strict_granger_df[
        ["Month", "Gold_Return"] + forward_regressors
    ]
    .dropna()
    .reset_index(drop=True)
)

X_forward = sm.add_constant(
    forward_df[forward_regressors],
    has_constant="add"
)

y_forward = forward_df["Gold_Return"]

forward_granger_model = sm.OLS(
    y_forward,
    X_forward
).fit(
    cov_type="HAC",
    cov_kwds={
        "maxlags": 3,
        "use_correction": True,
    }
)

# H0: lagged ETF flows do not predict gold returns
forward_etf_test = forward_granger_model.wald_test(
    [
        "ETF_Flow_L1 = 0",
        "ETF_Flow_L2 = 0",
    ],
    scalar=True
)

# H0: lagged CB change does not predict gold returns
forward_cb_test = forward_granger_model.wald_test(
    "CB_IMF_Change_L1 = 0",
    scalar=True
)

forward_results = pd.DataFrame({
    "Coefficient": forward_granger_model.params,
    "HAC_SE": forward_granger_model.bse,
    "HAC_t": forward_granger_model.tvalues,
    "HAC_p": forward_granger_model.pvalues,
    "CI_Lower_95pct": forward_granger_model.conf_int()[0],
    "CI_Upper_95pct": forward_granger_model.conf_int()[1],
})

forward_tests = pd.DataFrame({
    "Direction": [
        "ETF → Gold",
        "IMF CB → Gold",
    ],
    "Null_Hypothesis": [
        "ETF L1 = ETF L2 = 0",
        "CB L1 = 0",
    ],
    "Statistic": [
        scalar_value(forward_etf_test.statistic),
        scalar_value(forward_cb_test.statistic),
    ],
    "p_value": [
        scalar_value(forward_etf_test.pvalue),
        scalar_value(forward_cb_test.pvalue),
    ],
})

forward_tests["Reject_5pct"] = (
    forward_tests["p_value"] < 0.05
)

print("Strict forward Granger model")
print("Observations:", int(forward_granger_model.nobs))
print(
    "Period:",
    forward_df["Month"].min(),
    "to",
    forward_df["Month"].max()
)
print(
    "Adjusted R-squared:",
    round(forward_granger_model.rsquared_adj, 4)
)

display(
    forward_results.loc[
        [
            "Gold_Return_L1",
            "ETF_Flow_L1",
            "ETF_Flow_L2",
            "CB_IMF_Change_L1",
        ]
    ]
)

display(forward_tests)

Strict forward Granger model
Observations: 89
Period: 2019-03-01 00:00:00 to 2026-07-01 00:00:00
Adjusted R-squared: 0.0173


,Coefficient,HAC_SE,HAC_t,HAC_p,CI_Lower_95pct,CI_Upper_95pct
Gold_Return_L1,0.1470,0.1336,1.1003,0.2712,-0.1148,0.4087
ETF_Flow_L1,0.0091,0.0082,1.1092,0.2673,-0.0070,0.0253
ETF_Flow_L2,-0.0118,0.0078,-1.5091,0.1313,-0.0272,0.0035
CB_IMF_Change_L1,-0.0022,0.0099,-0.2237,0.8230,-0.0216,0.0172


,Direction,Null_Hypothesis,Statistic,p_value,Reject_5pct
0,ETF → Gold,ETF L1 = ETF L2 = 0,2.3536,0.3083,False
1,IMF CB → Gold,CB L1 = 0,0.0500,0.8230,False


In [70]:
# CELL 45: Reverse Granger causality — gold returns to ETF flows

reverse_regressors = [
    "ETF_Flow_L1",
    "ETF_Flow_L2",
    "Gold_Return_L1",
    "Gold_Return_L2",
    "CB_IMF_Change_L1",
] + lagged_control_variables

reverse_df = (
    strict_granger_df[
        ["Month", "ETF_Flow"] + reverse_regressors
    ]
    .dropna()
    .reset_index(drop=True)
)

X_reverse = sm.add_constant(
    reverse_df[reverse_regressors],
    has_constant="add"
)

y_reverse = reverse_df["ETF_Flow"]

reverse_granger_model = sm.OLS(
    y_reverse,
    X_reverse
).fit(
    cov_type="HAC",
    cov_kwds={
        "maxlags": 3,
        "use_correction": True,
    }
)

# H0: lagged gold returns do not predict ETF flows
reverse_gold_test = reverse_granger_model.wald_test(
    [
        "Gold_Return_L1 = 0",
        "Gold_Return_L2 = 0",
    ],
    scalar=True
)

reverse_results = pd.DataFrame({
    "Coefficient": reverse_granger_model.params,
    "HAC_SE": reverse_granger_model.bse,
    "HAC_t": reverse_granger_model.tvalues,
    "HAC_p": reverse_granger_model.pvalues,
    "CI_Lower_95pct": reverse_granger_model.conf_int()[0],
    "CI_Upper_95pct": reverse_granger_model.conf_int()[1],
})

reverse_test_table = pd.DataFrame({
    "Direction": ["Gold → ETF"],
    "Null_Hypothesis": [
        "Gold-return L1 = Gold-return L2 = 0"
    ],
    "Statistic": [
        scalar_value(reverse_gold_test.statistic)
    ],
    "p_value": [
        scalar_value(reverse_gold_test.pvalue)
    ],
})

reverse_test_table["Reject_5pct"] = (
    reverse_test_table["p_value"] < 0.05
)

print("Strict reverse Granger model")
print("Observations:", int(reverse_granger_model.nobs))
print(
    "Period:",
    reverse_df["Month"].min(),
    "to",
    reverse_df["Month"].max()
)
print(
    "Adjusted R-squared:",
    round(reverse_granger_model.rsquared_adj, 4)
)

display(
    reverse_results.loc[
        [
            "ETF_Flow_L1",
            "ETF_Flow_L2",
            "Gold_Return_L1",
            "Gold_Return_L2",
        ]
    ]
)

display(reverse_test_table)

Strict reverse Granger model
Observations: 89
Period: 2019-03-01 00:00:00 to 2026-07-01 00:00:00
Adjusted R-squared: 0.4053


,Coefficient,HAC_SE,HAC_t,HAC_p,CI_Lower_95pct,CI_Upper_95pct
ETF_Flow_L1,0.4029,0.1080,3.7294,0.0002,0.1912,0.6146
ETF_Flow_L2,-0.0407,0.1297,-0.3134,0.7540,-0.2949,0.2136
Gold_Return_L1,-0.4833,1.4736,-0.3279,0.7430,-3.3715,2.4050
Gold_Return_L2,2.4466,1.6090,1.5205,0.1284,-0.7071,5.6002


,Direction,Null_Hypothesis,Statistic,p_value,Reject_5pct
0,Gold → ETF,Gold-return L1 = Gold-return L2 = 0,2.8451,0.2411,False


In [71]:
# CELL 46: Nested control specifications for fixed ADL(1,2,1)

nested_control_specs = {
    "M3": [
        "Broad_USD_Change",
        "Real10Y_Change_pp",
        "VIX_Change",
    ],
    "M4": [
        "Broad_USD_Change",
        "Real10Y_Change_pp",
        "VIX_Change",
        "CPI_MoM",
    ],
    "M5": [
        "Broad_USD_Change",
        "Real10Y_Change_pp",
        "VIX_Change",
        "CPI_MoM",
        "Fed_Funds_Change_pp",
    ],
    "M6": [
        "Broad_USD_Change",
        "Real10Y_Change_pp",
        "VIX_Change",
        "CPI_MoM",
        "Fed_Funds_Change_pp",
        "Term_Spread_Change_pp",
    ],
    "M7": [
        "Broad_USD_Change",
        "Real10Y_Change_pp",
        "VIX_Change",
        "CPI_MoM",
        "Fed_Funds_Change_pp",
        "Term_Spread_Change_pp",
        "EPU_Change",
    ],
    "M8": [
        "Broad_USD_Change",
        "Real10Y_Change_pp",
        "VIX_Change",
        "CPI_MoM",
        "Fed_Funds_Change_pp",
        "Term_Spread_Change_pp",
        "EPU_Change",
        "HY_OAS_Change_pp",
    ],
    "M9": [
        "Broad_USD_Change",
        "Real10Y_Change_pp",
        "VIX_Change",
        "CPI_MoM",
        "Fed_Funds_Change_pp",
        "Term_Spread_Change_pp",
        "EPU_Change",
        "HY_OAS_Change_pp",
        "Brent_Return_pct",
    ],
}

dynamic_terms_121 = [
    "Gold_Return_L1",
    "ETF_Flow",
    "ETF_Flow_L1",
    "ETF_Flow_L2",
    "CB_IMF_Change",
    "CB_IMF_Change_L1",
]

final_control_set = nested_control_specs["M9"]

# One common sample for every specification
nested_common_df = (
    primary_adl_df[
        ["Month", "Gold_Return"]
        + dynamic_terms_121
        + final_control_set
    ]
    .dropna()
    .reset_index(drop=True)
)

print("Common observations:", len(nested_common_df))
print(
    "Common period:",
    nested_common_df["Month"].min(),
    "to",
    nested_common_df["Month"].max()
)

Common observations: 89
Common period: 2019-03-01 00:00:00 to 2026-07-01 00:00:00


In [72]:
# CELL 47: Estimate nested ADL(1,2,1) specifications

nested_models = {}
nested_model_summary = []
nested_coefficient_results = []
nested_hypothesis_results = []

for model_name, controls in nested_control_specs.items():

    regressors = dynamic_terms_121 + controls

    X = sm.add_constant(
        nested_common_df[regressors],
        has_constant="add"
    )
    y = nested_common_df["Gold_Return"]

    model = sm.OLS(y, X).fit(
        cov_type="HAC",
        cov_kwds={
            "maxlags": 3,
            "use_correction": True,
        }
    )

    nested_models[model_name] = model

    # Joint ETF-lag test
    etf_lag_test = model.wald_test(
        [
            "ETF_Flow_L1 = 0",
            "ETF_Flow_L2 = 0",
        ],
        scalar=True
    )

    # Lagged CB test
    cb_lag_test = model.wald_test(
        "CB_IMF_Change_L1 = 0",
        scalar=True
    )

    # Complete ETF channel
    complete_etf_test = model.wald_test(
        [
            "ETF_Flow = 0",
            "ETF_Flow_L1 = 0",
            "ETF_Flow_L2 = 0",
        ],
        scalar=True
    )

    # Complete CB channel
    complete_cb_test = model.wald_test(
        [
            "CB_IMF_Change = 0",
            "CB_IMF_Change_L1 = 0",
        ],
        scalar=True
    )

    # Cumulative effects
    cumulative_etf_test = model.t_test(
        "ETF_Flow + ETF_Flow_L1 + ETF_Flow_L2 = 0"
    )

    cumulative_cb_test = model.t_test(
        "CB_IMF_Change + CB_IMF_Change_L1 = 0"
    )

    nested_model_summary.append({
        "Model": model_name,
        "Controls": len(controls),
        "Parameters": len(model.params),
        "N": int(model.nobs),
        "R2": model.rsquared,
        "Adjusted_R2": model.rsquared_adj,
        "AIC": model.aic,
        "BIC": model.bic,
        "ETF_Lag_Block_p": scalar_value(
            etf_lag_test.pvalue
        ),
        "CB_Lag_p": scalar_value(
            cb_lag_test.pvalue
        ),
        "Complete_ETF_Channel_p": scalar_value(
            complete_etf_test.pvalue
        ),
        "Complete_CB_Channel_p": scalar_value(
            complete_cb_test.pvalue
        ),
        "Cumulative_ETF_Coefficient": (
            model.params["ETF_Flow"]
            + model.params["ETF_Flow_L1"]
            + model.params["ETF_Flow_L2"]
        ),
        "Cumulative_ETF_p": scalar_value(
            cumulative_etf_test.pvalue
        ),
        "Cumulative_CB_Coefficient": (
            model.params["CB_IMF_Change"]
            + model.params["CB_IMF_Change_L1"]
        ),
        "Cumulative_CB_p": scalar_value(
            cumulative_cb_test.pvalue
        ),
    })

    for variable in dynamic_terms_121:

        nested_coefficient_results.append({
            "Model": model_name,
            "Controls": len(controls),
            "Variable": variable,
            "Coefficient": model.params[variable],
            "HAC_SE": model.bse[variable],
            "HAC_p": model.pvalues[variable],
            "CI_Lower": model.conf_int().loc[variable, 0],
            "CI_Upper": model.conf_int().loc[variable, 1],
        })

nested_model_summary = pd.DataFrame(
    nested_model_summary
)

nested_coefficients = pd.DataFrame(
    nested_coefficient_results
)

display(nested_model_summary)

,Model,Controls,Parameters,N,R2,Adjusted_R2,AIC,BIC,ETF_Lag_Block_p,CB_Lag_p,Complete_ETF_Channel_p,Complete_CB_Channel_p,Cumulative_ETF_Coefficient,Cumulative_ETF_p,Cumulative_CB_Coefficient,Cumulative_CB_p
0,M3,3,10,89,0.4983,0.4412,449.9492,474.8356,0.0008,0.3027,0.0000,0.2163,0.0149,0.0037,-0.0019,0.8256
1,M4,4,11,89,0.4988,0.4345,451.8661,479.2411,0.0008,0.3056,0.0000,0.2072,0.0150,0.0034,-0.0015,0.8625
2,M5,5,12,89,0.4992,0.4276,453.8018,483.6654,0.0022,0.3336,0.0000,0.2018,0.0143,0.0230,-0.0018,0.8400
3,M6,6,13,89,0.5020,0.4234,455.2871,487.6394,0.0026,0.3529,0.0000,0.1997,0.0141,0.0280,-0.0022,0.8060
4,M7,7,14,89,0.5108,0.4261,455.7005,490.5414,0.0032,0.3627,0.0000,0.1747,0.0123,0.0538,-0.0024,0.7840
5,M8,8,15,89,0.5126,0.4204,457.3782,494.7078,0.0048,0.3815,0.0000,0.2143,0.0123,0.0610,-0.0019,0.8248
6,M9,9,16,89,0.5154,0.4158,458.8731,498.6912,0.0068,0.2642,0.0000,0.1167,0.0130,0.0382,-0.0020,0.8135


In [73]:
# CELL 48: Demand-channel coefficient comparison

coefficient_comparison = (
    nested_coefficients
    .pivot(
        index="Model",
        columns="Variable",
        values="Coefficient"
    )
    .reset_index()
)

pvalue_comparison = (
    nested_coefficients
    .pivot(
        index="Model",
        columns="Variable",
        values="HAC_p"
    )
    .reset_index()
)

print("ADL(1,2,1) coefficient comparison")
display(coefficient_comparison)

print("HAC p-value comparison")
display(pvalue_comparison)

ADL(1,2,1) coefficient comparison


Variable,Model,CB_IMF_Change,CB_IMF_Change_L1,ETF_Flow,ETF_Flow_L1,ETF_Flow_L2,Gold_Return_L1
0,M3,-0.0085,0.0066,0.0356,-0.0079,-0.0128,0.1896
1,M4,-0.0083,0.0068,0.0357,-0.0078,-0.0129,0.1906
2,M5,-0.0085,0.0067,0.0353,-0.0080,-0.0130,0.1903
3,M6,-0.0084,0.0063,0.0355,-0.0080,-0.0134,0.2054
4,M7,-0.0082,0.0058,0.0329,-0.0072,-0.0133,0.1953
5,M8,-0.0077,0.0057,0.0330,-0.0077,-0.0131,0.1945
6,M9,-0.0089,0.0069,0.0328,-0.0076,-0.0122,0.1971


HAC p-value comparison


Variable,Model,CB_IMF_Change,CB_IMF_Change_L1,ETF_Flow,ETF_Flow_L1,ETF_Flow_L2,Gold_Return_L1
0,M3,0.1552,0.3027,0.0000,0.2381,0.0339,0.0359
1,M4,0.1373,0.3056,0.0000,0.2657,0.0387,0.0363
2,M5,0.1308,0.3336,0.0000,0.2567,0.0420,0.0363
3,M6,0.1276,0.3529,0.0000,0.2596,0.0471,0.0283
4,M7,0.1252,0.3627,0.0000,0.3003,0.0431,0.0409
5,M8,0.1453,0.3815,0.0000,0.2834,0.0496,0.0435
6,M9,0.0958,0.2642,0.0000,0.2854,0.0696,0.0413


In [74]:
# CELL 49: Plotly coefficient stability across control specifications

from plotly.subplots import make_subplots
import plotly.graph_objects as go

variables_to_plot = [
    "ETF_Flow",
    "ETF_Flow_L1",
    "ETF_Flow_L2",
    "CB_IMF_Change",
    "CB_IMF_Change_L1",
    "Gold_Return_L1",
]

fig = make_subplots(
    rows=3,
    cols=2,
    subplot_titles=variables_to_plot,
    vertical_spacing=0.12,
    horizontal_spacing=0.10,
)

for position, variable in enumerate(variables_to_plot):

    row = position // 2 + 1
    col = position % 2 + 1

    plot_df = nested_coefficients[
        nested_coefficients["Variable"] == variable
    ].copy()

    fig.add_trace(
        go.Scatter(
            x=plot_df["Controls"],
            y=plot_df["Coefficient"],
            error_y=dict(
                type="data",
                symmetric=False,
                array=(
                    plot_df["CI_Upper"]
                    - plot_df["Coefficient"]
                ),
                arrayminus=(
                    plot_df["Coefficient"]
                    - plot_df["CI_Lower"]
                ),
            ),
            mode="lines+markers",
            text=plot_df["Model"],
            customdata=plot_df["HAC_p"],
            hovertemplate=(
                "%{text}<br>"
                "Controls: %{x}<br>"
                "Coefficient: %{y:.4f}<br>"
                "HAC p-value: %{customdata:.4f}"
                "<extra></extra>"
            ),
            name=variable,
        ),
        row=row,
        col=col,
    )

    fig.add_hline(
        y=0,
        line_dash="dash",
        line_color="black",
        row=row,
        col=col,
    )

fig.update_layout(
    title=(
        "ADL(1,2,1): Demand-Coefficient Stability "
        "Across Nested Controls"
    ),
    template="plotly_white",
    height=1050,
    width=1400,
    showlegend=False,
)

fig.update_xaxes(
    title_text="Number of controls",
    dtick=1
)

fig.show()

In [75]:
# CELL 50: Joint test of controls added beyond M3

m9_model = nested_models["M9"]

additional_controls = [
    "CPI_MoM",
    "Fed_Funds_Change_pp",
    "Term_Spread_Change_pp",
    "EPU_Change",
    "HY_OAS_Change_pp",
    "Brent_Return_pct",
]

additional_controls_test = m9_model.wald_test(
    [
        f"{variable} = 0"
        for variable in additional_controls
    ],
    scalar=True
)

additional_controls_result = pd.DataFrame({
    "Test": [
        "Six controls added beyond M3 jointly equal zero"
    ],
    "Number_of_Restrictions": [
        len(additional_controls)
    ],
    "Wald_Statistic": [
        scalar_value(additional_controls_test.statistic)
    ],
    "p_value": [
        scalar_value(additional_controls_test.pvalue)
    ],
})

additional_controls_result["Reject_5pct"] = (
    additional_controls_result["p_value"] < 0.05
)

additional_controls_result["Reject_10pct"] = (
    additional_controls_result["p_value"] < 0.10
)

display(additional_controls_result)

,Test,Number_of_Restrictions,Wald_Statistic,p_value,Reject_5pct,Reject_10pct
0,Six controls added beyond M3 jointly equal zero,6,3.4529,0.7502,False,False


In [76]:
primary_final_model = nested_models["M3"]

primary_final_controls = [
    "Broad_USD_Change",
    "Real10Y_Change_pp",
    "VIX_Change",
]

primary_final_regressors = (
    dynamic_terms_121
    + primary_final_controls
)

In [77]:
# CELL 51: Nested-control comparison for ADL(1,2,0)

dynamic_terms_120 = [
    "Gold_Return_L1",
    "ETF_Flow",
    "ETF_Flow_L1",
    "ETF_Flow_L2",
    "CB_IMF_Change",
]

# Common sample based on the complete M9 specification
nested_120_common_df = (
    primary_adl_df[
        ["Month", "Gold_Return"]
        + dynamic_terms_120
        + nested_control_specs["M9"]
    ]
    .dropna()
    .reset_index(drop=True)
)

nested_120_models = {}
nested_120_summary = []
nested_120_coefficients_list = []

for model_name, controls in nested_control_specs.items():

    regressors = dynamic_terms_120 + controls

    X = sm.add_constant(
        nested_120_common_df[regressors],
        has_constant="add"
    )

    y = nested_120_common_df["Gold_Return"]

    model = sm.OLS(y, X).fit(
        cov_type="HAC",
        cov_kwds={
            "maxlags": 3,
            "use_correction": True,
        }
    )

    nested_120_models[model_name] = model

    # Lagged ETF block
    etf_lag_test = model.wald_test(
        [
            "ETF_Flow_L1 = 0",
            "ETF_Flow_L2 = 0",
        ],
        scalar=True
    )

    # Complete ETF channel
    complete_etf_test = model.wald_test(
        [
            "ETF_Flow = 0",
            "ETF_Flow_L1 = 0",
            "ETF_Flow_L2 = 0",
        ],
        scalar=True
    )

    # Contemporaneous CB coefficient
    cb_current_test = model.t_test(
        "CB_IMF_Change = 0"
    )

    # Cumulative ETF coefficient
    cumulative_etf_test = model.t_test(
        "ETF_Flow + ETF_Flow_L1 + ETF_Flow_L2 = 0"
    )

    cumulative_etf_coefficient = (
        model.params["ETF_Flow"]
        + model.params["ETF_Flow_L1"]
        + model.params["ETF_Flow_L2"]
    )

    nested_120_summary.append({
        "Model": model_name,
        "Controls": len(controls),
        "Parameters": len(model.params),
        "N": int(model.nobs),
        "R2": model.rsquared,
        "Adjusted_R2": model.rsquared_adj,
        "AIC": model.aic,
        "BIC": model.bic,
        "ETF_Lag_Block_p": scalar_value(
            etf_lag_test.pvalue
        ),
        "Complete_ETF_Channel_p": scalar_value(
            complete_etf_test.pvalue
        ),
        "Cumulative_ETF_Coefficient": (
            cumulative_etf_coefficient
        ),
        "Cumulative_ETF_p": scalar_value(
            cumulative_etf_test.pvalue
        ),
        "CB_Current_Coefficient": (
            model.params["CB_IMF_Change"]
        ),
        "CB_Current_p": scalar_value(
            cb_current_test.pvalue
        ),
    })

    for variable in dynamic_terms_120:

        nested_120_coefficients_list.append({
            "Model": model_name,
            "Controls": len(controls),
            "Variable": variable,
            "Coefficient": model.params[variable],
            "HAC_SE": model.bse[variable],
            "HAC_p": model.pvalues[variable],
            "CI_Lower": model.conf_int().loc[
                variable, 0
            ],
            "CI_Upper": model.conf_int().loc[
                variable, 1
            ],
        })

nested_120_summary = pd.DataFrame(
    nested_120_summary
)

nested_120_coefficients = pd.DataFrame(
    nested_120_coefficients_list
)

print("ADL(1,2,0) nested-control comparison")
print("Observations:", len(nested_120_common_df))
print(
    "Period:",
    nested_120_common_df["Month"].min(),
    "to",
    nested_120_common_df["Month"].max()
)

display(nested_120_summary)

ADL(1,2,0) nested-control comparison
Observations: 89
Period: 2019-03-01 00:00:00 to 2026-07-01 00:00:00


,Model,Controls,Parameters,N,R2,Adjusted_R2,AIC,BIC,ETF_Lag_Block_p,Complete_ETF_Channel_p,Cumulative_ETF_Coefficient,Cumulative_ETF_p,CB_Current_Coefficient,CB_Current_p
0,M3,3,9,89,0.4947,0.4442,448.5933,470.9911,0.0003,0.0000,0.0144,0.0030,-0.0077,0.2131
1,M4,4,10,89,0.4949,0.4374,450.5497,475.4361,0.0003,0.0000,0.0144,0.0028,-0.0075,0.1924
2,M5,5,11,89,0.4955,0.4308,452.4539,479.8289,0.0008,0.0000,0.0135,0.0211,-0.0077,0.1807
3,M6,6,12,89,0.4988,0.4272,453.8624,483.7260,0.0010,0.0000,0.0134,0.0253,-0.0077,0.1754
4,M7,7,13,89,0.5080,0.4304,454.2098,486.5620,0.0014,0.0000,0.0116,0.0517,-0.0076,0.1758
5,M8,8,14,89,0.5099,0.4250,455.8724,490.7133,0.0022,0.0000,0.0116,0.0582,-0.0070,0.1987
6,M9,9,15,89,0.5116,0.4192,457.5645,494.8941,0.0027,0.0000,0.0120,0.0417,-0.0079,0.1626


In [78]:
# CELL 52: ADL(1,2,0) coefficient stability tables

coefficient_comparison_120 = (
    nested_120_coefficients
    .pivot(
        index="Model",
        columns="Variable",
        values="Coefficient"
    )
    .reset_index()
)

pvalue_comparison_120 = (
    nested_120_coefficients
    .pivot(
        index="Model",
        columns="Variable",
        values="HAC_p"
    )
    .reset_index()
)

print("ADL(1,2,0) coefficient comparison")
display(coefficient_comparison_120)

print("ADL(1,2,0) HAC p-value comparison")
display(pvalue_comparison_120)

ADL(1,2,0) coefficient comparison


Variable,Model,CB_IMF_Change,ETF_Flow,ETF_Flow_L1,ETF_Flow_L2,Gold_Return_L1
0,M3,-0.0077,0.0354,-0.0078,-0.0133,0.1808
1,M4,-0.0075,0.0355,-0.0076,-0.0134,0.1813
2,M5,-0.0077,0.0350,-0.0080,-0.0135,0.1811
3,M6,-0.0077,0.0353,-0.0080,-0.0139,0.1979
4,M7,-0.0076,0.0326,-0.0072,-0.0138,0.1881
5,M8,-0.0070,0.0328,-0.0076,-0.0136,0.1874
6,M9,-0.0079,0.0326,-0.0075,-0.0130,0.1883


ADL(1,2,0) HAC p-value comparison


Variable,Model,CB_IMF_Change,ETF_Flow,ETF_Flow_L1,ETF_Flow_L2,Gold_Return_L1
0,M3,0.2131,0.0000,0.2572,0.0265,0.0452
1,M4,0.1924,0.0000,0.2815,0.0316,0.0458
2,M5,0.1807,0.0000,0.2688,0.0338,0.0455
3,M6,0.1754,0.0000,0.2715,0.0375,0.0324
4,M7,0.1758,0.0000,0.3139,0.0338,0.0461
5,M8,0.1987,0.0000,0.2953,0.0395,0.0487
6,M9,0.1626,0.0000,0.2974,0.0529,0.0489


In [79]:
# CELL 53: Test whether M4-M9 controls jointly improve ADL(1,2,0)

m9_120_model = nested_120_models["M9"]

additional_controls_120_test = (
    m9_120_model.wald_test(
        [
            "CPI_MoM = 0",
            "Fed_Funds_Change_pp = 0",
            "Term_Spread_Change_pp = 0",
            "EPU_Change = 0",
            "HY_OAS_Change_pp = 0",
            "Brent_Return_pct = 0",
        ],
        scalar=True
    )
)

additional_controls_120_result = pd.DataFrame({
    "Test": [
        "Six controls added beyond M3 jointly equal zero"
    ],
    "Number_of_Restrictions": [6],
    "Wald_Statistic": [
        scalar_value(
            additional_controls_120_test.statistic
        )
    ],
    "p_value": [
        scalar_value(
            additional_controls_120_test.pvalue
        )
    ],
})

additional_controls_120_result["Reject_5pct"] = (
    additional_controls_120_result["p_value"] < 0.05
)

additional_controls_120_result["Reject_10pct"] = (
    additional_controls_120_result["p_value"] < 0.10
)

display(additional_controls_120_result)

,Test,Number_of_Restrictions,Wald_Statistic,p_value,Reject_5pct,Reject_10pct
0,Six controls added beyond M3 jointly equal zero,6,3.6521,0.7236,False,False


In [80]:
# CELL 54: ADL(1,2,0) specification curves

variables_120_to_plot = [
    "ETF_Flow",
    "ETF_Flow_L1",
    "ETF_Flow_L2",
    "CB_IMF_Change",
    "Gold_Return_L1",
]

fig = make_subplots(
    rows=3,
    cols=2,
    subplot_titles=variables_120_to_plot + [""],
    vertical_spacing=0.12,
    horizontal_spacing=0.10,
)

for position, variable in enumerate(
    variables_120_to_plot
):

    row = position // 2 + 1
    col = position % 2 + 1

    plot_df = nested_120_coefficients[
        nested_120_coefficients["Variable"]
        == variable
    ].copy()

    fig.add_trace(
        go.Scatter(
            x=plot_df["Controls"],
            y=plot_df["Coefficient"],
            error_y=dict(
                type="data",
                symmetric=False,
                array=(
                    plot_df["CI_Upper"]
                    - plot_df["Coefficient"]
                ),
                arrayminus=(
                    plot_df["Coefficient"]
                    - plot_df["CI_Lower"]
                ),
            ),
            mode="lines+markers",
            text=plot_df["Model"],
            customdata=plot_df["HAC_p"],
            hovertemplate=(
                "%{text}<br>"
                "Controls: %{x}<br>"
                "Coefficient: %{y:.4f}<br>"
                "HAC p-value: %{customdata:.4f}"
                "<extra></extra>"
            ),
        ),
        row=row,
        col=col,
    )

    fig.add_hline(
        y=0,
        line_dash="dash",
        line_color="black",
        row=row,
        col=col,
    )

fig.update_layout(
    title=(
        "ADL(1,2,0): Demand-Coefficient Stability "
        "Across Nested Controls"
    ),
    template="plotly_white",
    height=1000,
    width=1400,
    showlegend=False,
)

fig.update_xaxes(
    title_text="Number of controls",
    dtick=1
)

fig.show()

In [81]:
final_adl_model = nested_120_models["M3"]

final_adl_controls = [
    "Broad_USD_Change",
    "Real10Y_Change_pp",
    "VIX_Change",
]

final_adl_regressors = [
    "Gold_Return_L1",
    "ETF_Flow",
    "ETF_Flow_L1",
    "ETF_Flow_L2",
    "CB_IMF_Change",
] + final_adl_controls

In [82]:
# CELL 55: Convert regional ETF panel into monthly wide format

panel_df["Month"] = pd.to_datetime(
    panel_df["Month"]
)

# Identify the regional ETF column safely
regional_etf_candidates = [
    "ETF_Net_Demand_Regional_WGC_tonnes",
    "ETF_Regional_Level_tonnes",
    "ETF_Net_Demand_tonnes",
]

regional_etf_column = next(
    (
        column
        for column in regional_etf_candidates
        if column in panel_df.columns
    ),
    None
)

if regional_etf_column is None:
    raise KeyError(
        "Regional ETF variable was not found. "
        f"Available columns: {panel_df.columns.tolist()}"
    )

# Check that each month-region combination is unique
duplicate_region_rows = panel_df.duplicated(
    subset=["Month", "Region"],
    keep=False
)

if duplicate_region_rows.any():
    display(
        panel_df.loc[
            duplicate_region_rows,
            ["Month", "Region", regional_etf_column]
        ].sort_values(["Month", "Region"])
    )
    raise ValueError(
        "Duplicate month-region observations detected."
    )

regional_etf_wide = (
    panel_df
    .pivot(
        index="Month",
        columns="Region",
        values=regional_etf_column
    )
    .reset_index()
)

required_regions = [
    "North America",
    "Europe",
    "Asia",
    "Others",
]

missing_regions = [
    region
    for region in required_regions
    if region not in regional_etf_wide.columns
]

if missing_regions:
    raise KeyError(
        f"Missing ETF regions: {missing_regions}"
    )

regional_etf_wide = regional_etf_wide[
    ["Month"] + required_regions
].rename(columns={
    "North America": "ETF_North_America",
    "Europe": "ETF_Europe",
    "Asia": "ETF_Asia",
    "Others": "ETF_Others",
})

regional_model_df = (
    model_2_df
    .merge(
        regional_etf_wide,
        on="Month",
        how="left",
        validate="one_to_one"
    )
    .sort_values("Month")
    .reset_index(drop=True)
)

regional_etf_variables = [
    "ETF_North_America",
    "ETF_Europe",
    "ETF_Asia",
    "ETF_Others",
]

# Confirm that regional flows reconcile to global ETF flow
regional_model_df["Regional_ETF_Sum"] = (
    regional_model_df[
        regional_etf_variables
    ].sum(axis=1, min_count=4)
)

regional_model_df["ETF_Reconciliation_Error"] = (
    regional_model_df["Regional_ETF_Sum"]
    - regional_model_df["ETF_Flow"]
)

print(
    "Maximum regional-to-global ETF reconciliation error:",
    regional_model_df[
        "ETF_Reconciliation_Error"
    ].abs().max()
)

display(
    regional_model_df[
        ["Month", "ETF_Flow"]
        + regional_etf_variables
        + [
            "Regional_ETF_Sum",
            "ETF_Reconciliation_Error",
        ]
    ].head()
)

Maximum regional-to-global ETF reconciliation error: 0.0


,Month,ETF_Flow,ETF_North_America,ETF_Europe,ETF_Asia,ETF_Others,Regional_ETF_Sum,ETF_Reconciliation_Error
0,2019-01-01,71.8000,53.0000,20.0000,0.1000,-1.3000,71.8000,0.0000
1,2019-02-01,-32.4000,-29.0000,-0.3000,-3.0000,-0.1000,-32.4000,0.0000
2,2019-03-01,1.7000,2.5000,0.2000,-1.2000,0.2000,1.7000,0.0000
3,2019-04-01,-56.9000,-46.0000,-8.0000,-2.5000,-0.4000,-56.9000,0.0000
4,2019-05-01,-2.4000,-13.7000,15.9000,-4.1000,-0.5000,-2.4000,0.0000


In [83]:
# CELL 56: Regional ETF ADL(1,2,1) models

regional_adl_df = regional_model_df.copy()

# Common gold and CB lags
regional_adl_df["Gold_Return_L1"] = (
    regional_adl_df["Gold_Return"].shift(1)
)

regional_adl_df["CB_IMF_Change_L1"] = (
    regional_adl_df["CB_IMF_Change"].shift(1)
)

# Regional ETF lags
for etf_variable in regional_etf_variables:
    for lag in [1, 2]:
        regional_adl_df[
            f"{etf_variable}_L{lag}"
        ] = regional_adl_df[
            etf_variable
        ].shift(lag)

regional_m3_controls = [
    "Broad_USD_Change",
    "Real10Y_Change_pp",
    "VIX_Change",
]

regional_required_columns = (
    [
        "Gold_Return",
        "Gold_Return_L1",
        "CB_IMF_Change",
        "CB_IMF_Change_L1",
    ]
    + regional_m3_controls
)

for etf_variable in regional_etf_variables:
    regional_required_columns += [
        etf_variable,
        f"{etf_variable}_L1",
        f"{etf_variable}_L2",
    ]

# Identical sample for every regional regression
regional_adl_common = (
    regional_adl_df
    .dropna(subset=regional_required_columns)
    .reset_index(drop=True)
)

region_name_map = {
    "ETF_North_America": "North America",
    "ETF_Europe": "Europe",
    "ETF_Asia": "Asia",
    "ETF_Others": "Others",
}

regional_adl_models = {}
regional_adl_summary = []
regional_adl_coefficients = []

for etf_variable in regional_etf_variables:

    region = region_name_map[etf_variable]

    etf_l1 = f"{etf_variable}_L1"
    etf_l2 = f"{etf_variable}_L2"

    regressors = [
        "Gold_Return_L1",
        etf_variable,
        etf_l1,
        etf_l2,
        "CB_IMF_Change",
        "CB_IMF_Change_L1",
    ] + regional_m3_controls

    X = sm.add_constant(
        regional_adl_common[regressors],
        has_constant="add"
    )

    y = regional_adl_common["Gold_Return"]

    model = sm.OLS(y, X).fit(
        cov_type="HAC",
        cov_kwds={
            "maxlags": 3,
            "use_correction": True,
        }
    )

    regional_adl_models[region] = model

    # Lagged regional ETF block
    etf_lag_test = model.wald_test(
        [
            f"{etf_l1} = 0",
            f"{etf_l2} = 0",
        ],
        scalar=True
    )

    # Complete regional ETF channel
    complete_etf_test = model.wald_test(
        [
            f"{etf_variable} = 0",
            f"{etf_l1} = 0",
            f"{etf_l2} = 0",
        ],
        scalar=True
    )

    # Cumulative regional ETF coefficient
    cumulative_etf_test = model.t_test(
        f"{etf_variable} + {etf_l1} + "
        f"{etf_l2} = 0"
    )

    # Complete CB channel
    complete_cb_test = model.wald_test(
        [
            "CB_IMF_Change = 0",
            "CB_IMF_Change_L1 = 0",
        ],
        scalar=True
    )

    cumulative_etf_coefficient = (
        model.params[etf_variable]
        + model.params[etf_l1]
        + model.params[etf_l2]
    )

    regional_adl_summary.append({
        "Region": region,
        "N": int(model.nobs),
        "Parameters": len(model.params),
        "R2": model.rsquared,
        "Adjusted_R2": model.rsquared_adj,
        "AIC": model.aic,
        "BIC": model.bic,

        "ETF_Current_Coefficient":
            model.params[etf_variable],
        "ETF_Current_p":
            model.pvalues[etf_variable],

        "ETF_L1_Coefficient":
            model.params[etf_l1],
        "ETF_L1_p":
            model.pvalues[etf_l1],

        "ETF_L2_Coefficient":
            model.params[etf_l2],
        "ETF_L2_p":
            model.pvalues[etf_l2],

        "ETF_Lag_Block_p":
            scalar_value(etf_lag_test.pvalue),

        "Complete_ETF_Channel_p":
            scalar_value(complete_etf_test.pvalue),

        "Cumulative_ETF_Coefficient":
            cumulative_etf_coefficient,
        "Cumulative_ETF_p":
            scalar_value(cumulative_etf_test.pvalue),

        "CB_Current_Coefficient":
            model.params["CB_IMF_Change"],
        "CB_Current_p":
            model.pvalues["CB_IMF_Change"],

        "CB_L1_Coefficient":
            model.params["CB_IMF_Change_L1"],
        "CB_L1_p":
            model.pvalues["CB_IMF_Change_L1"],

        "Complete_CB_Channel_p":
            scalar_value(complete_cb_test.pvalue),

        "Gold_L1_Coefficient":
            model.params["Gold_Return_L1"],
        "Gold_L1_p":
            model.pvalues["Gold_Return_L1"],
    })

    coefficient_mapping = {
        etf_variable: "ETF_Current",
        etf_l1: "ETF_L1",
        etf_l2: "ETF_L2",
        "CB_IMF_Change": "CB_Current",
        "CB_IMF_Change_L1": "CB_L1",
        "Gold_Return_L1": "Gold_L1",
    }

    for variable, label in coefficient_mapping.items():

        regional_adl_coefficients.append({
            "Region": region,
            "Variable": label,
            "Coefficient": model.params[variable],
            "HAC_SE": model.bse[variable],
            "HAC_p": model.pvalues[variable],
            "CI_Lower": model.conf_int().loc[
                variable, 0
            ],
            "CI_Upper": model.conf_int().loc[
                variable, 1
            ],
        })

regional_adl_summary = pd.DataFrame(
    regional_adl_summary
)

regional_adl_coefficients = pd.DataFrame(
    regional_adl_coefficients
)

print("Regional ADL(1,2,1) observations:",
      len(regional_adl_common))

print(
    "Period:",
    regional_adl_common["Month"].min(),
    "to",
    regional_adl_common["Month"].max()
)

display(
    regional_adl_summary.sort_values(
        "AIC"
    ).reset_index(drop=True)
)

Regional ADL(1,2,1) observations: 89
Period: 2019-03-01 00:00:00 to 2026-07-01 00:00:00


,Region,N,Parameters,R2,Adjusted_R2,AIC,BIC,ETF_Current_Coefficient,ETF_Current_p,ETF_L1_Coefficient,ETF_L1_p,ETF_L2_Coefficient,ETF_L2_p,ETF_Lag_Block_p,Complete_ETF_Channel_p,Cumulative_ETF_Coefficient,Cumulative_ETF_p,CB_Current_Coefficient,CB_Current_p,CB_L1_Coefficient,CB_L1_p,Complete_CB_Channel_p,Gold_L1_Coefficient,Gold_L1_p
0,North America,89,10,0.4936,0.4359,450.7833,475.6697,0.0551,0.0000,-0.0174,0.0693,-0.0145,0.0628,0.0045,0.0000,0.0231,0.0035,-0.0122,0.0415,0.0067,0.2619,0.0507,0.1677,0.1049
1,Asia,89,10,0.4543,0.3922,457.4314,482.3178,0.1462,0.0000,0.0063,0.7897,-0.0625,0.0206,0.0671,0.0004,0.0900,0.0068,-0.0171,0.0258,0.0093,0.1189,0.0336,0.0703,0.3959
2,Others,89,10,0.3624,0.2898,471.2868,496.1731,0.7695,0.0127,0.2618,0.2471,-0.4254,0.1737,0.2658,0.0071,0.6059,0.0569,-0.0063,0.4073,0.0066,0.2527,0.2890,0.1773,0.0715
3,Europe,89,10,0.3006,0.2209,479.5224,504.4088,0.0262,0.2057,0.0099,0.5750,-0.0145,0.2417,0.5040,0.3165,0.0216,0.2201,-0.0068,0.3390,0.0073,0.2074,0.2145,0.2240,0.0043
